# Setup Beamforming

No need to run these

In [ ]:
!pip install librosa soundfile

# SPIR Beamforming
Impulse Response-based Multidirectional Beamforming

Works on 27 Aug 2025

In [ ]:
# Rifqi Ikhwanuddin
# 27 Aug 2025
# PhD Design Engineering
# Imperial College London

import os
import sys
import numpy as np
import scipy.signal as signal
from scipy.signal import stft, istft
import librosa
import soundfile as sf
import glob
import gc
import numpy.linalg as la
import time
from datetime import datetime

class AutoBeamformer:
    def __init__(self, folder_name, drive_base, base_path, spir_path, fs_target=16000, fs_ir_original=48000, fc=1000):
        self.folder_name = folder_name
        self.fs_target = fs_target
        self.fs_ir_original = fs_ir_original
        self.fc = fc
        self.fs = 16000
        self.drive_base = drive_base
        self.input_base_path = base_path
        self.output_base_path = spir_path
        self.flac_name = os.path.join(self.input_base_path, f'{folder_name}.flac')

        # IRs Folder Location
        self.folder_ir = os.path.join(drive_base, '2025 Imperial/Silwood Park IRs')

        # Filter spesifikasi
        Wp = fc / (self.fs / 2)
        Ws = Wp / 2
        Rp = 3
        Rs = 40
        self.n, self.Wn = signal.buttord(Wp, Ws, Rp, Rs)
        self.b, self.a = signal.butter(self.n, self.Wn, btype='high')

        # Baca file FLAC
        print("Reading audio file...")
        self.raw, _ = librosa.load(self.flac_name, sr=self.fs, mono=False)
        self.filtered_audio = self._apply_highpass_filter(self.raw)

    def _apply_highpass_filter(self, audio):
        return signal.filtfilt(self.b, self.a, audio)

    def compute_weights_vectorized(self, Rxx, IRR):
        n_channels, n_freq, n_frame = IRR.shape

        # Pre-allocate W
        W = np.empty((n_freq, n_channels, n_frame), dtype=np.complex128)
        Rxx = Rxx.astype(np.complex128)
        # Solve for invRd and calculate weights
        for ifreq in range(n_freq):
            invRd = la.solve(Rxx, IRR[:, ifreq, :])

            # Replace np.einsum with a loop-based dot product
            projection = np.zeros(n_frame, dtype=np.complex128)
            for iframe in range(n_frame):
                projection[iframe] = np.dot(IRR[:, ifreq, iframe].conj(), invRd[:, iframe])

            W[ifreq, :, :] = np.conj(invRd / projection)  # Normalize and conjugate
        return W

    # SPIR
    def process_speaker(self, speaker_range=(1, 7, 1), degrees_step=60):
        for speakers in range(*speaker_range):
            print(f"Processing speakers: {speakers}")
            speaker = 2**speakers
            print(f"Speaker: {speaker}")
            speaker = f"{speaker:02d}"
            for degrees in range(180, 240, degrees_step):
                print(f"Processing {speakers} at {degrees} degrees...")
                self._process_single_ir(speakers, degrees)
        del self.filtered_audio

    def _process_single_ir(self, speaker, degrees):
        # Baca dan resample IR
        filename = f"{self.folder_ir}/{speaker:02d}m_{degrees:03d}.wav"
        print(f"Processing {speaker} at {degrees} degrees...")

        ir_original, _ = librosa.load(filename, sr=self.fs_ir_original, mono=False)
        ir = librosa.resample(ir_original, orig_sr=self.fs_ir_original, target_sr=self.fs_target)

        # STFT
        framelen = int(0.02 * self.fs)
        frameinc = framelen // 2
        X = librosa.stft(self.filtered_audio, n_fft=framelen, hop_length=frameinc, window='hamming')

        # Beamforming
        IR = np.fft.rfft(ir, n=framelen)
        n_channel = IR.shape[0]
        IRR = IR / IR[0, :]

        # Reshape IRR to have 3 dimensions if necessary
        if IRR.ndim == 2:
            IRR = IRR.reshape(IRR.shape[0], IRR.shape[1], 1)
            #Adding a dimension for n_frames

        n_freq, n_frame = X.shape[1], X.shape[2]
        Rxx = np.eye(n_channel)  # Identity covariance matrix

        # Hitung weights
        W = self.compute_weights_vectorized(Rxx, IRR) #(Rxx, IRR, n_freq, n_frame)

        # Terapkan beamforming
        Z = np.empty((X.shape[1], 1, X.shape[2]), dtype=np.complex128)
        Y = np.transpose(X, (1, 0, 2))  # Transpose X to [nfreq, nchan, nframe]
        Z = np.sum(W * Y, axis=1, keepdims=True)
        del W, Y, X, IR, IRR

        z = librosa.istft(Z[:, 0, :], hop_length=frameinc, window='hamming')
        del Z

        z = np.real(z) / np.max(np.abs(z))  # Normalisasi
        self._save_output(z, speaker, degrees) #, index
        del z


    def _save_output(self, audio, speaker, degrees): #index
        output_folder_path = os.path.join(self.output_base_path, self.folder_name)
        print(f"Output folder path: {output_folder_path}")
        os.makedirs(output_folder_path, exist_ok=True)

        #SPIR
        output_filename = os.path.join(output_folder_path, f"{self.folder_name}_spir({speaker:02d}m_{degrees:03d}).wav")

        sf.write(output_filename, audio, self.fs)
        print(f"Saved: {output_filename}")
        del audio


# === MAIN PROGRAM ===
if __name__ == "__main__":
    start_time = time.time()
    # folders = [2,4,8,16]
    for folder in range(0,7,1): #folders:
        for degrees in range(0, 360, 60):
            degree = f"{degrees:03d}"
            folder_name = f"{folder}" #f"{folder:02d}m" #_{degree}
            drive_base = '/content/drive/MyDrive'
            print(f"Folder name: {folder_name}")
            print(f"Drive base: {drive_base}")
            base_path = os.path.join(drive_base, '2025 Imperial/202508 Lab/01Experiment_LabIR')
            spir_path = os.path.join(drive_base, '2025 Imperial/202508 Lab/01Experiment_SPIR')
            print(f"Base path: {base_path}")

            # Arahkan output print ke file teks
            # log_file_path = os.path.join(base_path, folder_name, f'{folder_name}_logs.txt')
            # os.makedirs(os.path.dirname(log_file_path), exist_ok=True)
            # sys.stdout = open(log_file_path, "w")

            beamformer = AutoBeamformer(folder_name, drive_base, base_path, spir_path)

            #SPIR
            # speakers = [1,2,4,5,6,8,10,12,14,16,18,20,22,24]
            speakers = [2,4,8,16]
            for speaker in speakers:
                print(f"Folder: {folder}\nDegree: {degree}")
                # index untuk variasi impulse responses
                # for index in range(1, 4, 1): index=index
                beamformer._process_single_ir(speaker=speaker, degrees=degrees)

            # beamformer.process_speaker(speaker_range=(1, 13, 1), degrees_step=60)

            end_time = time.time()
            duration = end_time - start_time
            print(f"Done! Total execution time: {duration:.2f} seconds")

    # sys.stdout.close()
    # sys.stdout = sys.__stdout__

    current_datetime = datetime.now()
    timestamp = current_datetime.strftime("%Y%m%d_%H%M%S")
    print(f"Finish at {timestamp}")

# LabIR Beamforming
Impulse Response-based Multidirectional Beamforming

Works on 14 Jul 2026

In [ ]:
# Rifqi Ikhwanuddin
# 14 Jul 2026
# PhD Design Engineering
# Imperial College London

import os
import sys
import numpy as np
import scipy.signal as signal
from scipy.signal import stft, istft
import librosa
import soundfile as sf
import glob
import gc
import numpy.linalg as la
import time
from datetime import datetime

class AutoBeamformer:
    def __init__(self, folder_name, drive_base, base_path, fs_target=16000, fs_ir_original=48000, fc=1000):
        self.folder_name = folder_name
        self.fs_target = fs_target
        self.fs_ir_original = fs_ir_original
        self.fc = fc
        self.fs = 16000  # Sampling frequency
        self.drive_base = drive_base
        self.output_base_path = base_path
        self.flac_name = os.path.join(self.output_base_path, f'{folder_name}.flac')

        # IRs Folder Location
        # self.folder_ir = os.path.join(drive_base, '2025 Imperial/Silwood Park IRs')
        self.folder_ir = os.path.join(drive_base, 'IR_Sonisphere_408')

        # Filter spesifikasi (High-pass 1000 Hz)
        Wp = fc / (self.fs / 2)
        Ws = Wp / 2
        Rp = 3
        Rs = 40
        self.n, self.Wn = signal.buttord(Wp, Ws, Rp, Rs)
        self.b, self.a = signal.butter(self.n, self.Wn, btype='high')

        # Filter spesifikasi (Low-pass 4000 Hz)
        fc_lp = 4000
        Wp_lp = fc_lp / (self.fs / 2)
        Ws_lp = Wp_lp * 1.2 # Transition band
        self.n_lp, self.Wn_lp = signal.buttord(Wp_lp, Ws_lp, Rp, Rs)
        self.b_lp, self.a_lp = signal.butter(self.n_lp, self.Wn_lp, btype='low')

        # Baca file FLAC
        print("Reading audio file...")
        self.raw, _ = librosa.load(self.flac_name, sr=self.fs, mono=False)
        self.filtered_audio = self._apply_filters(self.raw)

    def _apply_filters(self, audio):
        # Terapkan High-pass
        filtered = signal.filtfilt(self.b, self.a, audio)
        # Terapkan Low-pass
        filtered = signal.filtfilt(self.b_lp, self.a_lp, filtered)
        return filtered

    def compute_weights_vectorized(self, Rxx, IRR):
        n_channels, n_freq, n_frame = IRR.shape

        # Pre-allocate W
        W = np.empty((n_freq, n_channels, n_frame), dtype=np.complex128)
        Rxx = Rxx.astype(np.complex128)
        # Solve for invRd and calculate weights
        for ifreq in range(n_freq):
            invRd = la.solve(Rxx, IRR[:, ifreq, :])

            # Replace np.einsum with a loop-based dot product
            projection = np.zeros(n_frame, dtype=np.complex128)
            for iframe in range(n_frame):
                projection[iframe] = np.dot(IRR[:, ifreq, iframe].conj(), invRd[:, iframe])

            W[ifreq, :, :] = np.conj(invRd / projection)  # Normalize and conjugate
        return W

    # LabIR
    def process_speaker(self, speaker_range=(1, 12, 1), degrees_step=10):
        for speakers in range(*speaker_range):
            speaker = f"{speakers:02d}"
            for degrees in range(0, 360, degrees_step):
                # print(f"Processing {speaker} at {degrees} degrees...")
                self._process_single_ir(speaker, degrees)
        del self.filtered_audio


    # SPIR
    # def process_speaker(self, speaker_range=(1, 7, 1), degrees_step=60):
    #     for speakers in range(*speaker_range):
    #         print(f"Processing speakers: {speakers}")
    #         speaker = 2**speakers
    #         print(f"Speaker: {speaker}")
    #         speaker = f"{speaker:02d}"
    #         for degrees in range(180, 240, degrees_step):
    #             print(f"Processing {speakers} at {degrees} degrees...")
    #             self._process_single_ir(speakers, degrees)
    #     del self.filtered_audio

    def _process_single_ir(self, speaker, degrees):
        # Baca dan resample IR
        # filename = f"{self.folder_ir}/{speaker:02d}m_{degrees:03d}.wav"
        print(f"Processing {speaker} at {degrees} degrees...")
        filename = f"{self.folder_ir}/Lab_IR_S{speaker}_{degrees:03d}.wav"

        ir_original, _ = librosa.load(filename, sr=self.fs_ir_original, mono=False)
        ir = librosa.resample(ir_original, orig_sr=self.fs_ir_original, target_sr=self.fs_target)

        # STFT
        framelen = int(0.02 * self.fs)
        frameinc = framelen // 2
        X = librosa.stft(self.filtered_audio, n_fft=framelen, hop_length=frameinc, window='hamming')
        # f, t, X = stft(self.filtered_audio, fs=self.fs, window='hamming', nperseg=framelen, noverlap=framelen-frameinc)

        # Beamforming
        IR = np.fft.rfft(ir, n=framelen)
        n_channel = IR.shape[0]
        IRR = IR / IR[0, :]

        # Reshape IRR to have 3 dimensions if necessary
        if IRR.ndim == 2:
            IRR = IRR.reshape(IRR.shape[0], IRR.shape[1], 1)
            #Adding a dimension for n_frames
            #IRR = np.expand_dims(IRR, axis=2)
            # Alternative way using np.expand_dims

        n_freq, n_frame = X.shape[1], X.shape[2]
        Rxx = np.eye(n_channel)  # Identity covariance matrix

        # Hitung weights
        W = self.compute_weights_vectorized(Rxx, IRR) #(Rxx, IRR, n_freq, n_frame)

        # Terapkan beamforming
        Z = np.empty((X.shape[1], 1, X.shape[2]), dtype=np.complex128)
        Y = np.transpose(X, (1, 0, 2))  # Transpose X to [nfreq, nchan, nframe]
        Z = np.sum(W * Y, axis=1, keepdims=True)

        del W, Y, X, IR, IRR

        z = librosa.istft(Z[:, 0, :], hop_length=frameinc, window='hamming')

        del Z

        z = np.real(z) / np.max(np.abs(z))  # Normalisasi

        # Simpan output
        self._save_output(z, speaker, degrees) #, index

        del z


    def _save_output(self, audio, speaker, degrees): #index
        output_folder_path = os.path.join(self.output_base_path, self.folder_name)
        os.makedirs(output_folder_path, exist_ok=True)
        #SPIR
        # output_filename = os.path.join(output_folder_path, f"{self.folder_name}_irbf({speaker:02d}m_{degrees:03d}).wav")

        #LabIR
        output_filename = os.path.join(output_folder_path, f"{self.folder_name}_LabIR(S{speaker}_{degrees:03d}).wav") #_{index}

        sf.write(output_filename, audio, self.fs)
        print(f"Saved: {output_filename}")
        del audio


# === MAIN PROGRAM ===
if __name__ == "__main__":
    start_time = time.time()
    # folders = [1,2,4,5,6,8,10,12,14,16,18,20,22,24]
    # folders = [2,4,8,16]
    # folders = [1]
    # for folder in folders:
    #     for degrees in range(60, 360, 60):
    #         degree = f"{degrees:03d}"
    #         folder_name = f"{folder:02d}m" #_{degree}
    #         drive_base = '/content/drive/MyDrive'
    #         print(f"Folder name: {folder_name}")
    #         print(f"Drive base: {drive_base}")
    #         base_path = os.path.join(drive_base, '2025 Imperial/202507 Silwood Park/06Experiment_SPIR')

    #         # base_path = os.path.join(drive_base, '2025 Imperial/202507 Silwood Park/03Experiment_SPIR')
    #         print(f"Base path: {base_path}")

    #         # Arahkan output print ke file teks
    #         # log_file_path = os.path.join(base_path, folder_name, f'{folder_name}_logs.txt')
    #         # os.makedirs(os.path.dirname(log_file_path), exist_ok=True)
    #         # sys.stdout = open(log_file_path, "w")

    #         beamformer = AutoBeamformer(folder_name, drive_base, base_path)
    #         # LabIR
    #         # beamformer._process_single_ir(speaker='12', degrees=0)
    #         # beamformer.process_speaker(speaker_range=(1, 12, 1), degrees_step=60)

    #         #SPIR
    #         # speakers = [1,2,4,5,6,8,10,12,14,16,18,20,22,24]
    #         speakers = [2,4,8,16]
    #         for speaker in speakers:
    #             # print(f"Folder: {folder}\nDegree: {degree}")
    #             # index untuk variasi impulse responses
    #             # for index in range(1, 4, 1): index=index
    #             beamformer._process_single_ir(speaker=speaker, degrees=degrees)

    #         # beamformer.process_speaker(speaker_range=(1, 13, 1), degrees_step=60)

    #         end_time = time.time()
    #         duration = end_time - start_time
    #         print(f"Done! Total execution time: {duration:.2f} seconds")

    # sys.stdout.close()
    # sys.stdout = sys.__stdout__

    # ######
    # Kode untuk berkas flac lebih dari satu
    # Base path
    drive_base = '/content/drive/MyDrive/'
    base_path = os.path.join(drive_base, '2026 Imperial/07 Way Canguk LabIR/RPiID-00000000058096e0/2026-07-10/')

    # Cari semua file .flac
    flac_files = glob.glob(os.path.join(base_path, '*.flac'))

    if not flac_files:
        print(f"Tidak ada file FLAC yang ditemukan di {base_path}")
    else:
        # Proses setiap file FLAC yang ditemukan
        for flac_path in flac_files:
            # Dapatkan folder_name dari nama file (tanpa .flac)
            folder_name = os.path.splitext(os.path.basename(flac_path))[0]
            print(f"\nProcessing file: {folder_name}")

            # Inisialisasi beamformer
            beamformer = AutoBeamformer(folder_name, drive_base, base_path)

            # Proses speaker 12 dengan degree 0
            beamformer._process_single_ir(speaker='12', degrees=0)

            # Proses speaker lainnya (1,5,9) dengan step 45 derajat
            beamformer.process_speaker(
                speaker_range=(1, 12, 4),
                degrees_step=60
            )
            print(f"Selesai memproses: {folder_name}")
        print("\nSemua file FLAC telah selesai diproses!")
    end_time = time.time()
    duration = end_time - start_time
    current_datetime = datetime.now()
    timestamp = current_datetime.strftime("%Y%m%d_%H%M%S")
    print(f"Done! Total execution time: {duration} seconds")
    print(f"Finish at {timestamp}")

# Signal Averaging (SA)
14 Jul 2026

In [ ]:
# Rifqi Ikhwanuddin
# 14 Jul 2026
# PhD Design Engineering
# Imperial College London

import os
import numpy as np
import librosa
import soundfile as sf
import glob
import time
import scipy.signal as signal
from datetime import datetime # Import datetime

class DirectSumAverager:
    def __init__(self, folder_name, drive_base, base_path, sa_path, fs_target=16000, fc=1000):
        self.folder_name = folder_name
        self.fs_target = fs_target
        self.fc = fc
        self.fs = 16000  # Sampling frequency
        self.drive_base = drive_base
        self.input_base_path = base_path
        self.output_base_path = sa_path
        self.flac_name = os.path.join(self.input_base_path, f'{folder_name}.flac')

        # Filter specifications (high-pass Butterworth filter)
        Wp = fc / (self.fs / 2)  # Passband frequency
        Ws = Wp / 2              # Stopband frequency
        Rp = 3                   # Passband ripple (dB)
        Rs = 40                  # Stopband attenuation (dB)
        self.n, self.Wn = signal.buttord(Wp, Ws, Rp, Rs)
        self.b, self.a = signal.butter(self.n, self.Wn, btype='high')

        # Read the FLAC file
        print(f"Reading audio file: {self.flac_name}...")
        self.raw, _ = librosa.load(self.flac_name, sr=self.fs, mono=False)
        self.filtered_audio = self._apply_highpass_filter(self.raw)

    def _apply_highpass_filter(self, audio):
        """
        Apply the high-pass Butterworth filter to the audio.
        """
        print("Applying high-pass filter...")
        return signal.filtfilt(self.b, self.a, audio)

    def process_audio(self):
        """
        Process the 6-channel FLAC file using Direct Sum Averaging.
        """
        print(f"Processing {self.folder_name}...")

        # Step 1: Use the pre-filtered 6-channel audio
        audio = self.filtered_audio
        if audio.shape[0] != 6:
            raise ValueError(f"Expected 6 channels, but got {audio.shape[0]} channels.")

        # Step 2: Sum the 6 channels and average (divide by 6)
        output_audio = np.sum(audio, axis=0) / 6.0

        # Ensure no clipping by normalizing to avoid exceeding [-1, 1]
        max_amplitude = np.max(np.abs(output_audio))
        if max_amplitude > 1.0:
            output_audio = output_audio / max_amplitude

        # Step 3: Save the output as a single-channel WAV file
        self._save_output(output_audio)

        del audio, output_audio
        # print(f"Finished processing {self.folder_name}.")

    def _save_output(self, audio):
        """
        Save the processed audio to a WAV file.
        """
        output_folder_path = os.path.join(self.output_base_path, self.folder_name)
        os.makedirs(output_folder_path, exist_ok=True)
        output_filename = os.path.join(output_folder_path, f"{self.folder_name}_sa.wav")
        sf.write(output_filename, audio, self.fs_target)
        print(f"Saved: {output_filename}")

# === MAIN PROGRAM ===
if __name__ == "__main__":
    start_time = time.time()

    # Base path
    drive_base = '/content/drive/MyDrive/'
    base_path = os.path.join(drive_base, '2025 Imperial/202508 Lab/01Experiment_LabIR/')
    sa_path = os.path.join(drive_base, '2025 Imperial/202508 Lab/01Experiment_SA/')

    # Find all FLAC files
    flac_files = glob.glob(os.path.join(base_path, '*.flac'))

    if not flac_files:
        print(f"No FLAC files found in {base_path}")
    else:
        # Process each FLAC file
        for flac_path in flac_files:
            folder_name = os.path.splitext(os.path.basename(flac_path))[0]
            print(f"\nProcessing file: {folder_name}")

            # Initialize DirectSumAverager
            sa = DirectSumAverager(folder_name, drive_base, base_path, sa_path)

            # Process the audio file
            sa.process_audio()

            print(f"Finished processing: {folder_name}")
        print("\nAll FLAC files have been processed!")

    end_time = time.time()
    duration = end_time - start_time
    current_datetime = datetime.now()
    timestamp = current_datetime.strftime("%Y %m %d - %H:%M:%S")
    print(f"Done! Total execution time: {duration:0f} seconds")
    print(f"Finish at {timestamp}")

# Setup BirdNET

In [ ]:
# basic calculation
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob

# audio processing
from scipy.io import wavfile as wavefile
from IPython.display import Audio
import subprocess

!pip install birdnetlib
!pip install resampy
!pip install pydub

# Extract First Channel + BirdNET
Works on 14 Jul 2026

In [ ]:
import os
import re
import sys
import time
import json
import librosa
import resampy
import subprocess
import numpy as np
import soundfile as sf
from scipy import signal
from datetime import datetime
from pydub import AudioSegment
from jinja2 import Environment, FileSystemLoader
from birdnetlib.analyzer import Analyzer
from birdnetlib.batch import DirectoryMultiProcessingAnalyzer

LOCATION_DICT = {"silwood": {"lat": 51.409111, "long":-0.637820},
                 "waycanguk": {"lat": -5.6585004, "long": 104.4046997},}
# LOCATION = "lab" # change!
LOCATION = "waycanguk"

def birdnet_process_dir(folder_path):
    """Runs BirdNET analysis for all mp3 files in a given directory
    --> Processes several files simultaneously (multiprocessing)"""

    results_dict = {}   # Dictionary to save all recording data

    def on_analyze_directory_complete(recordings):
        """This function runs once analysis is complete
        --> Simply collates & exports the results"""

        # Iterate through all detections - collate into 1 dictionary
        for recording in recordings:
            if recording.error:
                print("Error: ", recording.error_message)
            else:
                chan_name = recording.path.split("/")[-1]   # Extract just the file name (i.e., remove the path)
                results_dict[chan_name] = recording.detections  # Add results to entire dictionary

        # Write the detection results to a JSON file--------------------------------
        # Specify the file path to write the detections
        results_path = folder_path + '/results.json'

        # Write the BirdNET detections to the file
        with open(results_path, 'w') as file:
            json.dump(results_dict, file, indent=4)

    analyzer = Analyzer()

    directory = folder_path
    # Setup analyser parameters...
    if LOCATION == "lab": # Don't specify lat and long for lab tests
        batch = DirectoryMultiProcessingAnalyzer(
            directory,
            analyzers=[analyzer],
            date=datetime(year=2026, month=7, day=14), # use date or week_48
            min_conf=0.4, # Set a minimum confidence of 0.4 (for probability)
            overlap=0 #.1 overlap OFF
        )
    else: # Specify lat and long
        batch = DirectoryMultiProcessingAnalyzer(
            directory,
            analyzers=[analyzer],
            lat=LOCATION_DICT[LOCATION]["lat"], # Manicore Lat & Long
            lon=LOCATION_DICT[LOCATION]["long"],
            date=datetime(year=2026, month=7, day=14), # use date or week_48
            min_conf=0.4, # Set a minimum confidence of 0.4 (for probability)
            overlap=0 #.1 overlap OFF
        )

    # Specify function to run once analysis is complete
    batch.on_analyze_directory_complete = on_analyze_directory_complete
    # Process our analysis (for the entire directory)
    batch.process()

def process_directory(folder_path):
    print("Processing through BirdNET:", folder_path)
    birdnet_process_dir(folder_path)
    print("\nBirdNET processing complete!")

def extract_first_channel(flac_file_path, overwrite=False):
    """
    Extracts the first channel from a multi-channel FLAC file,
    applies a high-pass filter at 1000 Hz, and saves it as an MP3 file
    in a folder named after the original file.
    Uses highest quality MP3 settings.

    Args:
        flac_file_path (str): Path to the FLAC file

    Returns:
        str: Path to the output MP3 file or None if extraction failed
    """
    # Get the filename without extension
    filename = os.path.basename(flac_file_path)
    filename_without_ext = os.path.splitext(filename)[0]

    # Create folder if it doesn't exist
    folder_path = os.path.join(os.path.dirname(flac_file_path), filename_without_ext)
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
        print(f"Created folder: {folder_path}")
    else:
        print(f"Folder already exists: {folder_path}")

    # Define output path
    output_path = os.path.join(folder_path, "original_chan1.mp3")

    # Check if output file already exists
    if not overwrite and os.path.exists(output_path):
        print(f"Output file already exists: {output_path}")
        return output_path

    print(f"Processing file: {flac_file_path}")

    # Load the FLAC file
    try:
        # Using soundfile to read the FLAC file
        data, sample_rate = sf.read(flac_file_path)

        # Check if file is multi-channel
        if len(data.shape) < 2:
            print(f"Error: {filename} is not a multi-channel file")
            return None

        # Extract the first channel
        first_channel = data[:, 0]

        # Apply high-pass filter at 1000 Hz and low-pass filter at 4000 Hz
        print("Applying high-pass filter at 1000 Hz and low-pass filter at 4000 Hz...")
        filtered_channel = apply_highpass_filter(first_channel, sample_rate, cutoff=1000)
        filtered_channel = apply_lowpass_filter(filtered_channel, sample_rate, cutoff=4000)

        # Create a temporary WAV file (pydub works better with wav)
        temp_wav_path = os.path.join(folder_path, "temp.wav")
        sf.write(temp_wav_path, filtered_channel, sample_rate, subtype='PCM_24')  # Use 24-bit for better quality

        # Convert WAV to MP3 using pydub with highest quality settings
        audio = AudioSegment.from_wav(temp_wav_path)

        # Export with highest MP3 quality settings:
        # - bitrate: 320k (highest common MP3 bitrate)
        # - parameters: -q:a 0 sets the internal quality to highest for LAME encoder
        audio.export(
            output_path,
            format="mp3",
            bitrate="320k",
            parameters=["-q:a", "0"]
        )

        # Remove temporary WAV file
        os.remove(temp_wav_path)

        print(f"Successfully extracted first channel with high-pass and low-pass filter to: {output_path} (highest quality)")
        return output_path

    except Exception as e:
        print(f"Error processing {filename}: {str(e)}")
        return None


def apply_highpass_filter(audio_data, sample_rate, cutoff=1000, order=5):
    """
    Apply a high-pass Butterworth filter to audio data.

    Args:
        audio_data (numpy.ndarray): Input audio data
        sample_rate (int): Sample rate of the audio
        cutoff (float): Cutoff frequency in Hz (default: 1000 Hz)
        order (int): Filter order (default: 5)

    Returns:
        numpy.ndarray: Filtered audio data
    """
    # Calculate the Nyquist frequency
    nyquist = 0.5 * sample_rate

    # Normalize the cutoff frequency
    normal_cutoff = cutoff / nyquist

    # Design the Butterworth high-pass filter
    b, a = signal.butter(order, normal_cutoff, btype='high', analog=False)

    # Apply the filter using filtfilt for zero-phase filtering
    filtered_data = signal.filtfilt(b, a, audio_data)

    return filtered_data

def apply_lowpass_filter(audio_data, sample_rate, cutoff=4000, order=5):
    """
    Apply a low-pass Butterworth filter to audio data.
    """
    nyquist = 0.5 * sample_rate
    normal_cutoff = cutoff / nyquist
    b, a = signal.butter(order, normal_cutoff, btype='low', analog=False)
    filtered_data = signal.filtfilt(b, a, audio_data)
    return filtered_data

def process_flac_in_directory(directory_path):
    """
    Processes all .flac files in a directory and extracts first channel

    Args:
        directory_path (str): Path to the directory containing FLAC files

    Returns:
        list: List of paths to the extracted MP3 files
    """
    extracted_files = []
    flac_path = directory_path + ".flac" # Change wav or flac here!
    print(f"Processing Audio file: {flac_path}")
    mp3_path = extract_first_channel(flac_path, True) #overwrite = True
    if mp3_path:
        extracted_files.append(mp3_path)
        print(f"Extracted MP3 file: {mp3_path}")

    return extracted_files

# Main execution
if __name__ == "__main__":
    start_time = time.time()
    parent_dir = "/content/drive/MyDrive/2026 Imperial/07 Way Canguk LabIR/RPiID-00000000058096e0/2026-07-10"
    # parent_dirs = ["/content/drive/MyDrive/2025 Imperial/202508 Lab/01Experiment_LabIR2/0", "/content/drive/MyDrive/2025 Imperial/202508 Lab/01Experiment_LabIR2/1", "/content/drive/MyDrive/2025 Imperial/202508 Lab/01Experiment_LabIR2/2"]

    # Gabungkan timestamp ke nama file
    # log_file_name = f"BirdNET_multidir_logs_{timestamp}.txt"
    # log_file_path = os.path.join(parent_dir, log_file_name)

    # Setup logging
    # os.makedirs(os.path.dirname(log_file_path), exist_ok=True)
    # sys.stdout = open(log_file_path, "w")
    # item_path = "/content/drive/MyDrive/2025 Imperial/202507 Silwood Park/03Experiment_LabIR/06m"
    # Loop untuk semua berkas di dalam folder parent_dir

    for item in os.listdir(parent_dir):
        if item == "normality_plots":
            continue
        else:
            item_path = os.path.join(parent_dir, item)
            if os.path.isdir(item_path):
                print(f"\n=== Processing {item} ===")
                # extracted_mp3_files = process_flac_in_directory(item_path)
                # print(f"Extracted {len(extracted_mp3_files)} MP3 files from directory {item}")
                process_directory(item_path)

    end_time = time.time()
    duration = end_time - start_time
    current_datetime = datetime.now()
    timestamp = current_datetime.strftime("%Y %m %d, %H:%M:%S")
    print(f"Done! Total execution time: {duration:0f} seconds")
    print(f"Finish at {timestamp}")

# Copy original_chan1.mp3

In [ ]:
import os
import shutil

# Tentukan parent_dir (folder asal)
parent_dir = "/content/drive/MyDrive/2025 Imperial/202508 Lab/01Experiment_LabIR"

# Tentukan output_dir (folder tujuan)
output_dir = "/content/drive/MyDrive/2025 Imperial/202508 Lab/01Experiment_SA"

# Buat folder output_dir jika belum ada
os.makedirs(output_dir, exist_ok=True)
print(f"Output directory: {output_dir}")

# Loop melalui setiap item (subfolder) di dalam parent_dir
for item in os.listdir(parent_dir):
    item_path = os.path.join(parent_dir, item)

    # Pastikan item tersebut adalah direktori
    if os.path.isdir(item_path):
        # Tentukan path file source (original_chan1.mp3 di dalam subfolder)
        source_file = os.path.join(item_path, "original_chan1.mp3")

        # Tentukan path file destination (di dalam output_dir)
        destination_file = os.path.join(output_dir, f"{item}/original_chan1.mp3") # Beri nama file sesuai nama subfolder

        # Cek apakah file source ada
        if os.path.exists(source_file):
            try:
                # Salin file
                shutil.copy2(source_file, destination_file)
                print(f"Successfully copied: {source_file} to {destination_file}")
            except Exception as e:
                print(f"Error copying {source_file}: {e}")
        else:
            print(f"Source file not found: {source_file}")

print("\nCopy process finished.")

# Confidence Level Comparison (processed.json)
Works on 7 Sept 2025

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Any, Tuple

# ============================
# KONFIGURASI & KONSTANTA
# ============================

MIN_SAMPLE_SIZE = 0
MONO_COL = '#fc8d59'
BF_COL = '#91bfdb'
CONF_MIN = 0.3  # Minimum confidence threshold for all detections

# ============================
# FUNGSI PEMBACAAN DATA
# ============================

def read_results_from_file(file_path: str) -> Dict[str, Any]:
    """Read the BirdNET detections from a file"""
    try:
        with open(file_path, 'r') as file:
            return json.load(file)
    except Exception as e:
        print(f"❌ Error reading {file_path}: {e}")
        return {}

def extract_unique_bf_detections(results_dict: Dict, conf_thresh: float, beamformed_var: str) -> List[Dict]:
    """Ekstraksi deteksi beamformed unik dengan confidence tertinggi untuk setiap spesies dan waktu."""
    conf_detections = {}

    for channel in results_dict.keys():
        if beamformed_var.lower() in channel.lower():  # Case insensitive
            for det in results_dict[channel]:
                if det.get("confidence", 0) >= conf_thresh:
                    species_name = det.get("common_name", "Unknown")
                    start_time = round(det.get("start_time", 0), 1)
                    prim_key = f"{species_name}_{start_time}"

                    current_conf = det.get("confidence", 0)
                    if (prim_key not in conf_detections or
                        current_conf > conf_detections[prim_key].get("confidence", 0)):

                        det_copy = det.copy()
                        det_copy["start_time"] = start_time
                        det_copy["primary_channel"] = channel
                        conf_detections[prim_key] = det_copy
                        # print(f"✅ Selected: {species_name} at {start_time}s, conf: {current_conf:.3f}")

    print(f"📊 Total unique beamformed detections: {len(conf_detections)}")
    return list(conf_detections.values())

# ============================
# FUNGSI MANIPULASI DATA
# ============================

def update_species(species_dict: Dict, species_name: str, conf: float,
                  start_time: float, file_path: str, max_conf_chan: str = "") -> Dict:
    """Update confidence list dan tambahkan channel."""
    if species_name in species_dict:
        species_data = species_dict[species_name]
        species_data["conf_list"].append(conf)
        species_data["start_time_list"].append(start_time)
        identifier = f"{file_path}, {max_conf_chan or 'original_chan1.mp3'}, {start_time}"
        species_data["mp3_identifier"].append(identifier)
        if max_conf_chan:
            species_data.setdefault("max_conf_chan_list", []).append(max_conf_chan)
    return species_dict

def add_new_species(species_dict: Dict, species_name: str, conf: float,
                   start_time: float, file_path: str, max_conf_chan: str = "") -> Dict:
    """Menambahkan data untuk spesies baru ke dictionary."""
    identifier = f"{file_path}, {max_conf_chan or 'original_chan1.mp3'}, {start_time}"

    species_dict[species_name] = {
        "conf_list": [conf],
        "start_time_list": [start_time],
        "mp3_identifier": [identifier]
    }

    if max_conf_chan:
        species_dict[species_name]["max_conf_chan_list"] = [max_conf_chan]

    return species_dict

def process_results_dict(results_dict: Dict, overall_results: Dict, file_path: str,
                        beamformed_var: str) -> Dict:
    """Proses results.json untuk mencatat semua deteksi mono_channel dan beamformed secara independen."""

    # Proses mono detections
    mono_data = results_dict.get("original_chan1.mp3", [])
    print(f"🔊 Processing {len(mono_data)} mono detections")

    for mono_detection in mono_data:
        species_name = mono_detection.get("common_name", "Unknown")
        start_time = round(mono_detection.get("start_time", 0), 1)
        mono_conf = mono_detection.get("confidence", 0)

        if mono_conf >= CONF_MIN:
            if species_name in overall_results["mono_channel"]:
                overall_results["mono_channel"] = update_species(
                    overall_results["mono_channel"], species_name, mono_conf, start_time, file_path
                )
            else:
                overall_results["mono_channel"] = add_new_species(
                    overall_results["mono_channel"], species_name, mono_conf, start_time, file_path
                )

    # Proses beamformed detections
    bf_detections = extract_unique_bf_detections(results_dict, CONF_MIN, beamformed_var)
    print(f"📡 Processing {len(bf_detections)} beamformed detections")

    for bf_det in bf_detections:
        species_name = bf_det.get("common_name", "Unknown")
        start_time = bf_det.get("start_time", 0)
        max_bf_conf = bf_det.get("confidence", 0)
        max_conf_chan = bf_det.get("primary_channel", "")

        if species_name in overall_results["beamformed"]:
            overall_results["beamformed"] = update_species(
                overall_results["beamformed"], species_name, max_bf_conf, start_time, file_path, max_conf_chan
            )
        else:
            overall_results["beamformed"] = add_new_species(
                overall_results["beamformed"], species_name, max_bf_conf, start_time, file_path, max_conf_chan
            )

    return overall_results

# ============================
# FUNGSI OUTPUT & METRIK
# ============================

def add_conf_metrics(processed_dict: Dict) -> Dict:
    """Menghitung metrik untuk confidence di setiap kanal."""
    for chan, chan_data in processed_dict.items():
        for species, species_data in chan_data.items():
            conf_list = species_data.get("conf_list", [])
            count = len(conf_list)

            if conf_list:
                mean = round(np.mean(conf_list), 3)
                median = round(np.median(conf_list), 3)
                stdev = round(np.std(conf_list), 3) if len(conf_list) > 1 else 0.0
                max_conf = round(np.max(conf_list), 3)
            else:
                mean = median = stdev = max_conf = 0.0

            species_data.update({
                "conf_avg": mean,
                "conf_median": median,
                "conf_stdev": stdev,
                "conf_max": max_conf,
                "count": count
            })

    return processed_dict

def write_processed_to_file(processed_dict: Dict, file_path: str) -> bool:
    """Menulis hasil yang diproses ke file."""
    try:
        with open(file_path, 'w') as file:
            json.dump(processed_dict, file, indent=4, ensure_ascii=False)
        print(f"✅ Written processed results to {file_path}")
        return True
    except Exception as e:
        print(f"❌ Error writing to {file_path}: {e}")
        return False

def determine_location(dir_path: str) -> str:
    """Menentukan lokasi berdasarkan path."""
    path_lower = dir_path.lower()
    if "manicore" in path_lower:
        return "Manicore"
    elif "silwood" in path_lower:
        return "Silwood"
    else:
        return "Lab: Speaker Sphere"

# ============================
# FUNGSI UTAMA
# ============================

def process_experiment_folder(experiment_dir: str, item: str, beamformed_var: str) -> bool:
    """Memproses satu folder eksperimen."""
    dir_path = os.path.join(experiment_dir, item)

    if not os.path.isdir(dir_path):
        return False

    print(f"\n{'='*50}")
    print(f"🚀 Processing {item}")
    print(f"📂 Path: {dir_path}")

    # Tentukan lokasi
    location = determine_location(dir_path)
    print(f"📍 Location: {location}")
    print(f"📡 Beamformed variable: {beamformed_var}")

    # Inisialisasi hasil
    processed_results = {"mono_channel": {}, "beamformed": {}}

    # Cari dan proses semua results.json
    results_files_found = 0

    for root, dirs, files in os.walk(dir_path, topdown=False):
        for name in files:
            if name == "results.json":
                results_path = os.path.join(root, name)
                print(f"📄 Reading: {results_path}")

                try:
                    current_results_dict = read_results_from_file(results_path)
                    if current_results_dict:
                        print(f"   🔍 Keys found: {list(current_results_dict.keys())}")
                        processed_results = process_results_dict(
                            current_results_dict, processed_results, results_path, beamformed_var
                        )
                        results_files_found += 1
                except Exception as e:
                    print(f"   ❌ Error processing {results_path}: {e}")

    if results_files_found == 0:
        print("⚠️  No results.json files found!")
        return False

    # Tambahkan metrik dan simpan
    processed_results = add_conf_metrics(processed_results)
    output_file = os.path.join(dir_path, "processed.json")

    return write_processed_to_file(processed_results, output_file)

def main():
    """Main execution function."""
    # parent_dir = "/content/drive/MyDrive/2025 Imperial/202508 Lab/Experiment_LabIR"
    parent_dir = "/content/drive/MyDrive/2026 Imperial/07 Way Canguk LabIR/RPiID-00000000058096e0/2026-07-10"

    # Ekstrak beamformed variable dari nama folder
    beamformed_var = '_LabIR'
    print(f"🎯 Beamformed variable extracted: {beamformed_var}")

    print(f"📁 Processing experiment: {parent_dir}")

    # Proses setiap item dalam parent directory
    items_processed = 0
    items_total = 0

    for item in next(os.walk(parent_dir))[1]:
        if item == "normality_plots":
            continue
        else:
            items_total += 1
            if process_experiment_folder(parent_dir, item, beamformed_var):
                items_processed += 1

    # Ringkasan
    print(f"\n{'='*60}")
    print(f"🏁 PROCESSING COMPLETE")
    print(f"{'='*60}")
    print(f"Total folders: {items_total}")
    print(f"Successfully processed: {items_processed}")
    print(f"Failed: {items_total - items_processed}")

if __name__ == "__main__":
    main()

# Recap Process {target_species}
24 Juli

In [ ]:
import json
import numpy as np
import os
parent_dir = "/content/drive/MyDrive/2025 Imperial/202509 Silwood Park/Experiment_SA"
# SPECIES = ["Eurasian Blue Tit"]
# SPECIES = ["Eurasian Blue Tit", "Rose-ringed Parakeet"]
SPECIES = ["Large Wren-Babbler", "Rufous-tailed Shama"]
# SPECIES = ["Eurasian Blue Tit", "Common Buzzard", "Eurasian Green Woodpecker", "Eurasian Wren", "Eurasian Magpie", "Eurasian Jackdaw", "Common Chiffchaff", "Eurasian Blackcap"]
for item in os.listdir(parent_dir):
    DIR_PATH = os.path.join(parent_dir, item)
    if os.path.isdir(DIR_PATH):
        print(f"\n=== Processing {item} ===")
        for target_species in SPECIES:
            # for degree in range(0, 360, 60):
            # Path ke file JSON
            file_path = os.path.join(DIR_PATH, "processed.json")

            # Baca file JSON
            try:
                with open(file_path, 'r') as f:
                    data = json.load(f)
                print(f"Successfully read {file_path}")
            except Exception as e:
                print(f"Error reading {file_path}: {e}")
                continue

            # Ekstrak data untuk Rose-ringed Parakeet
            mono_data = data.get('mono_channel', {}).get(target_species, {})
            beam_data = data.get('beamformed', {}).get(target_species, {})

            # Ekstrak nilai statistik
            mono_stats = {
                'avg': mono_data.get('conf_avg'),
                'median': mono_data.get('conf_median'),
                'stdev': mono_data.get('conf_stdev'),
                'max': mono_data.get('conf_max'),
                'count': mono_data.get('count'),
                'conf_list': mono_data.get('conf_list', [])
            }

            beam_stats = {
                'avg': beam_data.get('conf_avg'),
                'median': beam_data.get('conf_median'),
                'stdev': beam_data.get('conf_stdev'),
                'max': beam_data.get('conf_max'),
                'count': beam_data.get('count'),
                'conf_list': beam_data.get('conf_list', [])
            }

            # Hitung nilai minimum
            mono_stats['min'] = min(mono_stats['conf_list']) if mono_stats['conf_list'] else None
            beam_stats['min'] = min(beam_stats['conf_list']) if beam_stats['conf_list'] else None

            # Simpan statistik ke file "processed-stats.json"
            stats_data = {
                "mono_channel": mono_stats,
                "beamformed": beam_stats
            }

            # Ganti spasi dengan garis bawah untuk nama file
            safe_species_name = target_species.replace(" ", "_")
            # Buat path untuk file "processed-stats.json" _{degree:03d}
            output_path = os.path.join(DIR_PATH, f"processed_{safe_species_name}.json")

            # Simpan data statistik ke file JSON
            try:
                with open(output_path, 'w') as outfile:
                    json.dump(stats_data, outfile, indent=4)
                print(f"Statistical information saved to: {output_path}")
            except Exception as e:
                print(f"Error writing to {output_path}: {e}")

# Recap Combined Analysis Target Species - OLD

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

species = ["Eurasian Blue Tit", "Rose-ringed Parakeet"]
for target_species in species
    for degree in range(180,240,60):

        # experiment_folders = [f"02m_{degree:03d}", f"04m_{degree:03d}", f"08m_{degree:03d}", f"16m_{degree:03d}"]
        # display_labels = ["01m", "02m", "04m", "06m", "08m", "10m", "12m", "14m", "16m", "18m", "20m", "22m", "24m"]
        display_labels = ["01m", "02m", "04m", "08m", "16m", "32m", "64m"]
        experiment_folders = display_labels

        # Inisialisasi list untuk menyimpan data
        mono_confidence_scores = []
        mono_stdev = []
        mono_max = []
        mono_min = []
        mono_median = []

        beamformed_confidence_scores = []
        beamformed_stdev = []
        beamformed_max = []
        beamformed_min = []
        beamformed_median = []

        # Inisialisasi list untuk menyimpan counts
        mono_counts = []
        beamformed_counts = []

        # Loop melalui setiap folder eksperimen untuk mengekstrak data
        for folder in experiment_folders:
            stats_json_path = os.path.join(parent_dir, folder, f"processed_{target_species}_{degree:03d}.json")
            counts_json_path = os.path.join(parent_dir, folder, "processed.json")

            # Ekstraksi data confidence scores
            try:
                with open(stats_json_path, 'r') as file:
                    data = json.load(file)

                    # Ekstrak statistik untuk mono_channel
                    mono_avg = data["mono_channel"].get("avg", 0)
                    mono_std = data["mono_channel"].get("stdev", 0)
                    mono_maximum = data["mono_channel"].get("max", 0)
                    mono_minimum = data["mono_channel"].get("min", 0)
                    mono_med = data["mono_channel"].get("median", 0)

                    # Ekstrak statistik untuk beamformed
                    beamformed_avg = data["beamformed"].get("avg", 0)
                    beamformed_std = data["beamformed"].get("stdev", 0)
                    beamformed_maximum = data["beamformed"].get("max", 0)
                    beamformed_minimum = data["beamformed"].get("min", 0)
                    beamformed_med = data["beamformed"].get("median", 0)

                    # Tambahkan ke list
                    mono_confidence_scores.append(mono_avg if mono_avg is not None else 0)
                    mono_stdev.append(mono_std if mono_std is not None else 0)
                    mono_max.append(mono_maximum if mono_maximum is not None else 0)
                    mono_min.append(mono_minimum if mono_minimum is not None else 0)
                    mono_median.append(mono_med if mono_med is not None else 0)

                    beamformed_confidence_scores.append(beamformed_avg if beamformed_avg is not None else 0)
                    beamformed_stdev.append(beamformed_std if beamformed_std is not None else 0)
                    beamformed_max.append(beamformed_maximum if beamformed_maximum is not None else 0)
                    beamformed_min.append(beamformed_minimum if beamformed_minimum is not None else 0)
                    beamformed_median.append(beamformed_med if beamformed_med is not None else 0)

            except FileNotFoundError:
                print(f"File confidence scores tidak ditemukan: {stats_json_path}")
                # Tambahkan nilai 0 untuk semua statistik jika file tidak ditemukan
                mono_confidence_scores.append(0)
                mono_stdev.append(0)
                mono_max.append(0)
                mono_min.append(0)
                mono_median.append(0)

                beamformed_confidence_scores.append(0)
                beamformed_stdev.append(0)
                beamformed_max.append(0)
                beamformed_min.append(0)
                beamformed_median.append(0)
            except (json.JSONDecodeError, KeyError) as e:
                print(f"Error pada {stats_json_path}: {e}")
                # Tambahkan nilai 0 untuk semua statistik jika terjadi error
                mono_confidence_scores.append(0)
                mono_stdev.append(0)
                mono_max.append(0)
                mono_min.append(0)
                mono_median.append(0)

                beamformed_confidence_scores.append(0)
                beamformed_stdev.append(0)
                beamformed_max.append(0)
                beamformed_min.append(0)
                beamformed_median.append(0)

            # Ekstraksi data counts
            try:
                with open(counts_json_path, 'r') as file:
                    data = json.load(file)

                    # Extract Eurasian Blue Tit count from mono_channel data
                    mono_count = 0
                    if "mono_channel" in data and target_species in data["mono_channel"]:
                        mono_count = data["mono_channel"][target_species].get("count", 0)
                    mono_counts.append(mono_count)

                    # Extract Eurasian Blue Tit count from beamformed data
                    beamformed_count = 0
                    if "beamformed" in data and target_species in data["beamformed"]:
                        beamformed_count = data["beamformed"][target_species].get("count", 0)
                    beamformed_counts.append(beamformed_count)

            except FileNotFoundError:
                print(f"File counts tidak ditemukan: {counts_json_path}")
                mono_counts.append(0)
                beamformed_counts.append(0)
            except (json.JSONDecodeError, KeyError) as e:
                print(f"Error pada {counts_json_path}: {e}")
                mono_counts.append(0)
                beamformed_counts.append(0)


        # Buat figure dengan 2 subplot
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True, gridspec_kw={'height_ratios': [3, 1]})

        # Tambahkan judul utama di atas figure, pecah menjadi 2 baris
        fig.suptitle(f'BirdNET Analysis: {target_species}\nMono Channel vs. Beamformed Across Distance Experiments',
                    fontsize=16, y=1.02)

        # Posisi bar
        x = np.arange(len(display_labels))
        width = 0.35  # lebar bar

        # --- SUBPLOT 1: Bar chart untuk confidence scores dengan error bars ---
        # Plot bar untuk mono_channel dengan error bars menunjukkan standar deviasi
        mono_bars_1 = ax1.bar(x - width/2, mono_confidence_scores, width, label='Mono Channel', color='skyblue', alpha=0.8, yerr=mono_stdev, capsize=5) #edgecolor='blue',

        # Plot bar untuk beamformed dengan error bars menunjukkan standar deviasi
        beam_bars_1 = ax1.bar(x + width/2, beamformed_confidence_scores, width, label='Beamformed', color='lightgreen', alpha=0.8, yerr=beamformed_stdev, capsize=5) #edgecolor='green',

        # Tambahkan overlay line chart
        ax1.plot(x - width/2, mono_confidence_scores, 'o--', color='blue', alpha=0.7, linewidth=1.5, markersize=8)
        ax1.plot(x + width/2, beamformed_confidence_scores, 's--', color='green', alpha=0.7, linewidth=1.5, markersize=8)

        # Fungsi untuk menambahkan label nilai di atas bar - hanya jika tidak nol
        def add_labels_m(bars, values, ax):
            for i, bar in enumerate(bars):
                height = bar.get_height()
                if height > 0:
                    ax.text(bar.get_x() , height-0.06,
                            f'{height:.2f}', ha='right', va='center', fontsize=11, color='blue', alpha=0.9)
                # elif height == 0:
                #     ax.text(bar.get_x() , height+0.04,
                #             f'{height:.2f}', ha='right', va='center', fontsize=11, color='blue', alpha=0.9)

        def add_labels_bf(bars, values, ax):
            for i, bar in enumerate(bars):
                height = bar.get_height()
                if height > 0:  # Hanya tampilkan jika nilai >= nol
                    ax.text(bar.get_x() + bar.get_width(), height+0.06,
                            f'{height:.2f}', ha='left', va='center', fontsize=11, color='green', alpha=0.9)

        # Tambahkan label nilai
        add_labels_m(mono_bars_1, mono_confidence_scores, ax1)
        add_labels_bf(beam_bars_1, beamformed_confidence_scores, ax1)

        for i in range(len(mono_confidence_scores)):
            if mono_confidence_scores[i] == 0 and beamformed_confidence_scores[i] > 0:
                        label = "Detected by\n beamforming"
                        color = 'darkgreen'
                        y_pos = 0.3 #(beamformed_confidence_scores[i])/3
                        ax1.annotate(label,
                                    xy=(x[i], y_pos),  # Annotation point stays the same
                                    ha='center',       # Keep center-aligned horizontally
                                    va='bottom',       # Keep base-aligned vertically
                                    fontsize=10,
                                    color=color,
                                    bbox=dict(boxstyle='round,pad=0.2', fc='#ffffe8', ec='gray', lw=0.5, alpha=0.9),
                                    rotation=90,       # Rotate 90° counter-clockwise
                                    rotation_mode='anchor')  # Rotate around anchor point

            elif mono_confidence_scores[i] == 0 and beamformed_confidence_scores[i] == 0:
                label = "Both no detection"
                color = 'darkred'
                y_pos = 0.3 #(mono_confidence_scores[i])/3
                ax1.annotate(label,
                                xy=(x[i], y_pos),
                                ha='center',
                                va='bottom',
                                fontsize=10,
                                color=color,
                                bbox=dict(boxstyle='round,pad=0.2', fc='#ffffe8', ec='gray', lw=0.5, alpha=0.9),
                                rotation=90,          # Rotate 90° counter-clockwise
                                rotation_mode='anchor')  # Rotate around anchor point

            elif mono_confidence_scores[i] > 0 and beamformed_confidence_scores[i] > 0:
                improvement = ((beamformed_confidence_scores[i] - mono_confidence_scores[i]) / mono_confidence_scores[i]) * 100
                if abs(improvement) > 1:  # Hanya tampilkan jika perubahan signifikan (>1%)
                    y_pos = 1.08 #(mono_confidence_scores[i])/2 + 0.05

                    if improvement > 0:
                        label = f"+{improvement:.0f}%" + ("↑")
                        color = 'darkgreen'
                    else:
                        label = f"{improvement:.0f}%" + ("↓")
                        color = 'darkred'

                    ax1.annotate(label,
                                xy=(x[i], y_pos),
                                ha='center', va='bottom',
                                fontsize=10, color=color,
                                bbox=dict(boxstyle='round,pad=0.2', fc='#ffffe8', ec='gray', lw=0.5, alpha=0.9))

        # --- Tambahkan Significance Stars (asumsi chi_square_results tersedia) ---
        # try:
        #     # Contoh chi_square_results (harus diganti dengan data aktual Anda)
        #     chi_square_results = [
        #         {'snr_level': '+30dB', 'p_value': 0.0000065882196, 'sig_stars': '***'},
        #         {'snr_level': '+20dB', 'p_value': 0.0680451844729, 'sig_stars': ''},
        #         {'snr_level': '+10dB', 'p_value': 0.1179685146082, 'sig_stars': ''},
        #         {'snr_level': '0dB', 'p_value': 0.0680451844729, 'sig_stars': '**'},
        #         {'snr_level': '-10dB', 'p_value': 0.0021255970744, 'sig_stars': '***'},
        #         {'snr_level': '-20dB', 'p_value': 0.0000001509762, 'sig_stars': '***'},
        #     ]
        #     t_test_results = [
        #         {'snr_level': '+30dB', 'p_value': 0.00000000000000000000000000000007, 'sig_stars': '***'},
        #         {'snr_level': '+20dB', 'p_value': 0.00000000000000000000000000000002, 'sig_stars': '***'},
        #         {'snr_level': '+10dB', 'p_value': 0.00000000000000000000000000831849, 'sig_stars': '***'},
        #         {'snr_level': '0dB', 'p_value': 0.00000000000000000371130713932336, 'sig_stars': '***'},
        #         {'snr_level': '-10dB', 'p_value': 0.00000000000000000099821666952706, 'sig_stars': '***'},
        #         {'snr_level': '-20dB', 'p_value': 0.00216515394534222000000000000000, 'sig_stars': '**'},
        #     ]

        #     # Tambahkan bintang pada subplot 2 (counts)
        #     for res in [r for r in t_test_results if not np.isnan(r['p_value']) and r['p_value'] < 0.05]:
        #         try:
        #             idx = display_labels.index(res['snr_level'])
        #             max_height = max(mono_counts[idx], beamformed_counts[idx])
        #             ax2.text(x[idx], max_height + 10, res['sig_stars'],
        #                     ha='center', fontweight='bold', fontsize=12)
        #         except ValueError:
        #             print(f"Warning: SNR level {res['snr_level']} not found in display_labels")

        #     # Tambahkan bintang pada subplot 1 (confidence scores)
        #     for res in [r for r in chi_square_results if not np.isnan(r['p_value']) and r['p_value'] < 0.1]:
        #         try:
        #             idx = display_labels.index(res['snr_level'])
        #             max_height = max(mono_confidence_scores[idx], beamformed_confidence_scores[idx])
        #             ax1.text(x[idx], max_height + 0.1, res['sig_stars'],
        #                     ha='center', fontweight='bold', fontsize=12)
        #         except ValueError:
        #             print(f"Warning: SNR level {res['snr_level']} not found in display_labels")
        # except NameError:
        #     print("Warning: chi_square_results or t_test_results not defined. Significance stars will not be displayed.")

        # Tambahkan label subplot
        ax1.text(-0.08, 0.5, '(a)', transform=ax1.transAxes, fontsize=16)
        ax1.set_ylabel('Mean Confidence Score', fontsize=14)
        ax1.grid(axis='y', linestyle='--', alpha=0.7)
        ax1.legend(fontsize=12, loc='upper right')
        ax1.set_ylim(0, 1.19)

        # --- SUBPLOT 2: Bar chart untuk counts ---
        mono_bars_2 = ax2.bar(x - width/2, mono_counts, width, label='Mono Channel',
                            color='skyblue', alpha=0.8, linewidth=1.5)
        beam_bars_2 = ax2.bar(x + width/2, beamformed_counts, width, label='Beamformed',
                            color='lightgreen', alpha=0.8, linewidth=1.5)

        # Baris baru: Pastikan sumbu Y bilangan bulat
        ax2.yaxis.set_major_locator(plt.MaxNLocator(integer=True))  # <-- Baris baru untuk bilangan bulat

        # Add count values above bars - hanya jika tidak nol
        def add_count_labels(bars, values, ax):
            for i, bar in enumerate(bars):
                height = bar.get_height()
                if height > 0:  # Hanya tampilkan jika nilai tidak nol
                    ax.annotate(f'{int(height)}',
                              xy=(bar.get_x() + bar.get_width()/2 , height + 7),
                              ha='center', va='center',
                              fontsize=11)

        add_count_labels(mono_bars_2, mono_counts, ax2)
        add_count_labels(beam_bars_2, beamformed_counts, ax2)

        # Add connecting lines for visual tracking
        ax2.plot(x - width/2, mono_counts, 'o--', color='blue', alpha=0.5, linewidth=1.5, markersize=8)
        ax2.plot(x + width/2, beamformed_counts, 's--', color='green', alpha=0.5, linewidth=1.5, markersize=8)

        # Add detection improvement percentages dengan posisi yang konsisten dengan subplot 1
        for i in range(len(mono_counts)):
            if mono_counts[i] > 0 and beamformed_counts[i] > 0:
                improvement = ((beamformed_counts[i] - mono_counts[i]) / mono_counts[i]) * 100
                if improvement != 0:
                    # Gunakan rumus yang sama untuk konsistensi dengan subplot 1
                    # Untuk count kita gunakan posisi 1/3 dari tinggi mono
                    y_pos = 30

                    ax2.annotate(f"{improvement:.0f}% improvement" if improvement > 0 else f"{abs(improvement):.0f}% decrease",
                              xy=(x[i], y_pos),
                              ha='center', va='center',
                              fontsize=11, color='darkred',
                              bbox=dict(boxstyle='round,pad=0.3', fc='#ffeeee', ec='none', alpha=1))

        # Tambahkan label subplot
        ax2.text(-0.08, 0.5, '(b)', transform=ax2.transAxes, fontsize=16)
        ax2.set_xlabel(f'Distance, from Angle = {degree} degrees', fontsize=14)
        ax2.set_ylabel('Detections', fontsize=14)
        ax2.set_xticks(x)
        ax2.set_xticklabels(display_labels, fontsize=12)
        # ax2.legend(fontsize=12, loc='upper right')
        ax2.grid(axis='y', linestyle='--', alpha=0.7)

        # Set y-axis to start at 0 and have some headroom above the highest bar
        max_count = max(max(mono_counts, default=0), max(beamformed_counts, default=0))
        ax2.set_ylim(0, max_count * 1.1)

        # Atur layout dan simpan grafik dengan jarak antar subplot yang lebih kecil
        plt.tight_layout()
        plt.subplots_adjust(top=0.95, hspace=0.02)
        figpng = os.path.join(parent_dir, f"{target_species}_{degree:03d}_combined_analysis.png")
        plt.savefig(figpng, dpi=300, bbox_inches='tight')

        # Tampilkan grafik
        plt.show()

# Recap Combined Analysis Specific Target Species - NEW
24 July 2025

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from matplotlib.patches import Rectangle
parent_dir = "/content/drive/MyDrive/2025 Imperial/202508 Lab/Experiment_LabIR"
DISPLAY_LABELS = ["+30dB", "+20dB", "+10dB", "0dB", "-10dB", "-20dB", "-30dB"]
EXPERIMENT_FOLDERS = ["0", "1", "2", "3", "4", "5", "6"]

# EXPERIMENT_FOLDERS = ["01m", "02m", "04m", "08m", "16m", "32m", "64m"]
# DISPLAY_LABELS = EXPERIMENT_FOLDERS

# ----------------------------------------------------------------------
# 4 methods that will appear in the plot
# ----------------------------------------------------------------------
METHODS = [
    {"key": "mono",         "label": "Mono",  "color": "skyblue"},
    {"key": "SA",           "label": "SA",            "color": "lightcoral"},
    {"key": "LabIR",        "label": "Lab‑IR‑BF",     "color": "#fcc006"},
    {"key": "SPIR",         "label": "SP‑IR‑BF",      "color": "lightgreen"},
]

# Mapping from method key → folder that contains its JSON files
METHOD_FOLDERS = {
    "mono"  : parent_dir,                                   # baseline lives in the root folder
    "SA"    : "/content/drive/MyDrive/2025 Imperial/202508 Lab/Experiment_SA",
    "LabIR" : "/content/drive/MyDrive/2025 Imperial/202508 Lab/Experiment_LabIR",
    "SPIR"  : "/content/drive/MyDrive/2025 Imperial/202508 Lab/Experiment_SPIR",
}
experiment_colors = {
    "Mono": "skyblue",
    "SA": "lightcoral",
    "Lab-IR-BF": "#fcc006",
    "SP-IR-BF": "lightgreen"
}
legend_experiments = ["Mono", "SA", "Lab-IR-BF", "SP-IR-BF"]


def load_significance_data(parent_dir, species):
    """Load significance levels from JSON file."""
    json_path = os.path.join(parent_dir, f"stats_significance_{species.replace(' ', '_')}.json")
    if os.path.exists(json_path):
        try:
            with open(json_path, 'r') as f:
                print(f"Loading significance data for {species} from {json_path}")
                data = json.load(f)
                # print(f"Significance data loaded: {data}")
                return data
        except json.JSONDecodeError as e:
            print(f"Error decoding {json_path} for {species}: {e}. Using empty significance data.")
            return {}
    else:
        print(f"Significance file {json_path} for {species} not found. Using empty significance data.")
        return {}

def initialize_data_lists():
    """Create a dict that will store every statistic for each method."""
    data = {}
    for m in METHODS:
        data[m["key"]] = {
            "confidence_scores": [],   # mean confidence
            "stdev"            : [],   # confidence SD
            "max"              : [],   # optional, not plotted
            "min"              : [],   # optional, not plotted
            "median"           : [],   # optional, not plotted
            "counts"           : []    # detection counts
        }
    data["species"] = ""           # will be filled later
    return data

def extract_stats_data(json_path, data_type, species):
    """Extract statistical data from JSON file with detailed error logging."""
    if not os.path.exists(json_path):
        print(f"File not found: {json_path}")
        return {'avg': 0, 'stdev': 0, 'max': 0, 'min': 0, 'median': 0}

    try:
        with open(json_path, 'r') as file:
            data = json.load(file)
            # print(f"Loaded JSON from {json_path}: {data}")

            channel_data = data.get(data_type, {})
            if not channel_data:
                print(f"No '{data_type}' data found in {json_path}")
                return {'avg': 0, 'stdev': 0, 'max': 0, 'min': 0, 'median': 0}

            return {
                'avg': channel_data.get('avg', 0) or 0,
                'stdev': channel_data.get('stdev', 0) or 0,
                'max': channel_data.get('max', 0) or 0,
                'min': channel_data.get('min', 0) or 0,
                'median': channel_data.get('median', 0) or 0
            }
    except json.JSONDecodeError as e:
        print(f"JSON decode error in {json_path}: {e}")
        return {'avg': 0, 'stdev': 0, 'max': 0, 'min': 0, 'median': 0}
    except Exception as e:
        print(f"Unexpected error in {json_path}: {e}")
        return {'avg': 0, 'stdev': 0, 'max': 0, 'min': 0, 'median': 0}

def extract_counts_data(json_path, data_type, species):
    """Extract count data from JSON file."""
    species = species.replace("_", " ")
    if not os.path.exists(json_path):
        print(f"File not found: {json_path}")
        return 0

    try:
        with open(json_path, 'r') as file:
            data = json.load(file)
            channel_data = data.get(data_type, {})
            return channel_data.get(species, {}).get('count', 0) or 0
    except (json.JSONDecodeError, KeyError) as e:
        print(f"Error processing {json_path}: {e}")
        return 0

def add_value_labels(bars, values, ax, offset, color, ha):
    """Add value labels above bars if non-zero."""
    for i, bar in enumerate(bars):
        height = bar.get_height()
        if height > 0:
            ax.text(
                bar.get_x() + bar.get_width() / 2 + offset,
                height + (0.06 if offset > 0 else -0.06),
                f'{height:.2f}',
                ha=ha,
                va='center',
                fontsize=11,
                color=color,
                alpha=0.9
            )

def add_count_labels(bars, values, ax):
    """Add count labels above bars if non-zero."""
    for i, bar in enumerate(bars):
        height = bar.get_height()
        if height >= 0:
            ax.annotate(
                f'{int(height)}',
                xy=(bar.get_x() + bar.get_width() / 2, height + 15),
                # height + 0.07*height
                ha='center',
                va='center',
                fontsize=11
            )

def add_significance_labels(ax, x, significance_data, data, folder_index, y_offset=0.1):
    """Add significance stars above bars based on statistical tests from JSON."""
    folder = EXPERIMENT_FOLDERS[folder_index]
    distance = DISPLAY_LABELS[folder_index].replace(" m", "m")
    print(f"Adding significance labels for {distance} (species: {data.get('species', 'unknown')})")
    print(f"Significance data: {significance_data}")
    # Get significance from relevant tests
    sig_wilcoxon = significance_data.get('wilcoxon', {}).get(distance, {}).get('sig_stars', '')
    sig_mcnemar = significance_data.get('mcnemar', {}).get(distance, {}).get('sig_stars', '')
    sig_t_test = significance_data.get('t_test', {}).get(distance, {}).get('sig_stars', '')
    sig_chi_square = significance_data.get('chi_square', {}).get(distance, {}).get('sig_stars', '')
    # Determine primary test based on axis type
    if 'Confidence' in ax.get_ylabel():
        combined_sig = sig_wilcoxon if sig_wilcoxon else sig_t_test  # Prioritize Wilcoxon, fallback to t-test
    else:  # Counts
        combined_sig = sig_mcnemar if sig_mcnemar else sig_chi_square  # Prioritize McNemar, fallback to Chi-square
    if combined_sig:
        print(f"Adding significance label for {distance}: {combined_sig}")
        # Adjust y-position based on axis
        y_pos = 1.0 + y_offset if 'Confidence' in ax.get_ylabel() else max(data['beamformed']['counts'][folder_index], data['mono']['counts'][folder_index]) + 30
        ax.text(
            x[folder_index],
            y_pos,
            combined_sig,
            ha='center',
            va='bottom',
            fontsize=10,
            color='black',
            weight='bold'
        )

def add_annotations(ax, x, mono_scores, beamformed_scores):
    """Add detection and improvement annotations."""
    for i in range(len(mono_scores)):
        y_pos = 0.3
        if mono_scores[i] == 0 and beamformed_scores[i] > 0:
            ax.annotate(
                "Detected by\n beamforming",
                xy=(x[i] - 0.1, y_pos),
                ha='center',
                va='bottom',
                fontsize=11,
                color='darkgreen',
                bbox=dict(boxstyle='round,pad=0.2', fc='#ffffe8', ec='gray', lw=0.5, alpha=0.9),
                rotation=90,
                rotation_mode='anchor'
            )
        elif mono_scores[i] == 0 and beamformed_scores[i] == 0:
            ax.annotate(
                "Both no detection",
                xy=(x[i], y_pos),
                ha='center',
                va='bottom',
                fontsize=11,
                color='darkred',
                bbox=dict(boxstyle='round,pad=0.2', fc='#ffffe8', ec='gray', lw=0.5, alpha=0.9),
                rotation=90,
                rotation_mode='anchor'
            )
        elif mono_scores[i] > 0 and beamformed_scores[i] > 0:
            improvement = ((beamformed_scores[i] - mono_scores[i]) / mono_scores[i]) * 100
            if abs(improvement) > 1:
                y_pos = 0.2
                label = f"+{improvement:.0f}%↑" if improvement > 0 else f"{improvement:.0f}%↓"
                color = 'darkgreen' if improvement > 0 else 'darkred'
                ax.annotate(
                    label,
                    xy=(x[i], y_pos),
                    ha='center',
                    va='bottom',
                    fontsize=11,
                    color=color,
                    bbox=dict(boxstyle='round,pad=0.2', fc='#ffffe8', ec='gray', lw=0.5, alpha=0.9)
                )

def add_count_improvement(ax, x, mono_counts, beamformed_counts):
    """Add count improvement percentages."""
    max_height = 0
    for i in range(len(mono_counts)):
        if max(beamformed_counts[i], mono_counts[i]) > max_height:
            max_height = max(beamformed_counts[i], mono_counts[i])
        else:
            continue
        print(f"{i} Max height: {max_height}")

    for i in range(len(mono_counts)):
        if mono_counts[i] > 0 and beamformed_counts[i] > 0:
            improvement = ((beamformed_counts[i] - mono_counts[i]) / mono_counts[i]) * 100

            y_pos = max_height/3 + 20

            if improvement != 0:
                color = 'darkgreen' if improvement > 0 else 'darkred'
                ax.annotate(
                    f"+{improvement:.0f}%↑" if improvement > 0 else f"{improvement:.0f}%↓",
                    xy=(x[i], y_pos),
                    ha='center',
                    va='center',
                    fontsize=11,
                    color=color,
                    bbox=dict(boxstyle='round,pad=0.2', fc='#ffffe8', ec='gray', lw=0.5, alpha=0.9)
                )

def create_plots(parent_dir, species):
    """Create the two‑panel figure (confidence + counts) for all four methods."""
    data = initialize_data_lists()
    data["species"] = species

    # ------------------------------------------------------------------
    # 1)  Read statistics (mean, stdev, …) for every method / distance
    # ------------------------------------------------------------------
    for folder in EXPERIMENT_FOLDERS:               # 01m, 02m, …, 64m
        sp_name = species.replace(" ", "_")         # JSON uses underscores

        for m in METHODS:
            # Build the correct path for this method:
            #   - mono  → <parent_dir>/<folder>/processed_<species>.json
            #   - others → <METHOD_FOLDERS[key]>/<folder>/processed_<species>.json
            if m["key"] == "mono":
                stats_json_path = os.path.join(parent_dir, folder,
                                              f"processed_{sp_name}.json")
                counts_json_path = os.path.join(parent_dir, folder,
                                               "processed.json")
            else:
                base = METHOD_FOLDERS[m["key"]]
                stats_json_path = os.path.join(base, folder,
                                               f"processed_{sp_name}.json")
                counts_json_path = os.path.join(base, folder,
                                                "processed.json")

            # ----- confidence statistics -----
            mono_stats = extract_stats_data(stats_json_path,
                                            "mono_channel",
                                            species) if m["key"] == "mono" else {}
            beam_stats = extract_stats_data(stats_json_path,
                                            "beamformed",
                                            species)

            # For the baseline we keep the *mono* numbers.
            # For the other methods we keep the *beamformed* numbers.
            if m["key"] == "mono":
                data[m["key"]]["confidence_scores"].append(mono_stats["avg"])
                data[m["key"]]["stdev"]           .append(mono_stats["stdev"])
                data[m["key"]]["max"]             .append(mono_stats["max"])
                data[m["key"]]["min"]             .append(mono_stats["min"])
                data[m["key"]]["median"]          .append(mono_stats["median"])
            else:
                data[m["key"]]["confidence_scores"].append(beam_stats["avg"])
                data[m["key"]]["stdev"]           .append(beam_stats["stdev"])
                data[m["key"]]["max"]             .append(beam_stats["max"])
                data[m["key"]]["min"]             .append(beam_stats["min"])
                data[m["key"]]["median"]          .append(beam_stats["median"])

            # ----- detection counts -----
            cnt = extract_counts_data(counts_json_path,
                                      "mono_channel" if m["key"] == "mono"
                                      else "beamformed",
                                      species)
            data[m["key"]]["counts"].append(cnt)

    # ------------------------------------------------------------------
    # 2)  Load significance data (unchanged – same JSON for all methods)
    # ------------------------------------------------------------------
    significance_data = load_significance_data(parent_dir, species)

    # ------------------------------------------------------------------
    # 3)  Plot – now we have FOUR groups per distance
    # ------------------------------------------------------------------
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(5.7, 4),
                                   sharex=True,
                                   gridspec_kw={'height_ratios': [1, 1]})

    # title --------------------------------------------------------------
    species_disp = species.replace("_", " ")
    fig.suptitle(
        f'VSE Recordings Analysed with BirdNET – {species_disp}',
        fontsize=11, y=1, x=0.5, ha="center")

    x = np.arange(len(DISPLAY_LABELS))           # positions for the 7 distances
    n_methods = len(METHODS)
    total_width = 0.80                          # total width occupied by all bars
    bar_width = total_width / n_methods         # width of each individual bar
    offsets = np.linspace(-total_width/2 + bar_width/2,
                           total_width/2 - bar_width/2,
                           n_methods)

    # ------------------------------------------------------------------
    # 1) Confidence‑score panel
    # ------------------------------------------------------------------
    for off, m in zip(offsets, METHODS):
        bars = ax1.bar(x + off,
                       data[m["key"]]["confidence_scores"],
                       bar_width,
                       label=m["label"],
                       color=m["color"],
                       alpha=0.8,
                       yerr=data[m["key"]]["stdev"],
                       capsize=2,
                       error_kw=dict(
                       elinewidth=0.9,
                       capthick=0.9))

    # axis formatting
    ax1.set_ylabel('(a) Mean Confidence', fontsize=11, labelpad=2)
    ax1.set_ylim(0, 1.1)
    ax1.grid(axis='y', linestyle='--', alpha=0.6)
    # ax2.legend(fontsize=12, loc='upper right')
    # ax1.text(-0.16, 0.5, '(a)', transform=ax1.transAxes, fontsize=11)

    # ------------------------------------------------------------------
    # 2) Detection‑count panel
    # ------------------------------------------------------------------
    for off, m in zip(offsets, METHODS):
        bars = ax2.bar(x + off,
                       data[m["key"]]["counts"],
                       bar_width,
                       label=m["label"],
                       color=m["color"],
                       alpha=0.8)

    # axis formatting
    ax2.set_xlabel('Signal-to-noise ratio level experiments', fontsize=11)
    ax2.set_ylabel('(b) Detection Count', fontsize=11, labelpad=4)
    ax2.set_xticks(x)
    ax2.set_xticklabels(DISPLAY_LABELS, fontsize=10)
    ax2.grid(axis='y', linestyle='--', alpha=0.6)
    # ax2.text(-0.16, 0.5, '(b)', transform=ax2.transAxes, fontsize=11)
    # ax1.legend(fontsize=11, loc='auto')
    experiment_handles = [
    Rectangle((0,0), 1, 1, facecolor=experiment_colors[exp], label=exp)
    for exp in legend_experiments]
    ax2.legend(
        handles=experiment_handles,
        ncol=len(experiment_handles),   # 1 baris
        mode="expand",                  # menyesuaikan lebar menjadi penuh (atau minimal)
        title_fontsize=9,
        fontsize=9,
        # bbox_to_anchor=(0.5, -0.02),   # <‑‑ sedikit di luar figure
        bbox_transform=fig.transFigure,
        frameon=False,
        borderaxespad=0,                # buang jarak di sekitar legend
        handleheight=0.5,               # menurunkan tinggi handle (marker)
        handletextpad=0.2,              # jarak antara handle dan teks
        columnspacing=1.0,              # jarak antar kolom (kurangi pada multi‑col)
        labelspacing=0.4,               # jarak antar baris (kurangi pada multi‑row)
    )
    plt.subplots_adjust(bottom=0.15)
    # ------------------------------------------------------------------
    # 4) Set y‑limit a little higher than the highest count bar
    # ------------------------------------------------------------------
    # keep only the method entries (they are dicts that have a 'counts' list)
    counts_lists = [
        v["counts"]
        for k, v in data.items()
        if isinstance(v, dict) and "counts" in v
    ]

    # safeguard against an empty list (should not happen, but avoids a crash)
    if counts_lists:
        max_cnt = max(max(lst) for lst in counts_lists)
    else:
        max_cnt = 0

    ax2.set_ylim(0, max_cnt * 1.15)

    # ------------------------------------------------------------------
    # 3) (Optional) add significance stars – unchanged, just call once
    # ------------------------------------------------------------------
    # for i in range(len(EXPERIMENT_FOLDERS)):
    #     add_significance_labels(ax1, x, significance_data, data, i, y_offset=0.1)
    #     add_significance_labels(ax2, x, significance_data, data, i)

    # ------------------------------------------------------------------
    # 4) Save & show
    # ------------------------------------------------------------------
    plt.tight_layout()
    plt.subplots_adjust(top=0.95, hspace=0.03)

    timestamp = datetime.now().strftime("%Y%m%d")
    out_path = os.path.join(parent_dir,
                            f"lab_{species}_combined_analysis_{timestamp}.png")
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    print(f"Figure saved to: {out_path}")
    plt.show()

def determine_target_species(parent_dir, filename="paired_detections.json"):
    """
    Menentukan daftar target_species berdasarkan jumlah folder di parent_dir.
    Jika hanya satu folder, ambil semua spesies dari paired_detections.json.
    Jika banyak folder, gunakan daftar default.
    """
    folders = [f for f in os.listdir(parent_dir) if os.path.isdir(os.path.join(parent_dir, f))]
    if len(folders) <= 2:
        single_folder = folders[0]
        file_path = os.path.join(parent_dir, single_folder, filename)
        try:
            with open(file_path, 'r') as f:
                data = json.load(f)
                species_list = list(data.keys())
                print(f"Detected single folder '{single_folder}'. Target species set to: {species_list}")
                return species_list
        except FileNotFoundError:
            print(f"Error: File {file_path} not found. Using default species.")
            return ["Rose-ringed Parakeet", "Eurasian Blue Tit"]
        except json.JSONDecodeError:
            print(f"Error: File {file_path} is not a valid JSON. Using default species.")
            return ["Rose-ringed Parakeet", "Eurasian Blue Tit"]
        except Exception as e:
            print(f"Error processing {file_path}: {str(e)}. Using default species.")
            return ["Rose-ringed Parakeet", "Eurasian Blue Tit"]
    else:
        print(f"Multiple folders detected. Target species set to default: ['Eurasian Blue Tit']")
        return ["Eurasian Blue Tit"]

def main(parent_dir):
    """Main function to process species."""
    target_species = determine_target_species(parent_dir)
    print(f"Processing species: {target_species}")
    for species in target_species:
        create_plots(parent_dir, species)

if __name__ == "__main__":
    main(parent_dir)

# RECAP COMBINE Analysis ALL SPECIES
25 July 2025

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# Constants
SPECIES = [
    "Eurasian Blue Tit", "Common Buzzard", "Eurasian Green Woodpecker",
    "Eurasian Wren"
]
DISPLAY_LABELS = ["1 m"]
EXPERIMENT_FOLDERS = ["01m"]
beamformed_var = "LabIR"
parent_dir = "/content/drive/MyDrive/2025 Imperial/202507 Silwood Park/05Experiment_LabIR"

def initialize_data_lists(num_species):
    """Initialize lists to store confidence scores and counts for all species."""
    return {
        'mono': {
            'confidence_scores': [0] * num_species,
            'stdev': [0] * num_species,
            'counts': [0] * num_species
        },
        'beamformed': {
            'confidence_scores': [0] * num_species,
            'stdev': [0] * num_species,
            'counts': [0] * num_species
        }
    }

def extract_stats_data(json_path, data_type, target_species):
    """Extract statistical data from JSON file with detailed error logging."""
    if not os.path.exists(json_path):
        print(f"File not found: {json_path}")
        return {'avg': 0, 'stdev': 0}

    try:
        with open(json_path, 'r') as file:
            data = json.load(file)
            channel_data = data.get(data_type, {})
            if not channel_data:
                print(f"No '{data_type}' data found in {json_path}")
                return {'avg': 0, 'stdev': 0}
            return {
                'avg': channel_data.get('avg', 0) or 0,
                'stdev': channel_data.get('stdev', 0) or 0
            }
    except json.JSONDecodeError as e:
        print(f"JSON decode error in {json_path}: {e}")
        return {'avg': 0, 'stdev': 0}
    except Exception as e:
        print(f"Unexpected error in {json_path}: {e}")
        return {'avg': 0, 'stdev': 0}

def extract_counts_data(json_path, data_type, target_species):
    """Extract count data from JSON file."""
    target_species = target_species.replace("_", " ")
    if not os.path.exists(json_path):
        print(f"File not found: {json_path}")
        return 0

    try:
        with open(json_path, 'r') as file:
            data = json.load(file)
            channel_data = data.get(data_type, {})
            return channel_data.get(target_species, {}).get('count', 0) or 0
    except (json.JSONDecodeError, KeyError) as e:
        print(f"Error processing {json_path}: {e}")
        return 0

def add_value_labels(bars, values, ax, offset, color, ha):
    """Add value labels above bars if non-zero."""
    for i, bar in enumerate(bars):
        height = bar.get_height()
        if height > 0:
            ax.text(
                bar.get_x() + bar.get_width() / 2 + offset,
                height + 0.02,
                f'{height:.2f}',
                ha=ha,
                va='bottom',
                fontsize=10,
                color=color,
                alpha=0.9
            )

def add_count_labels(bars, values, ax):
    """Add count labels above bars if non-zero."""
    for i, bar in enumerate(bars):
        height = bar.get_height()
        if height >= 0:
            ax.annotate(
                f'{int(height)}',
                xy=(bar.get_x() + bar.get_width() / 2, height + 5),
                ha='center',
                va='bottom',
                fontsize=10
            )

def add_annotations(ax, x, mono_scores, beamformed_scores):
    """Add detection and improvement annotations."""
    for i in range(len(mono_scores)):
        y_pos = 0.3
        if mono_scores[i] == 0 and beamformed_scores[i] > 0:
            ax.annotate(
                "Detected by\nbeamforming",
                xy=(x[i] - 0.1, y_pos),
                ha='center',
                va='bottom',
                fontsize=9,
                color='darkgreen',
                bbox=dict(boxstyle='round,pad=0.2', fc='#ffffe8', ec='gray', lw=0.5, alpha=0.9),
                rotation=90,
                rotation_mode='anchor'
            )
        elif mono_scores[i] == 0 and beamformed_scores[i] == 0:
            ax.annotate(
                "No detection",
                xy=(x[i], y_pos),
                ha='center',
                va='bottom',
                fontsize=9,
                color='darkred',
                bbox=dict(boxstyle='round,pad=0.2', fc='#ffffe8', ec='gray', lw=0.5, alpha=0.9),
                rotation=90,
                rotation_mode='anchor'
            )
        elif mono_scores[i] > 0 and beamformed_scores[i] > 0:
            improvement = ((beamformed_scores[i] - mono_scores[i]) / mono_scores[i]) * 100
            if abs(improvement) > 1:
                y_pos = 0.2
                label = f"+{improvement:.0f}%↑" if improvement > 0 else f"{improvement:.0f}%↓"
                color = 'darkgreen' if improvement > 0 else 'darkred'
                ax.annotate(
                    label,
                    xy=(x[i], y_pos),
                    ha='center',
                    va='bottom',
                    fontsize=9,
                    color=color,
                    bbox=dict(boxstyle='round,pad=0.2', fc='#ffffe8', ec='gray', lw=0.5, alpha=0.9)
                )

def add_count_improvement(ax, x, mono_counts, beamformed_counts):
    """Add count improvement percentages."""
    for i in range(len(mono_counts)):
        if mono_counts[i] > 0 and beamformed_counts[i] > 0:
            improvement = ((beamformed_counts[i] - mono_counts[i]) / mono_counts[i]) * 100
            if improvement != 0:
                color = 'darkgreen' if improvement > 0 else 'darkred'
                ax.annotate(
                    f"+{improvement:.0f}%↑" if improvement > 0 else f"{improvement:.0f}%↓",
                    xy=(x[i], 5),
                    ha='center',
                    va='bottom',
                    fontsize=9,
                    color=color,
                    bbox=dict(boxstyle='round,pad=0.2', fc='#ffeee8', ec='gray', lw=0.5, alpha=0.9)
                )

def create_plots(parent_dir):
    """Create and save comparison plots for all species at a given angle."""
    data = initialize_data_lists(len(SPECIES))

    # Extract data for each species _{degree:03d}
    folder = EXPERIMENT_FOLDERS[0]  # Only one folder: "01m"
    for i, target_species in enumerate(SPECIES):
        target_species_json = target_species.replace(" ", "_")
        stats_json_path = os.path.join(parent_dir, folder, f"processed_{target_species_json}.json")
        counts_json_path = os.path.join(parent_dir, folder, "processed.json")

        # Extract confidence scores
        mono_stats = extract_stats_data(stats_json_path, "mono_channel", target_species)
        beamformed_stats = extract_stats_data(stats_json_path, "beamformed", target_species)

        # Store stats
        data['mono']['confidence_scores'][i] = mono_stats['avg'] or 0
        data['beamformed']['confidence_scores'][i] = beamformed_stats['avg'] or 0
        data['mono']['stdev'][i] = mono_stats['stdev'] or 0
        data['beamformed']['stdev'][i] = beamformed_stats['stdev'] or 0

        # Extract counts
        data['mono']['counts'][i] = extract_counts_data(counts_json_path, "mono_channel", target_species)
        data['beamformed']['counts'][i] = extract_counts_data(counts_json_path, "beamformed", target_species)

    # Create figure with two subplots Angle = {degree}°
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True, gridspec_kw={'height_ratios': [1, 1]})
    if beamformed_var=="LabIR":
        fig.suptitle(
            f'BirdNET Analysis: Mono Channel vs. Beamformed Recordings (LabIR)',
            fontsize=16,
            y=1.02
        )
    elif beamformed_var=="SPIR":
        fig.suptitle(
            f'BirdNET Analysis: Mono Channel vs. Beamformed Recordings (SPIR)',
            fontsize=16,
            y=1.02
        )
    elif beamformed_var=="SA":
        fig.suptitle(
            f'BirdNET Analysis: Mono Channel vs. Sum Averaging',
            fontsize=16,
            y=1.02
        )
    else:
        raise ValueError(f"Invalid beamformed_var: {beamformed_var}")

    # Plot setup
    x = np.arange(len(SPECIES))
    width = 0.35

    # Subplot 1: Confidence scores
    mono_bars = ax1.bar(
        x - width/2,
        data['mono']['confidence_scores'],
        width,
        label='Mono Channel',
        color='skyblue',
        alpha=0.8,
        yerr=data['mono']['stdev'],
        capsize=5
    )
    beam_bars = ax1.bar(
        x + width/2,
        data['beamformed']['confidence_scores'],
        width,
        label='LabIR',
        color='#fcc006',
        alpha=0.8,
        yerr=data['beamformed']['stdev'],
        capsize=5
    )

    # Add overlay line charts confidence_scores
    ax1.plot(x - width/2, data['mono']['confidence_scores'], 'o--', color='skyblue', alpha=0.5, linewidth=1.5, markersize=8)
    ax1.plot(x + width/2, data['beamformed']['confidence_scores'], 's--', color='orange', alpha=0.5, linewidth=1.5, markersize=8)

    # Add labels and annotations
    add_value_labels(mono_bars, data['mono']['confidence_scores'], ax1, -width/2, 'black', 'right')
    add_value_labels(beam_bars, data['beamformed']['confidence_scores'], ax1, width/2, 'black', 'left')
    add_annotations(ax1, x, data['mono']['confidence_scores'], data['beamformed']['confidence_scores'])

    # Subplot 1 settings
    ax1.text(-0.08, 0.5, '(a)', transform=ax1.transAxes, fontsize=16)
    ax1.set_ylabel('Mean Confidence Score', fontsize=14)
    ax1.grid(axis='y', linestyle='--', alpha=0.7)

    ax1.set_ylim(0, 1.1)

    # Subplot 2: Counts
    mono_count_bars = ax2.bar(
        x - width/2,
        data['mono']['counts'],
        width,
        label='Mono Channel',
        color='skyblue',
        alpha=0.8
    )
    beam_count_bars = ax2.bar(
        x + width/2,
        data['beamformed']['counts'],
        width,
        label='LabIR',
        color='#fcc006', #'#fcc006', # lightcoral lightgreen
        alpha=0.8
    )

    # Add overlay line charts for counts
    ax2.plot(x - width/2, data['mono']['counts'], 'o--', color='skyblue', alpha=0.5, linewidth=1.5, markersize=8)
    ax2.plot(x + width/2, data['beamformed']['counts'], 's--', color='orange', alpha=0.5, linewidth=1.5, markersize=8)

    # Add count labels and improvement annotations
    add_count_labels(mono_count_bars, data['mono']['counts'], ax2)
    add_count_labels(beam_count_bars, data['beamformed']['counts'], ax2)
    add_count_improvement(ax2, x, data['mono']['counts'], data['beamformed']['counts'])

    # Subplot 2 settings
    ax2.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax2.text(-0.08, 0.5, '(b)', transform=ax2.transAxes, fontsize=16)
    ax2.set_xlabel('Species', fontsize=14, ha="center")
    ax2.set_ylabel('Detection Counts', fontsize=14)
    ax2.set_xticks(x)
    ax2.set_xticklabels(SPECIES, fontsize=10, rotation=45, ha='right')
    ax2.grid(axis='y', linestyle='--', alpha=0.7)
    ax2.set_ylim(0, max(max(data['mono']['counts'], default=0), max(data['beamformed']['counts'], default=0)) * 1.1)
    ax2.legend(fontsize=12, loc='upper left')
    # Save and show plot {degree:03d}
    current_datetime = datetime.now()
    timestamp = current_datetime.strftime("%Y%m%d_%H%M%S")
    fig_path = os.path.join(parent_dir, f"all_species_combined_analysis_{timestamp}.png")
    print(f"Saving plot to: {fig_path}")
    plt.tight_layout()
    plt.subplots_adjust(top=0.95, hspace=0.2)
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    plt.show()

def main(parent_dir):
    """Main function to process angles."""
    # for degree in range(180, 240, 60): degree
    create_plots(parent_dir)

if __name__ == "__main__":
    main(parent_dir)

# Data Cleaning: "processed.json" -> "paired_detections.json"
7 Agustus

In [ ]:
import json
import os
from typing import Dict, List, Any, Optional

# ============================
# 1. KONFIGURASI & PARAMETER
# ============================

# FOLDERS = ["01m", "02m", "04m", "08m", "16m", "32m", "64m"]
FOLDERS = ["0", "1", "2", "3", "4", "5", "6"]
BASE_DIR = "/content/drive/MyDrive/2025 Imperial/202508 Lab/Experiment_LabIR"

INPUT_FILE = "processed.json"
OUTPUT_FILE = "paired_detections.json"

# ============================
# 2. FUNGSI UTAMA PEMROSESAN
# ============================

def pair_detections(mono_data: Dict[str, Any], beam_data: Dict[str, Any]) -> Dict[str, List[Dict[str, Any]]]:
    """
    Memasangkan deteksi mono dan beamformed berdasarkan start_time_list untuk setiap spesies.
    Menangani spesies yang hanya ada di beam_data.

    Args:
        mono_data: Data deteksi mono channel
        beam_data: Data deteksi beamformed

    Returns:
        Dict yang berisi data deteksi berpasangan per spesies
    """
    paired_data = {}

    # Iterasi melalui setiap spesies di beam_data (prioritas utama)
    for species in beam_data:
        # Inisialisasi list untuk spesies ini
        paired_data[species] = []

        # Ambil data mono untuk spesies ini (jika tidak ada, gunakan default kosong)
        mono_species_data = mono_data.get(species, {})
        mono_times = mono_species_data.get("start_time_list", [])
        mono_confs = mono_species_data.get("conf_list", [])

        # Ambil data beamformed untuk spesies ini
        beam_times = beam_data[species].get("start_time_list", [])
        beam_confs = beam_data[species].get("conf_list", [])

        # Validasi panjang data
        if len(mono_times) != len(mono_confs):
            print(f"⚠️  Warning: Mismatch in mono data length for {species}")
        if len(beam_times) != len(beam_confs):
            print(f"⚠️  Warning: Mismatch in beam data length for {species}")

        # Buat dictionary untuk lookup cepat O(1)
        mono_dict = dict(zip(mono_times, mono_confs))
        beam_dict = dict(zip(beam_times, beam_confs))

        # Gabungkan semua timestamp unik dan urutkan
        all_times = sorted(set(mono_times + beam_times))

        # Iterasi melalui semua timestamp
        for timestamp in all_times:
            event = {"timestamp": timestamp}

            # Cek deteksi mono
            if timestamp in mono_dict:
                event.update({
                    "mono_detected": True,
                    "mono_confidence": mono_dict[timestamp]
                })
            else:
                event.update({
                    "mono_detected": False,
                    "mono_confidence": None
                })

            # Cek deteksi beamformed
            if timestamp in beam_dict:
                event.update({
                    "beam_detected": True,
                    "beam_confidence": beam_dict[timestamp]
                })
            else:
                event.update({
                    "beam_detected": False,
                    "beam_confidence": None
                })

            paired_data[species].append(event)

    return paired_data

def process_single_file(input_path: str, output_path: str) -> bool:
    """
    Membaca processed.json dan menghasilkan JSON baru dengan deteksi berpasangan.

    Args:
        input_path: Path ke file input
        output_path: Path ke file output

    Returns:
        bool: True jika berhasil, False jika gagal
    """
    try:
        # Validasi file input
        if not os.path.exists(input_path):
            print(f"❌ File not found: {input_path}")
            return False

        # Baca file JSON
        with open(input_path, 'r') as f:
            data = json.load(f)

        # Validasi struktur data
        if not isinstance(data, dict):
            raise ValueError("Invalid JSON structure: expected dictionary")

        mono_data = data.get("mono_channel", {})
        beam_data = data.get("beamformed", {})

        # Validasi data
        if not beam_data:
            print(f"⚠️  Warning: No beamformed data found in {input_path}")
            return False

        # Proses data untuk memasangkan deteksi
        paired_data = pair_detections(mono_data, beam_data)

        # Simpan ke file JSON baru
        with open(output_path, 'w') as f:
            json.dump(paired_data, f, indent=2, ensure_ascii=False)

        species_count = len(paired_data)
        event_count = sum(len(events) for events in paired_data.values())
        print(f"✅ Processed: {input_path}")
        print(f"   📊 Species: {species_count}, Events: {event_count}")
        print(f"   💾 Saved to: {output_path}")

        return True

    except FileNotFoundError:
        print(f"❌ File not found: {input_path}")
    except json.JSONDecodeError as e:
        print(f"❌ Invalid JSON in {input_path}: {e}")
    except ValueError as e:
        print(f"❌ Data validation error in {input_path}: {e}")
    except PermissionError:
        print(f"❌ Permission denied: {input_path} or {output_path}")
    except Exception as e:
        print(f"❌ Unexpected error processing {input_path}: {str(e)}")

    return False

def batch_process_folders(base_dir: str, folders: List[str],
                         input_file: str, output_file: str) -> Dict[str, bool]:
    """
    Memproses semua folder secara batch.

    Args:
        base_dir: Direktori base
        folders: List folder untuk diproses
        input_file: Nama file input
        output_file: Nama file output

    Returns:
        Dict yang menunjukkan status proses untuk setiap folder
    """
    print(f"🚀 Starting batch processing...")
    print(f"📂 Base directory: {base_dir}")
    print(f"📁 Folders to process: {folders}")

    results = {}
    successful = 0

    for folder in folders:
        print(f"\n {'─' * 50}")
        print(f"Processing folder: {folder}")
        print(f" {'─' * 50}")

        folder_path = os.path.join(base_dir, folder)
        input_path = os.path.join(folder_path, input_file)
        output_path = os.path.join(folder_path, output_file)

        # Buat direktori output jika belum ada
        os.makedirs(folder_path, exist_ok=True)

        # Proses file
        success = process_single_file(input_path, output_path)
        results[folder] = success

        if success:
            successful += 1
        else:
            print(f"❌ Failed to process {folder}")

    # Ringkasan
    print(f"\n {'═' * 60}")
    print(f"📊 BATCH PROCESSING SUMMARY")
    print(f" {'═' * 60}")
    print(f"Total folders: {len(folders)}")
    print(f"Successful: {successful}")
    print(f"Failed: {len(folders) - successful}")
    print(f"Success rate: {successful/len(folders)*100:.1f}%")

    return results

# ============================
# 3. EKSEKUSI UTAMA
# ============================

def main():
    """Main execution function."""
    try:
        results = batch_process_folders(BASE_DIR, FOLDERS, INPUT_FILE, OUTPUT_FILE)

        # Detail hasil
        print(f"\n {'─' * 40}")
        print("Detailed Results:")
        print(f" {'─' * 40}")
        for folder, success in results.items():
            status = "✅" if success else "❌"
            print(f"{status} {folder}")

    except KeyboardInterrupt:
        print("\n\n⚠️  Process interrupted by user")
    except Exception as e:
        print(f"\n❌ Fatal error: {e}")

if __name__ == "__main__":
    main()

# Paired Detections -> stats_significance
7 Agustus

In [ ]:
!pip install numpy scipy pandas statsmodels

In [ ]:
import os
import json
import numpy as np
from scipy import stats
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests
from datetime import datetime
from typing import Dict, List, Tuple, Any, Optional

# ============================
# 1. SETUP & CONFIGURATION
# ============================

# Configuration
BASE_DIR = "/content/drive/MyDrive/2025 Imperial/202508 Lab/01Experiment_SA"
# FOLDERS = ["01m", "02m", "04m", "08m", "16m", "32m", "64m"]
DISTANCE_MAPPING = {
    folder: folder for folder in FOLDERS
}
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

# Constants
TOTAL_POSSIBLE_EVENTS = 271
ALPHA = 0.05

# ============================
# 2. UTILITY FUNCTIONS
# ============================

def check_normality(data: List[float], alpha: float = ALPHA) -> bool:
    """
    Check if data follows normal distribution using Shapiro-Wilk test.

    Args:
        data: List of numerical values
        alpha: Significance level

    Returns:
        bool: True if data is normally distributed
    """
    if len(data) < 3:
        return False
    try:
        stat, p = stats.shapiro(data)
        return p > alpha
    except Exception as e:
        print(f"Warning: Normality test failed - {e}")
        return False

def load_paired_data(base_dir: str, folders: List[str], species: str) -> Dict[str, List[Dict]]:
    """
    Load paired detection data from JSON files.

    Args:
        base_dir: Base directory path
        folders: List of folder names
        species: Target species name

    Returns:
        Dict mapping folder names to detection data
    """
    all_data = {}

    for folder in folders:
        json_path = os.path.join(base_dir, folder, "paired_detections.json")

        # Check file existence
        if not os.path.exists(json_path):
            print(f"⚠️  File not found: {json_path}")
            all_data[folder] = []
            continue

        # Load and parse JSON
        try:
            with open(json_path, 'r') as file:
                data = json.load(file)

            if species in data:
                all_data[folder] = data[species]
                print(f"✅ Loaded {len(data[species])} events for {species} from {folder}")
            else:
                print(f"⚠️  No data for {species} in {folder}")
                all_data[folder] = []

        except json.JSONDecodeError as e:
            print(f"❌ JSON decode error in {folder}: {e}")
            all_data[folder] = []
        except Exception as e:
            print(f"❌ Error loading {folder}: {e}")
            all_data[folder] = []

    return all_data

# ============================
# 3. STATISTICAL TEST FUNCTIONS
# ============================

def perform_paired_comparison_test(mono_scores: List[float], beam_scores: List[float],
                                 distance: str) -> Optional[Dict[str, Any]]:
    """
    Perform paired t-test or Wilcoxon signed-rank test based on normality.

    Args:
        mono_scores: List of mono confidence scores
        beam_scores: List of beam confidence scores
        distance: Distance label

    Returns:
        Dict with test results or None if insufficient data
    """
    if len(mono_scores) < 2 or len(beam_scores) < 2:
        return None

    try:
        # Check normality
        is_normal = check_normality(mono_scores) and check_normality(beam_scores)
        print(f"📊 Normality test for {distance}: {'Normal' if is_normal else 'Not Normal'}")

        if is_normal:
            # Paired t-test
            t_stat, p_value = stats.ttest_rel(beam_scores, mono_scores)
            diff = np.array(beam_scores) - np.array(mono_scores)
            cohen_d = np.mean(diff) / np.std(diff, ddof=1) if np.std(diff, ddof=1) != 0 else 0

            return {
                'test_type': 't_test',
                't_stat': float(t_stat),
                'p_value': float(p_value),
                'cohen_d': float(cohen_d)
            }
        else:
            # Wilcoxon signed-rank test
            stat, p_value = stats.wilcoxon(beam_scores, mono_scores)
            return {
                'test_type': 'wilcoxon',
                'stat': float(stat),
                'p_value': float(p_value)
            }
    except Exception as e:
        print(f"❌ Error in paired comparison test for {distance}: {e}")
        return None

def perform_mcnemar_test(events: List[Dict], distance: str) -> Optional[Dict[str, Any]]:
    """
    Perform McNemar's test for paired nominal data.

    Args:
        events: List of detection events
        distance: Distance label

    Returns:
        Dict with McNemar test results or None
    """
    if not events:
        return None

    try:
        # Build contingency table
        contingency = [[0, 0], [0, 0]]  # [[both, mono_only], [beam_only, neither]]

        for event in events:
            mono_detected = event.get("mono_detected", False)
            beam_detected = event.get("beam_detected", False)

            if mono_detected and beam_detected:
                contingency[0][0] += 1
            elif mono_detected and not beam_detected:
                contingency[0][1] += 1
            elif not mono_detected and beam_detected:
                contingency[1][0] += 1
            else:
                contingency[1][1] += 1

        # Perform test only if there are discordant pairs
        if contingency[0][1] + contingency[1][0] > 0:
            result = mcnemar(contingency, exact=True)
            return {
                'stat': float(result.statistic),
                'p_value': float(result.pvalue)
            }
    except Exception as e:
        print(f"❌ Error in McNemar test for {distance}: {e}")

    return None

def perform_association_test(events: List[Dict], total_possible: int, distance: str) -> Optional[Dict[str, Any]]:
    """
    Perform Chi-square or Fisher's exact test for association.

    Args:
        events: List of detection events
        total_possible: Total possible events
        distance: Distance label

    Returns:
        Dict with association test results or None
    """
    if not events:
        return None

    try:
        # Calculate detection counts
        mono_count = sum(1 for e in events if e.get("mono_detected", False))
        beam_count = sum(1 for e in events if e.get("beam_detected", False))

        # Validate counts
        if mono_count > total_possible or beam_count > total_possible:
            print(f"⚠️  Invalid counts for {distance}: mono={mono_count}, beam={beam_count}")
            return None

        # Create contingency table
        contingency = np.array([
            [beam_count, total_possible - beam_count],
            [mono_count, total_possible - mono_count]
        ])

        # Perform appropriate test
        if np.sum(contingency) > 0:
            # Check expected frequencies for Chi-square suitability
            chi2, p_value, dof, expected = stats.chi2_contingency(contingency)

            if np.any(expected < 5):
                # Use Fisher's exact test
                odds_ratio, p_value = stats.fisher_exact(contingency)
                return {
                    'test': 'Fisher',
                    'stat': float(odds_ratio),
                    'p_value': float(p_value),
                    'beam_count': int(beam_count),
                    'mono_count': int(mono_count)
                }
            else:
                # Use Chi-square test
                n = np.sum(contingency)
                min_dim = min(contingency.shape) - 1
                cramers_v = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 and n > 0 else 0

                return {
                    'test': 'Chi-Square',
                    'stat': float(chi2),
                    'p_value': float(p_value),
                    'effect_size': float(cramers_v),
                    'beam_count': int(beam_count),
                    'mono_count': int(mono_count)
                }
    except Exception as e:
        print(f"❌ Error in association test for {distance}: {e}")

    return None

# ============================
# 4. MAIN STATISTICAL ANALYSIS
# ============================

def perform_statistical_tests(data: Dict[str, List[Dict]],
                            total_possible: int = TOTAL_POSSIBLE_EVENTS) -> Tuple[List[Dict], List[Dict], List[Dict], List[Dict]]:
    """
    Perform all statistical tests on the data.

    Args:
        data: Dictionary mapping folders to event data
        total_possible: Total possible events

    Returns:
        Tuple of test results: (t_test, wilcoxon, mcnemar, chi_square)
    """
    t_test_results = []
    wilcoxon_results = []
    mcnemar_results = []
    chi_square_results = []

    print("🔬 Performing statistical tests...")

    for folder, events in data.items():
        distance = DISTANCE_MAPPING.get(folder, f"Unknown-{folder}")
        print(f"\n  Processing {distance} ({len(events)} events)")

        if not events:
            print(f"    ⚠️  No events for {distance}, skipping tests")
            continue

        # Extract scores
        mono_scores = [e["mono_confidence"] for e in events
                      if e.get("mono_confidence") is not None]
        beam_scores = [e["beam_confidence"] for e in events
                      if e.get("beam_confidence") is not None]
        paired_scores = [(e["mono_confidence"], e["beam_confidence"])
                        for e in events
                        if e.get("mono_confidence") is not None and e.get("beam_confidence") is not None]

        print(f"    Scores: mono={len(mono_scores)}, beam={len(beam_scores)}, paired={len(paired_scores)}")

        # Paired comparison test (t-test or Wilcoxon)
        if len(paired_scores) >= 2:
            mono_paired, beam_paired = zip(*paired_scores)
            test_result = perform_paired_comparison_test(list(mono_paired), list(beam_paired), distance)

            if test_result:
                result_entry = {
                    'folder': folder,
                    'distance': distance,
                    **test_result
                }

                if test_result['test_type'] == 't_test':
                    t_test_results.append(result_entry)
                else:
                    wilcoxon_results.append(result_entry)

        # McNemar's test
        mcnemar_result = perform_mcnemar_test(events, distance)
        if mcnemar_result:
            mcnemar_results.append({
                'folder': folder,
                'distance': distance,
                **mcnemar_result
            })

        # Association test (Chi-square or Fisher's exact)
        assoc_result = perform_association_test(events, total_possible, distance)
        if assoc_result:
            chi_square_results.append({
                'folder': folder,
                'distance': distance,
                **assoc_result
            })

    # Apply multiple testing correction
    print("\n🔤 Applying Bonferroni correction...")
    corrected_results = {}

    for name, results in [('t_test', t_test_results), ('wilcoxon', wilcoxon_results),
                         ('mcnemar', mcnemar_results), ('chi_square', chi_square_results)]:
        if results:
            p_values = [r['p_value'] for r in results]
            try:
                _, corrected_p, _, _ = multipletests(p_values, method='bonferroni')

                for i, result in enumerate(results):
                    result['corrected_p_value'] = float(corrected_p[i])
                    result['sig_stars'] = (
                        '***' if corrected_p[i] < 0.001 else
                        '**' if corrected_p[i] < 0.01 else
                        '*' if corrected_p[i] < 0.05 else ''
                    )
            except Exception as e:
                print(f"⚠️  Error in multiple testing correction for {name}: {e}")
                # Keep original p-values if correction fails
                for result in results:
                    result['corrected_p_value'] = result['p_value']
                    result['sig_stars'] = (
                        '***' if result['p_value'] < 0.001 else
                        '**' if result['p_value'] < 0.01 else
                        '*' if result['p_value'] < 0.05 else ''
                    )

    print(f"✅ Tests completed: t-test={len(t_test_results)}, wilcoxon={len(wilcoxon_results)}, "
          f"mcnemar={len(mcnemar_results)}, chi-square={len(chi_square_results)}")

    return t_test_results, wilcoxon_results, mcnemar_results, chi_square_results

# ============================
# 5. DATA OUTPUT & SERIALIZATION
# ============================

def create_summary_json(t_test_results: List[Dict], wilcoxon_results: List[Dict],
                       mcnemar_results: List[Dict], chi_square_results: List[Dict]) -> Dict[str, Any]:
    """
    Create structured JSON summary of statistical results.

    Args:
        t_test_results: Results from t-tests
        wilcoxon_results: Results from Wilcoxon tests
        mcnemar_results: Results from McNemar tests
        chi_square_results: Results from Chi-square/Fisher tests

    Returns:
        Dict with structured results
    """
    summary = {}
    distance_order = {v: i for i, v in enumerate(FOLDERS)}

    def sort_results(results: List[Dict]) -> List[Dict]:
        return sorted(results, key=lambda x: distance_order.get(x['distance'], float('inf')))

    # Process t-test results
    if t_test_results:
        summary['t_test'] = {
            r['distance']: {
                't_stat': r['t_stat'],
                'p_value': r['p_value'],
                'corrected_p_value': r['corrected_p_value'],
                'sig_stars': r['sig_stars'],
                'cohen_d': r['cohen_d']
            } for r in sort_results(t_test_results)
        }

    # Process Wilcoxon results
    if wilcoxon_results:
        summary['wilcoxon'] = {
            r['distance']: {
                'stat': r['stat'],
                'p_value': r['p_value'],
                'corrected_p_value': r['corrected_p_value'],
                'sig_stars': r['sig_stars']
            } for r in sort_results(wilcoxon_results)
        }

    # Process McNemar results
    if mcnemar_results:
        summary['mcnemar'] = {
            r['distance']: {
                'stat': r['stat'],
                'p_value': r['p_value'],
                'corrected_p_value': r['corrected_p_value'],
                'sig_stars': r['sig_stars']
            } for r in sort_results(mcnemar_results)
        }

    # Process Chi-square/Fisher results
    if chi_square_results:
        summary['chi_square'] = {
            r['distance']: {
                'test': r['test'],
                'stat': r['stat'],
                'p_value': r['p_value'],
                'corrected_p_value': r['corrected_p_value'],
                'sig_stars': r['sig_stars'],
                'effect_size': r.get('effect_size', r.get('stat', 0)),
                'beam_count': r['beam_count'],
                'mono_count': r['mono_count']
            } for r in sort_results(chi_square_results)
        }

    return summary

def determine_target_species(parent_dir: str, filename: str = "paired_detections.json") -> List[str]:
    """
    Determine target species based on directory structure.

    Args:
        parent_dir: Parent directory path
        filename: JSON filename to check

    Returns:
        List of species names
    """
    try:
        folders = [f for f in os.listdir(parent_dir)
                  if os.path.isdir(os.path.join(parent_dir, f))]

        if len(folders) <= 2 and folders:
            single_folder = folders[0]
            file_path = os.path.join(parent_dir, single_folder, filename)

            try:
                with open(file_path, 'r') as f:
                    data = json.load(f)
                    species_list = list(data.keys())
                    print(f"🎯 Detected single folder '{single_folder}'. Target species: {species_list}")
                    return species_list
            except FileNotFoundError:
                print(f"⚠️  File {file_path} not found. Using default species.")
            except json.JSONDecodeError:
                print(f"⚠️  Invalid JSON in {file_path}. Using default species.")
            except Exception as e:
                print(f"⚠️  Error processing {file_path}: {e}. Using default species.")
        else:
            print(f"📂 Multiple folders detected. Using default species.")

    except Exception as e:
        print(f"⚠️  Error accessing directory {parent_dir}: {e}")

    return ["Rose-ringed Parakeet", "Eurasian Blue Tit"]

# ============================
# 6. MAIN EXECUTION
# ============================

def main():
    """Main execution function."""
    print("🚀 Starting statistical analysis...")
    print(f"📂 Base directory: {BASE_DIR}")

    target_species = determine_target_species(BASE_DIR)
    print(f"🐦 Target species: {target_species}")

    for species in target_species:
        print(f"\n🔬 Processing {species}...")

        # Load data
        data = load_paired_data(BASE_DIR, FOLDERS, species)

        # Perform statistical tests
        t_test_results, wilcoxon_results, mcnemar_results, chi_square_results = perform_statistical_tests(data)

        # Create and save summary
        summary = create_summary_json(t_test_results, wilcoxon_results, mcnemar_results, chi_square_results)

        # Save to JSON file
        json_filename = os.path.join(BASE_DIR, f"stats_significance_{species.replace(' ', '_')}.json")
        try:
            with open(json_filename, 'w') as f:
                json.dump(summary, f, indent=2, ensure_ascii=False)
            print(f"✅ Saved statistical summary to {json_filename}")
        except Exception as e:
            print(f"❌ Error saving {json_filename}: {e}")

    print(f"\n🎉 Analysis complete! Results saved to {BASE_DIR}")

if __name__ == "__main__":
    main()

# Plot Normality
8 Agustus




In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

def determine_target_species(parent_dir, filename="paired_detections.json"):
    """
    Menentukan daftar target_species berdasarkan jumlah folder di parent_dir.
    Jika hanya satu folder, ambil semua spesies dari paired_detections.json.
    Jika banyak folder, gunakan daftar default.
    """
    folders = [f for f in os.listdir(parent_dir) if os.path.isdir(os.path.join(parent_dir, f))]
    if len(folders) <= 2:
        single_folder = folders[0]
        file_path = os.path.join(parent_dir, single_folder, filename)
        try:
            with open(file_path, 'r') as f:
                data = json.load(f)
                # Ambil semua kunci (spesies) dari data
                species_list = list(data.keys())
                print(f"Detected single folder '{single_folder}'. Target species set to: {species_list}")
                return species_list
        except FileNotFoundError:
            print(f"Error: File {file_path} not found. Using default species.")
            return ["Rose-ringed Parakeet", "Eurasian Blue Tit"]
        except json.JSONDecodeError:
            print(f"Error: File {file_path} is not a valid JSON. Using default species.")
            return ["Rose-ringed Parakeet", "Eurasian Blue Tit"]
        except Exception as e:
            print(f"Error processing {file_path}: {str(e)}. Using default species.")
            return ["Rose-ringed Parakeet", "Eurasian Blue Tit"]
    else:
        print(f"Multiple folders detected. Target species set to default: ['Rose-ringed Parakeet', 'Eurasian Blue Tit']")
        return ["Rose-ringed Parakeet", "Eurasian Blue Tit"]

def load_paired_data(parent_dir, filename="paired_detections.json"):
    """
    Membaca paired_detections.json untuk setiap jarak dan mengembalikan data confidence untuk daftar spesies target.
    """
    global target_species
    distances = [f for f in os.listdir(parent_dir) if os.path.isdir(os.path.join(parent_dir, f))]
    data_by_distance = {species: {} for species in target_species}

    for dist in distances:
        file_path = os.path.join(parent_dir, dist, filename)
        try:
            with open(file_path, 'r') as f:
                data = json.load(f)
            for species in target_species:
                species_data = data.get(species, [])
                mono_conf = [event["mono_confidence"] for event in species_data if event["mono_confidence"] is not None]
                beam_conf = [event["beam_confidence"] for event in species_data if event["beam_confidence"] is not None]
                data_by_distance[species][dist] = {"mono_conf": mono_conf, "beam_conf": beam_conf}
                print(f"Loaded data for {species} from {dist}")
        except FileNotFoundError:
            print(f"Error: File {file_path} not found")
        except json.JSONDecodeError:
            print(f"Error: File {file_path} is not a valid JSON")
        except Exception as e:
            print(f"Error processing {file_path}: {str(e)}")

    return data_by_distance

def plot_normality_checks(data_by_distance, output_dir):
    """
    Membuat visualisasi untuk memeriksa normalitas: Histogram, Q-Q Plot, Box Plot, dan KDE Plot untuk daftar spesies target.
    """
    os.makedirs(output_dir, exist_ok=True)

    for species in target_species:
        for dist, data in data_by_distance[species].items():
            mono_conf = data["mono_conf"]
            beam_conf = data["beam_conf"]

            # Hanya buat plot jika data tidak kosong
            if not mono_conf and not beam_conf:
                print(f"Skipping visualization for {species} at {dist}: No data available")
                continue

            # Buat figure dengan 2x2 subplot
            fig, axes = plt.subplots(2, 2, figsize=(12, 10))
            fig.suptitle(f"Normality Check for {species} at {dist}", fontsize=16)

            # 1. Histogram
            if mono_conf:
                sns.histplot(mono_conf, kde=False, color="blue", label="Mono", ax=axes[0, 0], stat="density")
            if beam_conf:
                sns.histplot(beam_conf, kde=False, color="orange", label="Beam", ax=axes[0, 0], stat="density")
            if mono_conf and beam_conf:
                skewness_mono = stats.skew(mono_conf)
                skewness_beam = stats.skew(beam_conf)
                axes[0, 0].text(0.5, 0.98, f'Skewness Mono: {skewness_mono:.2f}\nSkewness Beam: {skewness_beam:.2f}',
                               transform=axes[0, 0].transAxes, ha='center', va='top', fontsize=9)
            axes[0, 0].legend()
            axes[0, 0].set_title("Histogram")

            # 2. Q-Q Plot
            if mono_conf:
                stats.probplot(mono_conf, dist="norm", plot=axes[1, 0])
                axes[1, 0].set_title("Q-Q Plot (Mono)")
            if beam_conf:
                stats.probplot(beam_conf, dist="norm", plot=axes[1, 1])
                axes[1, 1].set_title("Q-Q Plot (Beam)")

            # 3. Box Plot
            box_data = []
            box_labels = []
            if mono_conf:
                box_data.append(mono_conf)
                box_labels.append("Mono")
            if beam_conf:
                box_data.append(beam_conf)
                box_labels.append("Beam")
            if box_data:
                axes[0, 1].boxplot(box_data, tick_labels=box_labels)
                axes[0, 1].set_title("Box Plot")

            # 4. KDE Plot (tambahan di histogram)
            if mono_conf:
                sns.kdeplot(mono_conf, color="blue", label="Mono KDE", ax=axes[0, 0])
            if beam_conf:
                sns.kdeplot(beam_conf, color="orange", label="Beam KDE", ax=axes[0, 0])

            plt.tight_layout(rect=[0, 0, 1, 0.95])
            output_path = os.path.join(output_dir, f"normality_check_{species.replace(' ', '_')}_{dist}.png")
            plt.savefig(output_path)
            plt.close()
            print(f"Saved normality check plot for {species} at {dist} to {output_path}")

# Direktori utama
parent_dir = "/content/drive/MyDrive/2025 Imperial/202508 Lab/01Experiment_SPIR"
output_dir = os.path.join(parent_dir, "normality_plots")

# Tentukan target_species berdasarkan isi parent_dir
target_species = determine_target_species(parent_dir)

# Load data untuk daftar spesies target
data_by_distance = load_paired_data(parent_dir)

# Buat visualisasi untuk daftar spesies target
plot_normality_checks(data_by_distance, output_dir)
print(f"Visualizations complete. Plots saved to {output_dir}")

# Median Difference Summary.json
10 Agustus

In [ ]:
import os
import json
import numpy as np
from pathlib import Path

# Daftar folder jarak (sesuaikan dengan struktur direktorimu)
parent_dir = "/content/drive/MyDrive/2025 Imperial/202508 Lab/01Experiment_SA"
# distance_folders = ["01m", "02m", "04m", "08m", "16m", "32m", "64m"]
distance_folders = FOLDERS

# Dictionary untuk menyimpan hasil
median_diff_results = {}

for dist_folder in distance_folders:
    # Path ke paired_detections.json di setiap folder jarak
    json_path = Path(f"{parent_dir}/{dist_folder}/paired_detections.json")

    if not json_path.exists():
        print(f"⚠️ File tidak ditemukan: {json_path}")
        continue

    with open(json_path, "r") as f:
        data = json.load(f)
        print(f"✅ File ditemukan: {json_path}")

    # Simpan hasil per jarak
    dist_results = {}

    for species, detections in data.items():
        mono_vals = []
        beam_vals = []

        # Filter hanya pasangan valid (mono_detected=true && beam_detected=true)
        for entry in detections:
            if entry["mono_detected"] and entry["beam_detected"]:
                # Pastikan confidence bukan null
                if entry["mono_confidence"] is not None and entry["beam_confidence"] is not None:
                    mono_vals.append(entry["mono_confidence"])
                    beam_vals.append(entry["beam_confidence"])

        # Hitung median difference hanya jika ada pasangan valid
        if mono_vals and beam_vals:
            diffs = np.array(beam_vals) - np.array(mono_vals)
            median_diff = np.median(diffs)
            dist_results[species] = {
                "median_diff": float(median_diff),
                "n_pairs": len(mono_vals),
                "sample_diffs": diffs.tolist()  # Simpan untuk bootstrap CI nanti
            }

    median_diff_results[dist_folder] = dist_results

# Simpan hasil ke file (opsional)
with open(f"{parent_dir}/median_diff_summary.json", "w") as f:
    json.dump(median_diff_results, f, indent=2)

print("✅ Ekstraksi selesai!")
# print(json.dumps(median_diff_results["01m"].get("Eurasian Blue Tit", {}), indent=2))

# Plot Median Diff Wilcoxon/T-test "speaker birdcalls"

In [ ]:
experiment_colors = {
    "LabIR": "#fcc006",
    "SPIR": "lightgreen",
    "SA": "lightcoral"
}

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import json
from matplotlib.patches import Rectangle
from datetime import datetime
# ----------------------------
# 1. SETUP DATA & PARAMETERS
# ----------------------------
base_dir = "/content/drive/MyDrive/2025 Imperial/202508 Lab/"
# species = ["Eurasian Blue Tit", "Rose-ringed Parakeet"]
species = ["Eurasian Blue Tit"]
explabel = "Experiment"
experiments = ["SA", "LabIR", "SPIR"]
# distances = ["01m", "02m", "04m", "08m", "16m", "32m", "64m"]
distances = ["0", "1", "2", "3", "4", "5", "6"]

bar_width = 1
gap_between_exps = 0.1
dist_gap = 0.5

experiment_colors = {
    "SA": "lightcoral",
    "Lab-IR-BF": "#fcc006",
    "SP-IR-BF": "lightgreen"
}
legend_experiments = ["SA", "Lab-IR-BF", "SP-IR-BF"]

# ----------------------------
# 2. KUMPULKAN DATA
# ----------------------------

# Inisialisasi dictionaries dengan pendekatan yang lebih efisien
all_vals = {sp: {exp: [0] * len(distances) for exp in experiments} for sp in species}
all_pvals = {sp: {exp: [1] * len(distances) for exp in experiments} for sp in species}

# Proses data dengan error handling yang lebih baik
for species_name in species:
    for experiment_name in experiments:
        exp_dir = os.path.join(base_dir, f"{explabel}_{experiment_name}")

        # Load median difference data
        diff_file_path = os.path.join(exp_dir, "median_diff_summary.json")
        try:
            with open(diff_file_path, 'r') as f:
                diff_data = json.load(f)

            # Proses setiap distance
            for dist_idx, distance in enumerate(distances):
                val = diff_data.get(distance, {}).get(species_name, {}).get("median_diff", 0)
                all_vals[species_name][experiment_name][dist_idx] = val

        except FileNotFoundError:
            print(f"Warning: File not found - {diff_file_path}")
        except json.JSONDecodeError:
            print(f"Warning: Invalid JSON - {diff_file_path}")
        except Exception as e:
            print(f"Warning: Error processing {diff_file_path}: {e}")

        # Load p-value data
        pfile_name = f"stats_significance_{species_name.replace(' ', '_')}.json"
        pfile_path = os.path.join(exp_dir, pfile_name)
        try:
            with open(pfile_path, 'r') as f:
                sig_data = json.load(f)

            # Proses setiap distance
            for dist_idx, distance in enumerate(distances):
                p = sig_data.get("wilcoxon", {}).get(distance, {}).get("corrected_p_value",
                    sig_data.get("t_test", {}).get(distance, {}).get("corrected_p_value", 1.0))
                all_pvals[species_name][experiment_name][dist_idx] = p

        except FileNotFoundError:
            print(f"Warning: File not found - {pfile_path}")
        except json.JSONDecodeError:
            print(f"Warning: Invalid JSON - {pfile_path}")
        except Exception as e:
            print(f"Warning: Error processing {pfile_path}: {e}")

# ----------------------------
# 3. KALKULASI POSISI BAR
# ----------------------------
x_positions = np.zeros((len(distances), len(experiments)))
current_x = 0
for d_idx in range(len(distances)):
    x_positions[d_idx, :] = [current_x + i * (bar_width + gap_between_exps) for i in range(len(experiments))]
    current_x += len(experiments) * (bar_width + gap_between_exps) + dist_gap

# ----------------------------
# 4. SETUP SUBPLOTS (DUA SPESIES TERPISAH)
# ----------------------------
if len(species) == 1:
    fig, axes = plt.subplots(1, 1, figsize=(6, 4), sharex=True, gridspec_kw={'hspace': 0.05})
    axes = [axes] # Convert single Axes object to a list
else:
    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True, gridspec_kw={'hspace': 0.05})

# ----------------------------
# 5. PLOT DATA PER SPESIES
# ----------------------------
for sp_idx, species_name in enumerate(species):
    ax = axes[sp_idx]

    for exp_idx, experiment_name in enumerate(experiments):
        # Gunakan list comprehension yang lebih efisien
        xs = x_positions[:, exp_idx].tolist()
        ys = all_vals[species_name][experiment_name]
        ps = all_pvals[species_name][experiment_name]

        if experiment_name == "LabIR":
            experiment_legend = "Lab-IR-BF"
        elif experiment_name == "SPIR":
            experiment_legend = "SP-IR-BF"
        elif experiment_name == "SA":
            experiment_legend = "SA"

        ax.bar(
            xs, ys,
            width=bar_width,
            color=experiment_colors[experiment_legend],
            linewidth=1.5,
            alpha=0.8
        )

        # Tambahkan signifikansi dengan logika yang lebih bersih
        for x, y, p in zip(xs, ys, ps):
            if p < 0.05:
                marker = "***" if p < 0.001 else "**" if p < 0.01 else "*"
                text_y = y + (0 if y >= 0 else -0.01)
                ax.text(
                    x, text_y, marker,
                    ha='center', va='bottom' if y >= 0 else 'top',
                    color='black', fontsize=8, #fontweight='bold'
                )

    # Setel properti subplot
    ax.axhline(0, color='black', linestyle='--', alpha=0.5, lw=1)
    ax.set_ylabel(f"Median Difference Confidence Score", fontsize=11)
    ax.grid(axis='y', linestyle='--', alpha=0.2)

# ----------------------------
# 6. MODIFIKASI SESUAI REQUEST
# ----------------------------
# GANTI JUDUL UTAMA
# fig.suptitle(
#     f"{species[0]} in Virtual Sound Environment",
#     fontsize=14, y=0.97
# )

# Ekstrak semua nilai numerik dengan metode yang lebih efisien
all_values = [
    value
    for species_dict in all_vals.values()
    for exp_list in species_dict.values()
    for value in exp_list
]

# Hitung skala Y-axis yang aman
if all_values:
    y_max = max(all_values)
    y_min = min(all_values)
    y_range = y_max - y_min
    # print(f"Y-axis limits: ({y_min:f}, {y_max:f})")
    # Set skala seragam dengan margin
    margin_factor_min = 0.03 if y_min < 0 else 0.05
    margin_factor_max = 0.03 if y_max > 0 else 0.05

    y_limit_min = y_min - margin_factor_min if y_min < 0 else -0.05 * y_range
    y_limit_max = y_max + margin_factor_max if y_max > 0 else 0.05 * y_range
    # print(f"Y-axis limits: ({y_limit_min:f}, {y_limit_max:f})")
    for ax in axes:
        ax.set_ylim(y_limit_min, y_limit_max)

# HAPUS LABEL Y
for ax in axes:
    ax.tick_params(axis='y', length=0)


# LEGENDA DI KANAN BAWAH
experiment_handles = [
    Rectangle((0,0), 1, 1, facecolor=experiment_colors[exp], label=exp)
    for exp in legend_experiments
]
# fig.legend(
#     handles=experiment_handles,
#     loc='upper left',
#     bbox_to_anchor=(0.14, 0.86),
#     bbox_transform=fig.transFigure,
#     title="Methods",
#     title_fontsize=9,
#     fontsize=9,
#     frameon=True,
# )

fig.text(
    0.5, 0.93,
    "Wilcoxon/T-test (* p < 0.05; ** p < 0.01; *** p < 0.001)",
    ha='center', va='top',
    fontsize=11,
    color='black'
)

# Tambahkan label (a) dan (b) dengan posisi yang lebih konsisten
# if len(species) > 1:
#     axes[0].text(-0.1, 0.5, '(a)', transform=axes[0].transAxes, fontsize=12, va='center')
#     axes[1].text(-0.1, 0.5, '(b)', transform=axes[1].transAxes, fontsize=12, va='center')
# elif len(species) == 1:
#     axes[0].text(-0.12, 0.5, '(a)', transform=axes[0].transAxes, fontsize=12, va='center')


# ----------------------------
# 7. SETEL SUMBU X & SIMPAN
# ----------------------------
xtick_positions = np.mean(x_positions, axis=1)
axes[-1].set_xticks(xtick_positions) # Use the last axis for ticks and labels

# Konversi label distance dengan cara yang lebih fleksibel
# distances = ["01m", "02m", "04m", "08m", "16m", "32m", "64m"]
# distance_labels = [d.replace("m", " m").replace("0", "").replace("  ", " ").strip() for d in distances]
# distance_labels = [label if label != "m" else "1 m" for label in distance_labels]

distance_labels = ["+30dB", "+20dB", "+10dB", "0dB", "-10dB", "-20dB", "-30dB"]

axes[-1].set_xticklabels(distance_labels, fontsize=10)
axes[-1].set_xlabel("Signal-to-noise ratio level experiments", fontsize=11)

# Adjust layout untuk legenda
plt.subplots_adjust(right=0.85, bottom=0.2)

current_datetime = datetime.now()
timestamp = current_datetime.strftime("%Y%m%d")

# Simpan plot dengan error handling yang lebih baik
try:
    # fromexperiment = exp_dir.split("/")[-1].split("_")[0]  # Lebih aman menggunakan -1
    output_filename = f"{base_dir}lab_median_difference_{timestamp}.png"
    plt.savefig(output_filename, dpi=150, bbox_inches='tight')
    print(f"✅ Plot disimpan sebagai {output_filename}")
except Exception as e:
    print(f"❌ Error saat menyimpan plot: {e}")

plt.show()

In [ ]:
import pandas as pd
import numpy as np

# === CREATE MEDIAN DIFFERENCE TABLE ===
print("\n===== MEDIAN DIFFERENCE CONFIDENCE SCORE TABLE LAB-IR-BF =====")

# Prepare data for table
table_data = []
for sp in species:
    for exp in experiments:
        row = {
            "Species": sp,
            "Experiment": exp,
        }
        # Collect all median differences for this experiment
        exp_median_diffs = []
        for d_idx, distance in enumerate(distances):
            median_diff = all_vals[sp][exp][d_idx]
            row[distance] = median_diff
            exp_median_diffs.append(median_diff)

        # Calculate min and max for the experiment
        if exp_median_diffs:
            row["Minimum"] = np.min(exp_median_diffs)
            row["Maximum"] = np.max(exp_median_diffs)
        else:
            row["Minimum"] = None
            row["Maximum"] = None

        table_data.append(row)

# Create DataFrame and display
median_diff_df = pd.DataFrame(table_data)

# Optional: Pivot for better display if multiple species/experiments
# pivot_table = median_diff_df.pivot_table(index=['Species', 'Experiment'], values=distances)
# print(pivot_table)

print(median_diff_df.to_string(index=False))

# Plot Counts McNemar's Test "speaker birdcalls"

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import json
from matplotlib.patches import Rectangle

# ----------------------------
# 1. SETUP DATA & PARAMETERS
# ----------------------------
base_dir = "/content/drive/MyDrive/2025 Imperial/202508 Lab/"
species = ["Eurasian Blue Tit"] # , "Rose-ringed Parakeet"
explabel = "Experiment"
experiments = ["SA", "LabIR", "SPIR"]
# distances = ["01m", "02m", "04m", "08m", "16m", "32m", "64m"]
distances = ["0", "1", "2", "3", "4", "5", "6"]

bar_width = 1
gap_between_exps = 0.1
dist_gap = 0.5

experiment_colors = {
    "SA": "lightcoral",
    "Lab-IR-BF": "#fcc006",
    "SP-IR-BF": "lightgreen"
}
legend_experiments = ["SA", "Lab-IR-BF", "SP-IR-BF"]

# ----------------------------
# 2. KUMPULKAN DATA - DIPERBAIKI DENGAN PENANGANAN ERROR YANG LEBIH BAIK
# ----------------------------

# Inisialisasi dictionaries dengan nilai default
all_vals = {sp: {exp: [0] * len(distances) for exp in experiments} for sp in species}
all_pvals = {sp: {exp: [1] * len(distances) for exp in experiments} for sp in species}

# Kumpulkan data count dari path yang benar
# print("🔍 Collecting data from correct paths...")
for species_name in species:
    # print(f"  Processing species: {species_name}")
    for experiment_name in experiments:
        exp_dir = os.path.join(base_dir, f"{explabel}_{experiment_name}")
        # print(f"    Checking experiment: {experiment_name} in {exp_dir}")

        # Load count data dari path yang benar
        for dist_idx, distance in enumerate(distances):
            # Path baru: {exp_dir}/{distance}/processed_{species_name}.json
            processed_file = f"processed_{species_name.replace(' ', '_')}.json"
            processed_path = os.path.join(exp_dir, distance, processed_file)

            # print(f"      Checking: {processed_path}")

            try:
                if os.path.exists(processed_path):
                    with open(processed_path, 'r') as f:
                        processed_data = json.load(f)

                    # Ambil count dari beamformed -> count
                    count = processed_data.get("beamformed", {}).get("count")

                    # Validasi count
                    if count is not None and isinstance(count, (int, float)):
                        all_vals[species_name][experiment_name][dist_idx] = count
                        # print(f"        Distance {distance}: beamformed count = {count}")
                    else:
                        print(f"        ⚠️  Invalid count value in {processed_path}: {count}")
                        # Gunakan default value 0
                else:
                    print(f"        ⚠️  File not found: {processed_path}")
                    # Gunakan default value 0 yang sudah diinisialisasi

            except json.JSONDecodeError as e:
                print(f"        ❌ JSON Error in {processed_path}: {e}")
                # Gunakan default value 0
            except Exception as e:
                print(f"        ❌ Error loading count data: {e}")
                # Gunakan default value 0

        # Load p-value data (dari file stats_significance seperti sebelumnya)
        try:
            pfile = f"stats_significance_{species_name.replace(' ', '_')}.json"
            pfile_path = os.path.join(exp_dir, pfile)
            # print(f"      Loading p-values from: {pfile_path}")

            if os.path.exists(pfile_path):
                with open(pfile_path, 'r') as f:
                    sig_data = json.load(f)
                # print(f"      Successfully loaded p-value data")

                # Proses setiap distance
                for dist_idx, distance in enumerate(distances):
                    p = sig_data.get("mcnemar", {}).get(distance, {}).get("corrected_p_value", 1.0)
                    # Validasi p-value
                    if p is not None and isinstance(p, (int, float)):
                        all_pvals[species_name][experiment_name][dist_idx] = p
                    else:
                        all_pvals[species_name][experiment_name][dist_idx] = 1.0
                    # print(f"        Distance {distance}: p-value = {p}")
            else:
                print(f"      ⚠️  P-value file not found: {pfile_path}")

        except json.JSONDecodeError as e:
            print(f"      ❌ JSON Error in p-value file: {e}")
        except Exception as e:
            print(f"      ❌ Error loading p-value data: {e}")

# Debug: Tampilkan data yang terkumpul DAN validasi
# print("\n📊 Data Summary (with validation):")
for sp in species:
    for exp in experiments:
        counts = all_vals[sp][exp]
        # Validasi bahwa semua nilai adalah angka
        validated_counts = []
        for i, val in enumerate(counts):
            if val is None:
                validated_counts.append(0)
                print(f"  ⚠️  None value found for {sp} - {exp} distance {distances[i]}, replaced with 0")
            elif not isinstance(val, (int, float)):
                validated_counts.append(0)
                print(f"  ⚠️  Invalid type {type(val)} for {sp} - {exp} distance {distances[i]}, replaced with 0")
            else:
                validated_counts.append(val)
        all_vals[sp][exp] = validated_counts
        # print(f"  {sp} - {exp}: {validated_counts}")

# ----------------------------
# 3. KALKULASI POSISI BAR
# ----------------------------
x_positions = np.zeros((len(distances), len(experiments)))
current_x = 0
for d_idx in range(len(distances)):
    x_positions[d_idx, :] = [current_x + i * (bar_width + gap_between_exps) for i in range(len(experiments))]
    current_x += len(experiments) * (bar_width + gap_between_exps) + dist_gap

# ----------------------------
# 4. BUAT PLOT - DENGAN VALIDASI TAMBAHAN
# ----------------------------
if len(species) == 1:
    fig, axes = plt.subplots(1, 1, figsize=(6, 2), sharex=True, gridspec_kw={'hspace': 0.15})
    axes = [axes]
else:
    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True, gridspec_kw={'hspace': 0.15})

for sp_idx, species_name in enumerate(species):
    ax = axes[sp_idx]
    for exp_idx, experiment_name in enumerate(experiments):
        # Ambil data untuk plotting
        xs = x_positions[:, exp_idx].tolist()
        ys = all_vals[species_name][experiment_name]
        ps = all_pvals[species_name][experiment_name]

        # Validasi data sebelum plotting
        validated_ys = []
        for i, y_val in enumerate(ys):
            if y_val is None or not isinstance(y_val, (int, float)):
                print(f"⚠️  Invalid y-value for {species_name} - {experiment_name} at index {i}: {y_val}")
                validated_ys.append(0)
            else:
                validated_ys.append(y_val)
        ys = validated_ys

        # print(f"Plotting {species_name} - {experiment_name}: {ys}")
        # print(f"Experiment Legend: {experiment_legend}")
        # print(f"Experiment Colors: {experiment_colors[experiment_legend]}")
        if experiment_name == "LabIR":
            experiment_legend = "Lab-IR-BF"
        elif experiment_name == "SPIR":
            experiment_legend = "SP-IR-BF"
        elif experiment_name == "SA":
            experiment_legend = "SA"

        # Buat bar chart
        ax.bar(
            xs, ys,
            width=bar_width,
            color=experiment_colors[experiment_legend],
            linewidth=1.5,
            alpha=0.8
        )

        # Tambahkan marker signifikansi
        for x, y, p in zip(xs, ys, ps):
            if isinstance(p, (int, float)) and p < 0.05:
                marker = "***" if p < 0.001 else "**" if p < 0.01 else "*"
                text_y = y + (0 if y >= 0 else -0.01)
                ax.text(
                    x, text_y, marker,
                    ha='center', va='bottom' if y >= 0 else 'top',
                    color='black', fontsize=8, #fontweight='bold'
                )

    # Format subplot
    ax.axhline(0, color='black', linestyle='--', alpha=0.5, lw=1)
    ax.set_ylabel(f"{species_name}")
    ax.grid(axis='y', linestyle='--', alpha=0.2)

# ----------------------------
# 5. FORMAT JUDUL & AXIS
# ----------------------------
# fig.suptitle(
#     f"{species_name} Detection Counts",
#     fontsize=11, y=1.05, x=0.5, ha='center'
# )

# Atur batas y-axis dengan validasi
all_values = []
for sp in species:
    for exp in experiments:
        for val in all_vals[sp][exp]:
            if isinstance(val, (int, float)) and val is not None:
                all_values.append(val)

if all_values:
    y_max = max(all_values) if all_values else 1
    y_min = min(all_values) if all_values else 0
    y_range = y_max - y_min if y_max != y_min else 1

    for ax in axes:
        ax.set_ylim(
            y_min - 0.15 * abs(y_min) if y_min < 0 else -0.05 * y_range,
            y_max + 0.15 * y_max if y_max > 0 else 0.05 * y_range
        )

# Hilangkan tick labels y-axis
for ax in axes:
    ax.tick_params(axis='y', length=0)

# ----------------------------
# 6. TAMBAHKAN LEGENDA & ANOTASI
# ----------------------------
# Legenda eksperimen

experiment_handles = [
    Rectangle((0,0), 1, 1, facecolor=experiment_colors[exp], label=exp)
    for exp in legend_experiments
]
# fig.legend(
#     handles=experiment_handles,
#     loc='lower right',
#     bbox_to_anchor=(1.05, 0.45),
#     bbox_transform=fig.transFigure,
#     title="Methods",
#     title_fontsize=9,
#     fontsize=9,
#     frameon=False
# )

# Keterangan statistik
fig.text(
    0.5, 0.97,
    "McNemar's test (* p < 0.05; ** p < 0.01; *** p < 0.001)",
    ha='center', va='top',
    fontsize=10,
    color='black'
)
# 🔧 TAMBAHAN: Dua teks di atas subplot
# Tambahkan ini setelah membuat bar chart di setiap subplot:
# if len(species) == 1:
#     # axes[0].set_title('Eurasian Blue Tit', loc='center', fontsize=10, ha="center")

# else:
#     axes[0].set_title('(a) Eurasian Blue Tit', loc='center', fontsize=10, ha="center")
#     axes[1].set_title('(b) Rose-ringed Parakeet', loc='center', fontsize=10, ha="center")

# ----------------------------
# 7. SETEL SUMBU X & SIMPAN
# ----------------------------
xtick_positions = np.mean(x_positions, axis=1)
axes[0].set_xticks(xtick_positions)

# Format label distance
# distances = ["01m", "02m", "04m", "08m", "16m", "32m", "64m"]
# distance_labels = [d.replace("m", " m").replace("0", "").replace("  ", " ").strip() for d in distances]
# distance_labels = [label if label != "m" else "1 m" for label in distance_labels]

distance_labels = ["+30dB", "+20dB", "+10dB", "0dB", "-10dB", "-20dB", "-30dB"]

if len(species) == 1:
    axes[0].set_ylabel("Detection Count", fontsize=11)
    axes[0].set_xticklabels(distance_labels, fontsize=10)
    axes[0].set_xlabel("Signal-to-noise ratio level experiments", fontsize=11)
else:
    axes[0].set_ylabel("Detection Count", fontsize=11)
    axes[1].set_xticklabels(distance_labels, fontsize=10)
    axes[1].set_xlabel("Distance", fontsize=11)
    axes[1].set_ylabel("Detection Count", fontsize=11)
    axes[0].tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)

# Adjust layout
plt.subplots_adjust(right=0.85, bottom=0.2)
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d")
# Simpan plot
try:
    fromexperiment = exp_dir.split("/")[-1].split("_")[0]  # Gunakan -1 untuk path yang lebih aman
    output_filename = f"{base_dir}lab_count_{timestamp}.png"
    plt.savefig(output_filename, dpi=300, bbox_inches='tight')
    print(f"✅ Plot saved as {output_filename}")
except Exception as e:
    print(f"❌ Error saving plot: {e}")

plt.show()

# Plot Median Diff Wilcoxon/T-test "natural birdcalls"

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import json
from matplotlib.patches import Rectangle

# ----------------------------
# 1. SETUP DATA & PARAMETERS
# ----------------------------
base_dir = "/content/drive/MyDrive/2025 Imperial/202507 Silwood Park/"
explabel = "05Experiment"
experiments = ["SA", "LabIR", "SPIR"]
distances = ["01m"]

bar_width = 0.55
gap_between_exps = 0.1
dist_gap = 1.0

experiment_colors = {
    "LabIR": "#fcc006",
    "SPIR": "lightgreen",
    "SA": "lightcoral"
}

# ----------------------------
# 2. KUMPULKAN DATA
# ----------------------------

def determine_target_species(parent_dir, filename="paired_detections.json"):
    """
    Menentukan daftar target_species berdasarkan jumlah folder di parent_dir.
    Jika hanya satu folder, ambil semua spesies dari paired_detections.json.
    Jika banyak folder, gunakan daftar default.
    """
    folders = [f for f in os.listdir(parent_dir) if os.path.isdir(os.path.join(parent_dir, f))]
    if len(folders) <= 2 and len(folders) > 0:
        single_folder = folders[0]
        file_path = os.path.join(parent_dir, single_folder, filename)
        try:
            with open(file_path, 'r') as f:
                data = json.load(f)
                # Ambil semua kunci (spesies) dari data
                species_list = list(data.keys())
                print(f"Detected single folder '{single_folder}'. Target species set to: {species_list}")
                return species_list
        except FileNotFoundError:
            print(f"Warning: File {file_path} not found. Using default species.")
            return ["Rose-ringed Parakeet", "Eurasian Blue Tit"]
        except json.JSONDecodeError:
            print(f"Warning: File {file_path} is not a valid JSON. Using default species.")
            return ["Rose-ringed Parakeet", "Eurasian Blue Tit"]
        except Exception as e:
            print(f"Warning: Error processing {file_path}: {str(e)}. Using default species.")
            return ["Rose-ringed Parakeet", "Eurasian Blue Tit"]
    else:
        print(f"Multiple folders detected or no folders. Target species set to default: ['Rose-ringed Parakeet', 'Eurasian Blue Tit']")
        return ["Rose-ringed Parakeet", "Eurasian Blue Tit"]

# Tentukan spesies target dari LabIR (sebagai referensi)
reference_exp = experiments[0]  # LabIR
parent_dir = os.path.join(base_dir, f"{explabel}_{reference_exp}")
species = determine_target_species(parent_dir)

# Inisialisasi dictionaries untuk menyimpan data
all_vals = {sp: {exp: [] for exp in experiments} for sp in species}
all_pvals = {sp: {exp: [] for exp in experiments} for sp in species}

# Kumpulkan data untuk setiap eksperimen dan spesies
for exp in experiments:
    exp_dir = os.path.join(base_dir, f"{explabel}_{exp}")
    print(f"Processing {exp_dir}")

    # Load median difference data
    try:
        diff_file_path = os.path.join(exp_dir, "median_diff_summary.json")
        with open(diff_file_path) as f:
            diff_data = json.load(f)
        print(f"Successfully loaded median_diff_summary.json for {exp}")
    except FileNotFoundError:
        print(f"Warning: File not found: {diff_file_path}")
        diff_data = {}
    except json.JSONDecodeError:
        print(f"Warning: Invalid JSON in {diff_file_path}")
        diff_data = {}
    except Exception as e:
        print(f"Warning: Error loading {diff_file_path}: {e}")
        diff_data = {}

    # Load p-value data untuk setiap spesies
    pvalue_data = {}
    for sp in species:
        try:
            pfile = f"stats_significance_{sp.replace(' ', '_')}.json"
            pfile_path = os.path.join(exp_dir, pfile)
            with open(pfile_path) as f:
                pvalue_data[sp] = json.load(f)
            # print(f"Successfully loaded {pfile} for {sp} in {exp}")
        except FileNotFoundError:
            # print(f"Warning: File not found: {pfile_path}")
            pvalue_data[sp] = {}
        except json.JSONDecodeError:
            print(f"Warning: Invalid JSON in {pfile_path}")
            pvalue_data[sp] = {}
        except Exception as e:
            print(f"Warning: Error loading {pfile_path}: {e}")
            pvalue_data[sp] = {}

    # Proses data untuk setiap spesies dan distance
    for sp in species:
        for d in distances:
            # Median difference
            val = diff_data.get(d, {}).get(sp, {}).get("median_diff", 0)
            all_vals[sp][exp].append(val)

            # P-value
            sig_data = pvalue_data.get(sp, {})
            p = sig_data.get("wilcoxon", {}).get(d, {}).get("corrected_p_value",
                sig_data.get("t_test", {}).get(d, {}).get("corrected_p_value", 1.0))
            all_pvals[sp][exp].append(p)

# ----------------------------
# 3. FILTER SPESIES BERDASARKAN THRESHOLD
# ----------------------------

# Tentukan threshold untuk filtering
threshold = 0.01

# Cari spesies yang memenuhi kriteria (abs median diff >= threshold)
filtered_species = []
max_abs_values = {}  # Untuk menyimpan nilai max abs untuk setiap spesies

for sp in species:
    # Cari nilai absolut maksimum dari semua eksperimen dan distances
    abs_values = []
    for exp in experiments:
        for val in all_vals[sp][exp]:
            abs_values.append(abs(val))
    max_abs_val = max(abs_values) if abs_values else 0
    max_abs_values[sp] = max_abs_val

    # Filter spesies berdasarkan threshold
    if max_abs_val >= threshold:
        filtered_species.append(sp)
        print(f"Species {sp} included (max abs diff: {max_abs_val:.3f})")
    else:
        print(f"Species {sp} filtered out (max abs diff: {max_abs_val:.3f} < {threshold})")

# Jika tidak ada spesies yang memenuhi kriteria, tampilkan peringatan
if not filtered_species:
    print(f"Warning: No species found with absolute median difference >= {threshold}")
    print("Showing all species instead...")
    filtered_species = species.copy()

print(f"Plotting {len(filtered_species)} species: {filtered_species}")

# ----------------------------
# 4. BUAT PLOT
# ----------------------------

# Hitung posisi x untuk bars
x_positions = np.zeros((len(distances), len(experiments)))
current_x = 0
for d_idx, d in enumerate(distances):
    x_positions[d_idx, :] = [current_x + i * (bar_width + gap_between_exps) for i in range(len(experiments))]
    current_x += len(experiments) * (bar_width + gap_between_exps) + dist_gap

n_species = len(filtered_species)
if n_species == 0:
    print("No species to plot. Exiting.")
    exit()

fig, axes = plt.subplots(1, n_species, figsize=(3 * n_species, 3), sharex=True, gridspec_kw={'hspace': 0.1})

if n_species == 1:
    axes = [axes]

# Plot data untuk setiap spesies yang difilter
for sp_idx, sp in enumerate(filtered_species):
    ax = axes[sp_idx]

    for exp_idx, exp in enumerate(experiments):
        xs = [x_positions[d_idx, exp_idx] for d_idx in range(len(distances))]
        ys = all_vals[sp][exp]
        ps = all_pvals[sp][exp]

        ax.bar(
            xs, ys,
            width=bar_width,
            color=experiment_colors[exp],
            linewidth=1.5,
            alpha=0.8
        )

        # Tambahkan significance markers
        for i, (x, y, p) in enumerate(zip(xs, ys, ps)):
            if p < 0.05:
                marker = "***" if p < 0.001 else "**" if p < 0.01 else "*"
                offset = 0.0 if y >= 0 else -0.03
                text_y = y + offset
                ax.text(
                    x, text_y, marker,
                    ha='center', va='bottom' if y >= 0 else 'top',
                    color='black', fontsize=9, fontweight='bold'
                )

    ax.axhline(0, color='black', linestyle='--', alpha=0.5, lw=1)
    ax.grid(axis='y', linestyle='--', alpha=0.2)

    # Tambahkan nama spesies sebagai title
    ax.text(0.5, 1.05, f"{sp}",
            transform=ax.transAxes,
            ha='center',
            va='center',
            fontsize=11,
            rotation=0)

# Formatting umum
title_text = f"Median Difference in Species Confidence Scores (|diff| ≥ {threshold})"
fig.suptitle(
    title_text,
    fontsize=13, y=1.1, x=0.5, ha='center'
)

# Atur batas y berdasarkan data yang difilter
all_filtered_values = [
    val
    for sp in filtered_species
    for exp in experiments
    for val in all_vals[sp][exp]
]

if all_filtered_values:
    y_max = max(all_filtered_values) if all_filtered_values else 0.1
    y_min = min(all_filtered_values) if all_filtered_values else -0.1
    y_range = y_max - y_min

    for ax in axes:
        ax.set_ylim(
            y_min - 0.2 * abs(y_min) if y_min < 0 else -0.05 * y_range,
            y_max + 0.2 * y_max if y_max > 0 else 0.05 * y_range
        )

# Formatting ticks dan labels
for ax in axes:
    ax.tick_params(axis='y', length=0)
    ax.set_ylabel("")

# Set x-ticks
experiment_xticks = np.mean(x_positions, axis=0)
for ax in axes:
    ax.set_xticks(experiment_xticks)
    ax.set_xticklabels(experiments, fontsize=10)

# Tambahkan legenda
experiment_handles = [
    Rectangle((0,0), 1, 1, facecolor=experiment_colors[exp], label=exp)
    for exp in experiments
]
fig.legend(
    handles=experiment_handles,
    bbox_to_anchor=(0.1, 0.9),
    bbox_transform=fig.transFigure,
    title="Experiment",
    title_fontsize=10,
    fontsize=10,
    frameon=True
)

# Tambahkan keterangan significance
fig.text(
    0.5, 0.97,
    "Wilcoxon/T-test (* p < 0.05; ** p < 0.01; *** p < 0.001)",
    ha='center', va='bottom',
    fontsize=10,
    color='black'
)

# Tambahkan panel labels (a), (b), etc.
for i, ax in enumerate(axes):
    ax.text(0.5, -.25, f'({chr(97+i)})', transform=ax.transAxes, fontsize=12, ha='center', va='bottom')

# Simpan dan tampilkan plot
fromexperiment = exp_dir.split("/")[-1].split("_")[0] if '_' in exp_dir.split("/")[-1] else "experiment"
output_filename = f"{base_dir}/{fromexperiment}_median_difference_confidence.png"
try:
    plt.savefig(output_filename, dpi=300, bbox_inches='tight')
    print(f"Plot saved to: {output_filename}")
except Exception as e:
    print(f"Warning: Could not save plot: {e}")

plt.show()

# Plot Counts McNemar's Test "natural birdcalls"

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import json
from matplotlib.patches import Rectangle

# ----------------------------
# 1. SETUP DATA & PARAMETERS
# ----------------------------
base_dir = "/content/drive/MyDrive/2025 Imperial/202507 Silwood Park/"
explabel = "05Experiment"
experiments = ["SA", "LabIR", "SPIR"]
distances = ["01m"]

bar_width = 0.55
gap_between_exps = 0.1
dist_gap = 1.0

# ----------------------------
# 2. KUMPULKAN DATA
# ----------------------------

def determine_target_species(parent_dir, filename="paired_detections.json"):
    """
    Menentukan daftar target_species berdasarkan jumlah folder di parent_dir.
    Jika hanya satu folder, ambil semua spesies dari paired_detections.json.
    Jika banyak folder, gunakan daftar default.
    """
    try:
        folders = [f for f in os.listdir(parent_dir) if os.path.isdir(os.path.join(parent_dir, f))]
        if len(folders) <= 2 and len(folders) > 0:
            single_folder = folders[0]
            file_path = os.path.join(parent_dir, single_folder, filename)
            try:
                with open(file_path, 'r') as f:
                    data = json.load(f)
                    species_list = list(data.keys())
                    return species_list
            except FileNotFoundError:
                print(f"Error: File {file_path} not found. Using default species.")
                return ["Rose-ringed Parakeet", "Eurasian Blue Tit"]
            except json.JSONDecodeError:
                print(f"Error: File {file_path} is not a valid JSON. Using default species.")
                return ["Rose-ringed Parakeet", "Eurasian Blue Tit"]
            except Exception as e:
                print(f"Error processing {file_path}: {str(e)}. Using default species.")
                return ["Rose-ringed Parakeet", "Eurasian Blue Tit"]
        else:
            print(f"Multiple folders detected or no folders. Target species set to default: ['Rose-ringed Parakeet', 'Eurasian Blue Tit']")
            return ["Rose-ringed Parakeet", "Eurasian Blue Tit"]
    except Exception as e:
        print(f"Error accessing directory {parent_dir}: {str(e)}. Using default species.")
        return ["Rose-ringed Parakeet", "Eurasian Blue Tit"]

# Tentukan spesies target dari eksperimen pertama
reference_exp = experiments[0]
parent_dir = os.path.join(base_dir, f"{explabel}_{reference_exp}")
species = determine_target_species(parent_dir)

# Inisialisasi dictionaries untuk menyimpan data
all_vals = {sp: {exp: [0] * len(distances) for exp in experiments} for sp in species}
all_pvals = {sp: {exp: [1] * len(distances) for exp in experiments} for sp in species}

# Kumpulkan data untuk setiap spesies dan eksperimen
for species_name in species:
    for experiment_name in experiments:
        exp_dir = os.path.join(base_dir, f"{explabel}_{experiment_name}")

        # Load count data
        try:
            pfile = f"stats_significance_{species_name.replace(' ', '_')}.json"
            pfile_path = os.path.join(exp_dir, pfile)
            with open(pfile_path, 'r') as f:
                count_data = json.load(f)

            for dist_idx, distance in enumerate(distances):
                val = count_data.get("chi_square", {}).get(distance, {}).get("beam_count", 0)
                all_vals[species_name][experiment_name][dist_idx] = val

        except (FileNotFoundError, json.JSONDecodeError, KeyError) as e:
            print(f"Warning: Error loading count data for {species_name} in {experiment_name}: {e}")

        # Load p-value data
        try:
            pfile = f"stats_significance_{species_name.replace(' ', '_')}.json"
            pfile_path = os.path.join(exp_dir, pfile)
            with open(pfile_path, 'r') as f:
                sig_data = json.load(f)

            for dist_idx, distance in enumerate(distances):
                p = sig_data.get("mcnemar", {}).get(distance, {}).get("corrected_p_value", 1.0)
                all_pvals[species_name][experiment_name][dist_idx] = p

        except (FileNotFoundError, json.JSONDecodeError, KeyError) as e:
            print(f"Warning: Error loading p-value data for {species_name} in {experiment_name}: {e}")

# ----------------------------
# 3. FILTER SPESIES BERDASARKAN KEHADIRAN DI KETIGA EKSPERIMEN
# ----------------------------

# Cek spesies yang memiliki deteksi > 0 di semua eksperimen
filtered_species = []
for sp in species:
    # Cek apakah spesies ini terdeteksi (count > 0) di semua eksperimen
    detected_in_all = True
    for exp in experiments:
        # Cek jika ada setidaknya satu nilai count > 0 untuk spesies ini di eksperimen ini
        max_count = max(all_vals[sp][exp]) if all_vals[sp][exp] else 0
        if max_count <= 0:
            detected_in_all = False
            break

    if detected_in_all:
        filtered_species.append(sp)
        print(f"Species {sp} included (detected in all experiments)")
    else:
        print(f"Species {sp} filtered out (not detected in all experiments)")

# Jika tidak ada spesies yang memenuhi kriteria, gunakan semua spesies
if not filtered_species:
    print("Warning: No species detected in all experiments. Showing all species.")
    filtered_species = species.copy()

print(f"Plotting {len(filtered_species)} species: {filtered_species}")

# ----------------------------
# 4. KALKULASI POSISI BAR
# ----------------------------
x_positions = np.zeros((len(distances), len(experiments)))
current_x = 0
for d_idx in range(len(distances)):
    x_positions[d_idx, :] = [current_x + i * (bar_width + gap_between_exps) for i in range(len(experiments))]
    current_x += len(experiments) * (bar_width + gap_between_exps) + dist_gap

# ----------------------------
# 5. BUAT PLOT
# ----------------------------
n_species = len(filtered_species)
if n_species == 0:
    print("No species to plot. Exiting.")
    exit()

fig, axes = plt.subplots(1, n_species, figsize=(3 * n_species, 3), sharex=True, gridspec_kw={'hspace': 0.1})

if n_species == 1:
    axes = [axes]

for sp_idx, sp in enumerate(filtered_species):
    ax = axes[sp_idx]

    for e_idx, exp in enumerate(experiments):
        xs = [x_positions[d_idx, e_idx] for d_idx in range(len(distances))]
        ys = all_vals[sp][exp]
        ps = all_pvals[sp][exp]

        ax.bar(
            xs, ys,
            width=bar_width,
            color=experiment_colors[exp],
            linewidth=1.5,
            alpha=0.8
        )

        for i, (x, y, p) in enumerate(zip(xs, ys, ps)):
            if p < 0.05:
                marker = "***" if p < 0.001 else "**" if p < 0.01 else "*"
                offset = 0.0 if y >= 0 else -0.03
                text_y = y + offset
                ax.text(
                    x, text_y, marker,
                    ha='center', va='bottom' if y >= 0 else 'top',
                    color='black', fontsize=9, fontweight='bold'
                )

    ax.axhline(0, color='black', linestyle='--', alpha=0.5, lw=1)
    ax.set_ylabel(f"{sp}")
    ax.grid(axis='y', linestyle='--', alpha=0.2)

    ax.text(0.5, 1.05, f"{sp}",
            transform=ax.transAxes,
            ha='center',
            va='center',
            fontsize=11,
            rotation=0)

fig.suptitle(
    "Species Detection Counts",
    fontsize=13, y=1.1, x=0.5, ha='center'
)

# Hitung range nilai untuk y-axis (dari spesies yang difilter)
all_filtered_values = [
    val
    for sp in filtered_species
    for exp in experiments
    for val in all_vals[sp][exp]
]

if all_filtered_values:
    y_max = max(all_filtered_values)
    y_min = min(all_filtered_values)
    y_range = y_max - y_min

    for ax in axes:
        ax.set_ylim(
            y_min - 0.2 * abs(y_min) if y_min < 0 else -0.05 * y_range,
            y_max + 0.2 * y_max if y_max > 0 else 0.05 * y_range
        )

# Formatting
for ax in axes:
    ax.tick_params(axis='y', length=0)
    ax.set_ylabel("")

experiment_xticks = np.mean(x_positions, axis=0)
for ax in axes:
    ax.set_xticks(experiment_xticks)
    ax.set_xticklabels(experiments, fontsize=10)

# Legenda
experiment_handles = [
    Rectangle((0,0), 1, 1, facecolor=experiment_colors[exp], label=exp)
    for exp in experiments
]
fig.legend(
    handles=experiment_handles,
    bbox_to_anchor=(0.1, 0.9),
    bbox_transform=fig.transFigure,
    title="Methods",
    title_fontsize=10,
    fontsize=10,
    frameon=True,
)

# Keterangan statistik
fig.text(
    0.5, 0.97,
    "McNemar's test (* p < 0.05; ** p < 0.01; *** p < 0.001)",
    ha='center', va='bottom',
    fontsize=10,
    color='black'
)

# Panel labels
for i, ax in enumerate(axes):
    ax.text(0.5, -.25, f'({chr(97+i)})', transform=ax.transAxes, fontsize=12, ha='center', va='bottom')

# Simpan plot
try:
    fromexperiment = exp_dir.split("/")[-1].split("_")[0]  # Lebih aman menggunakan -1
    output_filename = f"{base_dir}/{fromexperiment}_count.png"
    plt.savefig(output_filename, dpi=300, bbox_inches='tight')
    print(f"Plot saved as {output_filename}")
except Exception as e:
    print(f"Error saving plot: {e}")

plt.show()

# Large Comparisons

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

# Set up the data structure
base = "/content/drive/MyDrive/2025 Imperial/"
lab = "/content/drive/MyDrive/2025 Imperial/202508 Lab/"
sp = "/content/drive/MyDrive/2025 Imperial/202507 Silwood Park/"

lab_folders = ["01Experiment_SA", "01Experiment_LabIR", "01Experiment_SPIR"]
sp_folders = ["04Experiment_SA", "04Experiment_LabIR", "04Experiment_SPIR"]

# Target species
TARGET_SPECIES = "Eurasian Blue Tit"

def load_and_process_results(results_path):
    """Load results.json and extract Eurasian Blue Tit detections with proper counting"""
    try:
        with open(results_path, 'r') as f:
            data = json.load(f)

        # Group detections by method and start_time
        method_detections = {
            'mono': defaultdict(list),  # start_time -> [detections]
            'sa': defaultdict(list),
            'labir': defaultdict(list),  # LabIR keyword is now 'LabIR'
            'spir': defaultdict(list)
        }

        for filename, detections_list in data.items():
            # Filter for target species
            target_detections = [d for d in detections_list if d['common_name'] == TARGET_SPECIES]

            # Classify by processing method and group by start_time
            for detection in target_detections:
                start_time = detection['start_time']

                if 'original_chan1' in filename.lower():
                    method_detections['mono'][start_time].append(detection)
                elif 'sa' in filename.lower():
                    method_detections['sa'][start_time].append(detection)
                elif 'labir' in filename.lower():  # LabIR files contain 'LabIR'
                    method_detections['labir'][start_time].append(detection)
                elif 'spir' in filename.lower():
                    method_detections['spir'][start_time].append(detection)

        # For each method and start_time, keep only the detection with highest confidence
        processed_detections = {}
        for method, time_detections in method_detections.items():
            processed_detections[method] = []
            for start_time, detections_at_time in time_detections.items():
                if detections_at_time:
                    # Get detection with highest confidence for this start_time
                    best_detection = max(detections_at_time, key=lambda x: x['confidence'])
                    best_detection['start_time'] = start_time  # Ensure start_time is preserved
                    processed_detections[method].append(best_detection)

        return processed_detections
    except Exception as e:
        print(f"Error loading {results_path}: {e}")
        return None

def extract_experiment_data(base_path, folders, condition_type):
    """Extract data from all experiments - proper approach"""
    all_data = []
    mono_collected = set()  # Track which conditions already have mono data

    # First, collect all condition folders from any experiment (they should be the same)
    sample_exp_path = os.path.join(base_path, folders[0])
    if not os.path.exists(sample_exp_path):
        print(f"Warning: {sample_exp_path} does not exist")
        return []

    condition_folders = [f for f in os.listdir(sample_exp_path) if os.path.isdir(os.path.join(sample_exp_path, f))]
    condition_folders.sort()

    # Process each condition
    for i, condition_folder in enumerate(condition_folders):
        # Determine condition value
        if condition_type == 'SNR':
            condition_value = 30 - (i * 10)  # +30, +20, +10, 0, -10, -20, -30
        else:  # Distance
            condition_value = 2 ** i  # 1, 2, 4, 8, 16, 32, 64

        # Process each experiment for this condition
        for exp_folder in folders:
            method = exp_folder.split('_')[1]  # Extract SA, LabIR, or SPIR
            exp_path = os.path.join(base_path, exp_folder)

            if not os.path.exists(exp_path):
                print(f"Warning: {exp_path} does not exist")
                continue

            results_path = os.path.join(exp_path, condition_folder, 'results.json')

            if not os.path.exists(results_path):
                print(f"Warning: {results_path} does not exist")
                continue

            detections = load_and_process_results(results_path)
            if detections is None:
                continue

            # Add mono data only once per condition (from first available experiment)
            if (condition_value, condition_type) not in mono_collected and 'mono' in detections:
                mono_detections = detections['mono']
                if mono_detections:
                    detection_count = len(mono_detections)
                    max_confidence = max(d['confidence'] for d in mono_detections)
                    avg_confidence = np.mean([d['confidence'] for d in mono_detections])
                    detection_rate = min(detection_count, 1)
                else:
                    detection_count = 0
                    max_confidence = 0
                    avg_confidence = 0
                    detection_rate = 0

                all_data.append({
                    'condition_type': condition_type,
                    'condition_value': condition_value,
                    'method': 'Mono',
                    'detection_method': 'mono',
                    'detection_rate': detection_rate,
                    'max_confidence': max_confidence,
                    'avg_confidence': avg_confidence,
                    'detection_count': detection_count,
                    'subfolder': condition_folder
                })
                mono_collected.add((condition_value, condition_type))

            # Process the specific method (SA, LabIR, SPIR)
            method_key = method.lower()
            if method_key in detections:
                method_detections = detections[method_key]

                if method_detections:
                    detection_count = len(method_detections)
                    max_confidence = max(d['confidence'] for d in method_detections)
                    avg_confidence = np.mean([d['confidence'] for d in method_detections])
                    detection_rate = min(detection_count, 1)
                else:
                    detection_count = 0
                    max_confidence = 0
                    avg_confidence = 0
                    detection_rate = 0

                all_data.append({
                    'condition_type': condition_type,
                    'condition_value': condition_value,
                    'method': method,
                    'detection_method': method_key,
                    'detection_rate': detection_rate,
                    'max_confidence': max_confidence,
                    'avg_confidence': avg_confidence,
                    'detection_count': detection_count,
                    'subfolder': condition_folder
                })

    return all_data

# Extract data from both lab and SP experiments
print("Extracting Lab data...")
lab_data = extract_experiment_data(lab, lab_folders, 'SNR')

print("Extracting Silwood Park data...")
sp_data = extract_experiment_data(sp, sp_folders, 'Distance')

# Combine all data
all_data = lab_data + sp_data
df = pd.DataFrame(all_data)

print(f"Total records: {len(df)}")
print(f"Lab records: {len([d for d in all_data if d['condition_type'] == 'SNR'])}")
print(f"SP records: {len([d for d in all_data if d['condition_type'] == 'Distance'])}")
print(f"Mono records: {len([d for d in all_data if d['detection_method'] == 'mono'])}")
print(f"SA records: {len([d for d in all_data if d['detection_method'] == 'sa'])}")
print(f"LabIR records: {len([d for d in all_data if d['detection_method'] == 'labir'])}")
print(f"SPIR records: {len([d for d in all_data if d['detection_method'] == 'spir'])}")

# Display sample of the data
print("\nSample of extracted data:")
print(df.head(10))

# Create comprehensive analysis plots
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Eurasian Blue Tit Detection Analysis: Lab vs Silwood Park', fontsize=16, fontweight='bold')

# 1. Detection Rate by Condition - Lab (SNR)
lab_df = df[df['condition_type'] == 'SNR']
detection_rate_lab = lab_df.groupby(['condition_value', 'detection_method'])['detection_rate'].mean().reset_index()
detection_rate_lab_pivot = detection_rate_lab.pivot(index='condition_value', columns='detection_method', values='detection_rate')

axes[0,0].plot(detection_rate_lab_pivot.index, detection_rate_lab_pivot['mono'], 'o-', label='Mono', linewidth=2, markersize=6)
axes[0,0].plot(detection_rate_lab_pivot.index, detection_rate_lab_pivot['sa'], 's-', label='SA', linewidth=2, markersize=6)
if 'labir' in detection_rate_lab_pivot.columns:
    axes[0,0].plot(detection_rate_lab_pivot.index, detection_rate_lab_pivot['labir'], '^-', label='LabIR', linewidth=2, markersize=6)
if 'spir' in detection_rate_lab_pivot.columns:
    axes[0,0].plot(detection_rate_lab_pivot.index, detection_rate_lab_pivot['spir'], 'd-', label='SPIR', linewidth=2, markersize=6)
axes[0,0].set_title('Detection Rate - Lab (SNR Conditions)', fontweight='bold')
axes[0,0].set_xlabel('SNR (dB)')
axes[0,0].set_ylabel('Detection Rate')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)
axes[0,0].set_ylim(0, 1.1)

# 2. Detection Rate by Condition - SP (Distance)
sp_df = df[df['condition_type'] == 'Distance']
detection_rate_sp = sp_df.groupby(['condition_value', 'detection_method'])['detection_rate'].mean().reset_index()
detection_rate_sp_pivot = detection_rate_sp.pivot(index='condition_value', columns='detection_method', values='detection_rate')

axes[0,1].plot(detection_rate_sp_pivot.index, detection_rate_sp_pivot['mono'], 'o-', label='Mono', linewidth=2, markersize=6)
axes[0,1].plot(detection_rate_sp_pivot.index, detection_rate_sp_pivot['sa'], 's-', label='SA', linewidth=2, markersize=6)
if 'labir' in detection_rate_sp_pivot.columns:
    axes[0,1].plot(detection_rate_sp_pivot.index, detection_rate_sp_pivot['labir'], '^-', label='LabIR', linewidth=2, markersize=6)
if 'spir' in detection_rate_sp_pivot.columns:
    axes[0,1].plot(detection_rate_sp_pivot.index, detection_rate_sp_pivot['spir'], 'd-', label='SPIR', linewidth=2, markersize=6)
axes[0,1].set_title('Detection Rate - Silwood Park (Distance)', fontweight='bold')
axes[0,1].set_xlabel('Distance (m)')
axes[0,1].set_ylabel('Detection Rate')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)
axes[0,1].set_xscale('log', base=2)
axes[0,1].set_ylim(0, 1.1)

# 3. Confidence Score by Condition - Lab
conf_lab = lab_df[lab_df['max_confidence'] > 0].groupby(['condition_value', 'detection_method'])['max_confidence'].mean().reset_index()
conf_lab_pivot = conf_lab.pivot(index='condition_value', columns='detection_method', values='max_confidence')

axes[0,2].plot(conf_lab_pivot.index, conf_lab_pivot['mono'], 'o-', label='Mono', linewidth=2, markersize=6)
axes[0,2].plot(conf_lab_pivot.index, conf_lab_pivot['sa'], 's-', label='SA', linewidth=2, markersize=6)
if 'labir' in conf_lab_pivot.columns:
    axes[0,2].plot(conf_lab_pivot.index, conf_lab_pivot['labir'], '^-', label='LabIR', linewidth=2, markersize=6)
if 'spir' in conf_lab_pivot.columns:
    axes[0,2].plot(conf_lab_pivot.index, conf_lab_pivot['spir'], 'd-', label='SPIR', linewidth=2, markersize=6)
axes[0,2].set_title('Max Confidence Score - Lab (SNR)', fontweight='bold')
axes[0,2].set_xlabel('SNR (dB)')
axes[0,2].set_ylabel('Confidence Score')
axes[0,2].legend()
axes[0,2].grid(True, alpha=0.3)

# 4. Confidence Score by Condition - SP
conf_sp = sp_df[sp_df['max_confidence'] > 0].groupby(['condition_value', 'detection_method'])['max_confidence'].mean().reset_index()
conf_sp_pivot = conf_sp.pivot(index='condition_value', columns='detection_method', values='max_confidence')

axes[1,0].plot(conf_sp_pivot.index, conf_sp_pivot['mono'], 'o-', label='Mono', linewidth=2, markersize=6)
axes[1,0].plot(conf_sp_pivot.index, conf_sp_pivot['sa'], 's-', label='SA', linewidth=2, markersize=6)
if 'labir' in conf_sp_pivot.columns:
    axes[1,0].plot(conf_sp_pivot.index, conf_sp_pivot['labir'], '^-', label='LabIR', linewidth=2, markersize=6)
if 'spir' in conf_sp_pivot.columns:
    axes[1,0].plot(conf_sp_pivot.index, conf_sp_pivot['spir'], 'd-', label='SPIR', linewidth=2, markersize=6)
axes[1,0].set_title('Max Confidence Score - Silwood Park (Distance)', fontweight='bold')
axes[1,0].set_xlabel('Distance (m)')
axes[1,0].set_ylabel('Confidence Score')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)
axes[1,0].set_xscale('log', base=2)

# 5. Overall Detection Rate Comparison
overall_detection = df.groupby(['condition_type', 'detection_method'])['detection_rate'].mean().reset_index()
overall_pivot = overall_detection.pivot(index='detection_method', columns='condition_type', values='detection_rate')

x = np.arange(len(overall_pivot.index))
width = 0.35

bars1 = axes[1,1].bar(x - width/2, overall_pivot['SNR'], width, label='Lab (SNR)', alpha=0.8)
bars2 = axes[1,1].bar(x + width/2, overall_pivot['Distance'], width, label='Silwood Park (Distance)', alpha=0.8)

axes[1,1].set_title('Overall Detection Rate Comparison', fontweight='bold')
axes[1,1].set_xlabel('Detection Method')
axes[1,1].set_ylabel('Average Detection Rate')
axes[1,1].set_xticks(x)
axes[1,1].set_xticklabels(overall_pivot.index)
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar in bars1:
    height = bar.get_height()
    axes[1,1].text(bar.get_x() + bar.get_width()/2., height + 0.01,
                   f'{height:.2f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    height = bar.get_height()
    axes[1,1].text(bar.get_x() + bar.get_width()/2., height + 0.01,
                   f'{height:.2f}', ha='center', va='bottom', fontsize=9)

# 6. Overall Confidence Score Comparison
overall_conf = df[df['max_confidence'] > 0].groupby(['condition_type', 'detection_method'])['max_confidence'].mean().reset_index()
overall_conf_pivot = overall_conf.pivot(index='detection_method', columns='condition_type', values='max_confidence')

x = np.arange(len(overall_conf_pivot.index))
bars3 = axes[1,2].bar(x - width/2, overall_conf_pivot['SNR'], width, label='Lab (SNR)', alpha=0.8)
bars4 = axes[1,2].bar(x + width/2, overall_conf_pivot['Distance'], width, label='Silwood Park (Distance)', alpha=0.8)

axes[1,2].set_title('Overall Confidence Score Comparison', fontweight='bold')
axes[1,2].set_xlabel('Detection Method')
axes[1,2].set_ylabel('Average Confidence Score')
axes[1,2].set_xticks(x)
axes[1,2].set_xticklabels(overall_conf_pivot.index)
axes[1,2].legend()
axes[1,2].grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar in bars3:
    height = bar.get_height()
    if not np.isnan(height):
        axes[1,2].text(bar.get_x() + bar.get_width()/2., height + 0.01,
                       f'{height:.2f}', ha='center', va='bottom', fontsize=9)
for bar in bars4:
    height = bar.get_height()
    if not np.isnan(height):
        axes[1,2].text(bar.get_x() + bar.get_width()/2., height + 0.01,
                       f'{height:.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

# Print summary statistics
# Print detailed detection count information
print("\n4. DETECTION COUNT DETAILS (Unique Start Times)")
print("-" * 40)
count_summary = df.groupby(['condition_type', 'detection_method'])['detection_count'].agg(['sum', 'mean', 'std']).round(3)
print(count_summary)

print("\n5. DETECTION COUNT BY CONDITION")
print("-" * 40)
for condition in ['SNR', 'Distance']:
    print(f"\n{condition} Condition - Detection Counts:")
    cond_data = df[df['condition_type'] == condition]
    count_by_condition = cond_data.groupby(['condition_value', 'detection_method'])['detection_count'].sum().reset_index()
    count_pivot = count_by_condition.pivot(index='condition_value', columns='detection_method', values='detection_count')
    print(count_pivot.fillna(0).astype(int))
print("SUMMARY STATISTICS")
print("="*80)

# Detection Rate Summary
print("\n1. DETECTION RATE SUMMARY")
print("-" * 40)
detection_summary = df.groupby(['condition_type', 'detection_method'])['detection_rate'].agg(['mean', 'std', 'count']).round(3)
print(detection_summary)

# Confidence Score Summary (only for successful detections)
print("\n2. CONFIDENCE SCORE SUMMARY (Successful Detections Only)")
print("-" * 40)
conf_summary = df[df['max_confidence'] > 0].groupby(['condition_type', 'detection_method'])['max_confidence'].agg(['mean', 'std', 'count']).round(3)
print(conf_summary)

# Method Performance Improvement
print("\n3. PERFORMANCE IMPROVEMENT vs MONO BASELINE")
print("-" * 40)
for condition in ['SNR', 'Distance']:
    print(f"\n{condition} Condition:")
    cond_data = df[df['condition_type'] == condition]
    mono_detection = cond_data[cond_data['detection_method'] == 'mono']['detection_rate'].mean()

    for method in ['sa', 'labir', 'spir']:
        method_data = cond_data[cond_data['detection_method'] == method]
        if len(method_data) > 0:
            method_detection = method_data['detection_rate'].mean()
            improvement = ((method_detection - mono_detection) / mono_detection * 100) if mono_detection > 0 else 0
            method_name = 'LabIR' if method == 'labir' else method.upper()
            print(f"  {method_name}: {method_detection:.3f} ({improvement:+.1f}% vs Mono)")

print("\n" + "="*80)

# Create additional plot for detection counts
fig2, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Detection count by condition - Lab
lab_count = df[df['condition_type'] == 'SNR'].groupby(['condition_value', 'detection_method'])['detection_count'].sum().reset_index()
lab_count_pivot = lab_count.pivot(index='condition_value', columns='detection_method', values='detection_count').fillna(0)
lab_count_pivot = lab_count_pivot.sort_index(ascending=False)  # Reverse X-axis for lab data

for method in lab_count_pivot.columns:
    if method == 'mono':
        ax1.plot(lab_count_pivot.index, lab_count_pivot[method], 'o-', label='Mono', linewidth=2, markersize=6)
    elif method == 'sa':
        ax1.plot(lab_count_pivot.index, lab_count_pivot[method], 's-', label='SA', linewidth=2, markersize=6)
    elif method == 'labir':
        ax1.plot(lab_count_pivot.index, lab_count_pivot[method], '^-', label='LabIR', linewidth=2, markersize=6)
    elif method == 'spir':
        ax1.plot(lab_count_pivot.index, lab_count_pivot[method], 'd-', label='SPIR', linewidth=2, markersize=6)

ax1.set_title('Detection Count - Lab (SNR Conditions)', fontweight='bold')
ax1.set_xlabel('SNR (dB)')
ax1.set_ylabel('Total Detection Count')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Detection count by condition - SP
sp_count = df[df['condition_type'] == 'Distance'].groupby(['condition_value', 'detection_method'])['detection_count'].sum().reset_index()
sp_count_pivot = sp_count.pivot(index='condition_value', columns='detection_method', values='detection_count').fillna(0)

for method in sp_count_pivot.columns:
    if method == 'mono':
        ax2.plot(sp_count_pivot.index, sp_count_pivot[method], 'o-', label='Mono', linewidth=2, markersize=6)
    elif method == 'sa':
        ax2.plot(sp_count_pivot.index, sp_count_pivot[method], 's-', label='SA', linewidth=2, markersize=6)
    elif method == 'labir':
        ax2.plot(sp_count_pivot.index, sp_count_pivot[method], '^-', label='LabIR', linewidth=2, markersize=6)
    elif method == 'spir':
        ax2.plot(sp_count_pivot.index, sp_count_pivot[method], 'd-', label='SPIR', linewidth=2, markersize=6)

ax2.set_title('Detection Count - Silwood Park (Distance)', fontweight='bold')
ax2.set_xlabel('Distance (m)')
ax2.set_ylabel('Total Detection Count')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_xscale('log', base=2)

plt.tight_layout()
plt.show()

# Unified Analysis

# detections_summary

In [ ]:
!pip install pandas matplotlib seaborn

In [ ]:
import os
import json
import pandas as pd
from datetime import datetime
import re

# === Konfigurasi Path ===
base = "/content/drive/MyDrive/2025 Imperial"
lab = "202508 Lab"
silwood = "202507 Silwood Park"

# Eksperimen Lab dan SP
lab_experiments = ["01Experiment_SA", "01Experiment_LabIR", "01Experiment_SPIR"]
sp_experiments = ["04Experiment_SA", "04Experiment_LabIR", "04Experiment_SPIR"]

# Path subfolder
snr_folders = [str(i) for i in range(7)]
distance_folders = ["01m", "02m", "04m", "08m", "16m", "32m", "64m"]

# Target spesies
target_species = "Eurasian Blue Tit"

# Mapping file ke metodologi
def method_from_filename(fname):
    fname_lower = fname.lower()
    if fname_lower == "original_chan1.mp3":
        return "Mono"
    elif "sa" in fname_lower:
        return "SA"
    elif "labir" in fname_lower:
        return "LabIR"
    elif "spir" in fname_lower:
        return "SPIR"
    return "Unknown"

# ===== PENGUMPULAN DATA =====
data_rows = []

def process_folder(experiment_type, location, rank, subfolder, folder_path):
    results_path = os.path.join(folder_path, "results.json")
    if not os.path.exists(results_path):
        print(f"[❌] File tidak ditemukan: {results_path}")
        return

    try:
        with open(results_path, "r") as f:
            data = json.load(f)
    except Exception as e:
        print(f"[❌] Error membaca {results_path}: {e}")
        return

    # Proses semua file dalam results.json
    for fname, detections in data.items():
        method = method_from_filename(fname)
        if method == "Unknown":
            continue

        for entry in detections:
            if entry.get("common_name") == target_species:
                data_rows.append({
                    "Location": location,
                    "ExperimentType": experiment_type,
                    "EksperimenID": experiment_type.split("Experiment_")[0].replace("0", "", 1),
                    "Rank": rank,
                    "Distance_or_SNR": subfolder,
                    "Method": method,
                    "FileName": fname,
                    "Folder_Path": folder_path,
                    "CommonName": entry["common_name"],
                    "ScientificName": entry["scientific_name"],
                    "StartTime": entry["start_time"],
                    "EndTime": entry["end_time"],
                    "Confidence": entry["confidence"],
                    "Label": entry["label"],
                    "Time_Stamp": datetime.now()
                })

# === PROSES LAB ===
for exp in lab_experiments:
    for i, snr in enumerate(snr_folders):
        folder_path = os.path.join(base, lab, exp, snr)
        process_folder(exp, "Lab", i, snr, folder_path)

# === PROSES SILWOOD PARK ===
for exp in sp_experiments:
    for i, dist in enumerate(distance_folders):
        folder_path = os.path.join(base, silwood, exp, dist)
        process_folder(exp, "Silwood Park", i, dist, folder_path)

# === SIMPAN KE CSV ===
if data_rows:
    df = pd.DataFrame(data_rows)
    df = df[[
        "Location", "ExperimentType", "EksperimenID", "Rank",
        "Distance_or_SNR", "Method", "FileName", "Folder_Path",
        "CommonName", "ScientificName",
        "StartTime", "EndTime", "Confidence", "Label", "Time_Stamp"
    ]]

    today = datetime.now().strftime("%Y%m%d")
    output_path = os.path.join(base, f"{today}_detections_summary.csv")
    df.to_csv(output_path, index=False)
    print(f"\n✅ File CSV disimpan: {output_path}")
else:
    print("⚠️ Tidak ada deteksi spesies ditemukan.")

# comprehensive_beamforming_analysis

In [ ]:
import os
import json
import re
import pandas as pd
import numpy as np
from scipy.stats import binomtest, ttest_rel, wilcoxon
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# === Konfigurasi ===
base = "/content/drive/MyDrive/2025 Imperial"
lab_subdir = "202508 Lab"
silwood_subdir = "202509 Silwood Park"
TIMESTAMP = datetime.now().strftime("%Y%m%d")
lab_experiments = ["Experiment_SA", "Experiment_LabIR", "Experiment_SPIR"]
sp_experiments = ["Experiment_SA", "Experiment_LabIR", "Experiment_SPIR"]
snr_folders = [str(i) for i in range(7)]
distance_folders = ["01m", "02m", "04m", "08m", "16m", "32m", "64m"]
target_species = "Eurasian Blue Tit"

# Total calls per rank (TIDAK dibagi 7 - sesuai permintaan)
calls_per_rank = {'Lab': 124, 'Silwood Park': 240}  # Full dataset per rank
total_design = {'Lab': 124, 'Silwood Park': 240}

# Mapping dari nama file ke metode
def method_from_filename(fname):
    fname_lower = fname.lower()
    if fname_lower == "original_chan1.mp3":
        return "Mono"
    elif "sa" in fname_lower:
        return "SA"
    elif "labir" in fname_lower:
        return "LabIR"
    elif "spir" in fname_lower:
        return "SPIR"
    return "Unknown"

# === KUMPULKAN DETEKSI LENGKAP DENGAN RANK INFO ===

data_rows = []

def process_folder(folder_experiment, location, subfolder):
    folder_path = os.path.join(base, location, folder_experiment, subfolder)
    results_path = os.path.join(folder_path, "results.json")
    if not os.path.exists(results_path):
        return

    with open(results_path, 'r') as f:
        data = json.load(f)

    # Tentukan apakah ini folder LabIR
    is_labir_folder = "LabIR" in folder_experiment

    for fname, detections in data.items():
        method = method_from_filename(fname)
        if method == "Unknown":
            continue

        # FILTER: Mono hanya diambil dari folder LabIR
        if method == "Mono" and not is_labir_folder:
            continue  # Skip Mono dari folder SA dan SPIR

        for entry in detections:
            if entry.get("common_name") != target_species:
                continue

            # FILTER CONFIDENCE: Hanya ambil deteksi dengan confidence > 0.3
            confidence_score = float(entry["confidence"])
            if confidence_score <= 0.3:
                continue  # Skip deteksi dengan confidence rendah

            # Mapping rank berdasarkan subfolder
            if location == lab_subdir:
                rank = int(subfolder)  # 0,1,2,3,4,5,6 for SNR
                rank_label = f"SNR_{subfolder}"
            else:
                rank = distance_folders.index(subfolder)  # 0,1,2,3,4,5,6 for distance
                rank_label = f"Dist_{subfolder}"

            data_rows.append({
                "Location": location,
                "RecordingMethod": method,
                "Rank": rank,
                "RankLabel": rank_label,
                "FolderExperiment": folder_experiment,
                "StartTime": float(entry["start_time"]),
                "Confidence": confidence_score,  # Gunakan confidence_score yang sudah di-filter
                "FileName": fname
            })

# Proses semua eksperimen
for exp in lab_experiments:
    for snr in snr_folders:
        process_folder(exp, lab_subdir, snr)

for exp in sp_experiments:
    for dist in distance_folders:
        process_folder(exp, silwood_subdir, dist)

# === NORMALISASI LOKASI ===
df = pd.DataFrame(data_rows)
df["NormLocation"] = df["Location"].apply(
    lambda x: "Lab" if "Lab" in x else "Silwood Park"
)

print(f"Total data terkumpul: {len(df)}")

# === SORTING DAN DEDUPLICATION PER RANK ===
df_sorted = df.sort_values([
    'NormLocation',
    'RecordingMethod',
    'Rank',
    'StartTime',
    'Confidence'
], ascending=[True, True, True, True, False]).reset_index(drop=True)

# Pembulatan waktu
df_sorted["TimeRounded"] = np.floor(df_sorted["StartTime"] + 0.5).astype(int)

# Deduplication: Ambil confidence tertinggi per detik per rank per metode
valid_detections = (
    df_sorted.groupby(['NormLocation', 'RecordingMethod', 'Rank', 'RankLabel', 'TimeRounded'])
    .first()  # Ambil yang pertama (confidence tertinggi)
    .reset_index()
)

print(f"Data setelah deduplication: {len(valid_detections)}")

# === BUAT TEMPLATE LENGKAP UNTUK SEMUA KOMBINASI ===
locations = ['Lab', 'Silwood Park']
methods = ['Mono', 'SA', 'LabIR', 'SPIR']
ranks = list(range(7))
rank_labels_lab = [f"SNR_{i}" for i in range(7)]
rank_labels_sp = [f"Dist_{dist}" for dist in distance_folders]

template_rows = []
for loc in locations:
    rank_labels = rank_labels_lab if loc == 'Lab' else rank_labels_sp
    for method in methods:
        for rank in ranks:
            template_rows.append({
                'NormLocation': loc,
                'RecordingMethod': method,
                'Rank': rank,
                'RankLabel': rank_labels[rank],
                'CallsPerRank': calls_per_rank[loc]  # Full dataset
            })

template_df = pd.DataFrame(template_rows)

# === HITUNG DETECTION RATE PER RANK ===
detection_counts_per_rank = (
    valid_detections.groupby(['NormLocation', 'RecordingMethod', 'Rank', 'RankLabel'])
    .size()
    .reset_index(name='ValidDetections')
)

# MERGE dengan template untuk memastikan semua kombinasi ada
complete_results = template_df.merge(
    detection_counts_per_rank,
    on=['NormLocation', 'RecordingMethod', 'Rank', 'RankLabel'],
    how='left'
)

# Fill missing values dengan 0
complete_results['ValidDetections'] = complete_results['ValidDetections'].fillna(0).astype(int)
complete_results['DetectionRate'] = (
    complete_results['ValidDetections'] / complete_results['CallsPerRank']
)

print(f"\nComplete results shape: {complete_results.shape}")

# === TAMPILKAN HASIL PER RANK (TABEL LENGKAP) ===
print(f"\n===== DETECTION RATE PER RANK (COMPLETE WITH ZEROS) =====")

# Lab results
lab_results = complete_results[complete_results['NormLocation'] == 'Lab'].copy()
lab_results = lab_results.sort_values(['RecordingMethod', 'Rank'])
print(f"\n🔬 LAB RESULTS (SNR degradation):")
for method in ['Mono', 'SA', 'LabIR', 'SPIR']:
    method_data = lab_results[lab_results['RecordingMethod'] == method]
    print(f"\n{method}:")
    display_data = method_data[['RankLabel', 'ValidDetections', 'CallsPerRank', 'DetectionRate']].round(3)
    print(display_data.to_string(index=False))

# Silwood Park results
sp_results = complete_results[complete_results['NormLocation'] == 'Silwood Park'].copy()
sp_results = sp_results.sort_values(['RecordingMethod', 'Rank'])
print(f"\n🌲 SILWOOD PARK RESULTS (Distance increase):")
for method in ['Mono', 'SA', 'LabIR', 'SPIR']:
    method_data = sp_results[sp_results['RecordingMethod'] == method]
    print(f"\n{method}:")
    display_data = method_data[['RankLabel', 'ValidDetections', 'CallsPerRank', 'DetectionRate']].round(3)
    print(display_data.to_string(index=False))

# === SUMMARY TABLE ===
print(f"\n===== SUMMARY PIVOT TABLE (DETECTION RATES) =====")
summary_pivot = complete_results.pivot_table(
    index=['NormLocation', 'RankLabel'],
    columns='RecordingMethod',
    values='DetectionRate',
    fill_value=0  # Explicitly fill missing combinations with 0
).round(3)

print(summary_pivot)

# === HITUNG DETECTION RATE DIFFERENCE ===
def calculate_improvement_factors(results_df):
    """
    Menghitung selisih Detection Rate untuk setiap metode terhadap baseline Mono
    """
    improvement_data = []

    for location in ['Lab', 'Silwood Park']:
        location_data = results_df[results_df['NormLocation'] == location]

        # Ambil data Mono sebagai baseline
        mono_data = location_data[location_data['RecordingMethod'] == 'Mono'].set_index('Rank')['DetectionRate']

        for method in ['SA', 'LabIR', 'SPIR']:
            method_data = location_data[location_data['RecordingMethod'] == method].set_index('Rank')['DetectionRate']

            for rank in range(7):
                mono_rate = mono_data.get(rank, 0)
                method_rate = method_data.get(rank, 0)

                # Hitung DETECTION RATE DIFFERENCE (Primary Metric)
                detection_rate_difference = method_rate - mono_rate

                improvement_data.append({
                    'NormLocation': location,
                    'Method': method,
                    'Rank': rank,
                    'RankLabel': rank_labels_lab[rank] if location == 'Lab' else rank_labels_sp[rank],
                    'MonoRate': mono_rate,
                    'MethodRate': method_rate,
                    'DetectionRateDifference': detection_rate_difference,
                })

    return pd.DataFrame(improvement_data)

# Hitung improvement factors
improvement_df = calculate_improvement_factors(complete_results)

# === TAMPILKAN IMPROVEMENT FACTORS DALAM TABEL ===
print(f"\n===== DETECTION RATE DIFFERENCE PER RANK =====")

# Lab Improvement Factors
lab_improvement = improvement_df[improvement_df['NormLocation'] == 'Lab'].copy()
lab_improvement = lab_improvement.sort_values(['Method', 'Rank'])
print(f"\n🔬 LAB DETECTION RATE DIFFERENCE:")
for method in ['SA', 'LabIR', 'SPIR']:
    method_data = lab_improvement[lab_improvement['Method'] == method]
    print(f"\n{method} vs Mono:")
    display_data = method_data[['RankLabel', 'MonoRate', 'MethodRate', 'DetectionRateDifference']].round(3)
    print(display_data.to_string(index=False))

# Silwood Park Improvement Factors
sp_improvement = improvement_df[improvement_df['NormLocation'] == 'Silwood Park'].copy()
sp_improvement = sp_improvement.sort_values(['Method', 'Rank'])
print(f"\n🌲 SILWOOD PARK DETECTION RATE DIFFERENCE:")
for method in ['SA', 'LabIR', 'SPIR']:
    method_data = sp_improvement[sp_improvement['Method'] == method]
    print(f"\n{method} vs Mono:")
    display_data = method_data[['RankLabel', 'MonoRate', 'MethodRate', 'DetectionRateDifference']].round(3)
    print(display_data.to_string(index=False))

# === SUMMARY PIVOT TABLE DETECTION RATE DIFFERENCE ===
print(f"\n===== SUMMARY PIVOT TABLE (DETECTION RATE DIFFERENCE) =====")
improvement_pivot = improvement_df.pivot_table(
    index=['NormLocation', 'RankLabel'],
    columns='Method',
    values='DetectionRateDifference',  # CHANGED: Primary metric
    fill_value=0.0  # Default: no improvement
).round(3)

print(improvement_pivot)

# === VISUALISASI ENHANCED ===
plt.figure(figsize=(18, 15))

# Plot 1: Combined Detection Rate Comparison (spanning 2 columns)
plt.subplot2grid((3, 3), (0, 0), colspan=2)  # Row 0, Col 0-1

# Plot configurations
plot_configs = [
    {
        'location': 'Lab',
        'title': 'Lab: SNR Degradation',
        'xlabel': 'SNR Rank (0=best, 6=worst)',
        'marker': 'o',
        'linestyle': '-'
    },
    {
        'location': 'Silwood Park',
        'title': 'Silwood Park: Distance Increase',
        'xlabel': 'Distance Rank (0=closest, 6=farthest)',
        'marker': 's',
        'linestyle': '-'
    }
]

# Plot both locations on same subplot
for config in plot_configs:
    location_results = complete_results[complete_results['NormLocation'] == config['location']]
    for method in ['SA', 'LabIR', 'SPIR']:
        method_data = location_results[location_results['RecordingMethod'] == method]

        # Ubah label legend sesuai permintaan
        if method == 'SPIR':
            legend_label = 'SP-IR-BF'
        elif method == 'LabIR':
            legend_label = 'Lab-IR-BF'
        else:
            legend_label = method

        if config['location'] == 'Lab':
            plt.plot(method_data['Rank']+1, method_data['DetectionRate'],
                 marker=config['marker'], label=f"{legend_label} (Turret {config['location']})",
                 linewidth=2, markersize=6, linestyle=config['linestyle'])
        else:
            plt.plot(method_data['Rank']+1, method_data['DetectionRate'],
                    marker=config['marker'], label=f"{legend_label} ({config['location']})",
                    linewidth=2, markersize=6, linestyle=config['linestyle'])

for config in plot_configs:
    location_results = complete_results[complete_results['NormLocation'] == config['location']]
    for method in ['Mono']:
        method_data = location_results[location_results['RecordingMethod'] == method]
        if config['location'] == 'Lab':
            plt.plot(method_data['Rank']+1, method_data['DetectionRate'],
                 marker=config['marker'], label=f"{method} (Turret {config['location']})",
                 linewidth=2, markersize=6, linestyle=config['linestyle'])
        else:
            plt.plot(method_data['Rank']+1, method_data['DetectionRate'],
                    marker=config['marker'], label=f"{method} ({config['location']})",
                    linewidth=2, markersize=6, linestyle=config['linestyle'])

plt.title('(a) Detection Rate Comparison: Turret Lab vs Silwood Park', fontsize=12, fontweight='bold')
plt.xlabel('Rank')
plt.ylabel('Detection Rate')
plt.legend(loc='lower left', fontsize=9)
plt.grid(True, alpha=0.3)
plt.ylim(0, 1.1)
plt.xticks(range(1,8,1))

# Plot 3: DETECTION RATE DIFFERENCE (Detection Rate Difference)
plt.subplot2grid((3, 3), (1,0), colspan=2)

for location in ['Lab', 'Silwood Park']:
    location_data = improvement_df[improvement_df['NormLocation'] == location]
    for method in ['SA', 'LabIR', 'SPIR']:
        method_data = location_data[location_data['Method'] == method]
        marker = 'o' if location == 'Lab' else 's'

        # Ubah label legend sesuai permintaan
        if method == 'SPIR':
            legend_label = 'SP-IR-BF'
        elif method == 'LabIR':
            legend_label = 'Lab-IR-BF'
        else:
            legend_label = method

        if location == 'Lab':
            label = f'{legend_label} (Turret {location})'
        else:
            label = f'{legend_label} ({location})'

        plt.plot(method_data['Rank']+1, method_data['DetectionRateDifference'],
                 marker=marker, label=label, linewidth=2, markersize=6)

plt.axhline(y=0, color='red', linestyle='--', alpha=0.7, label='No Improvement')
plt.title('(c) Detection Rate Improvement over Mono', fontsize=12, fontweight='bold')
plt.xlabel('Rank')
plt.ylabel('Detection Rate Difference')
plt.legend(loc='best', fontsize=9)
plt.grid(True, alpha=0.3)
plt.xticks(range(1,8,1))

# Plot 4: Average DETECTION RATE DIFFERENCE Comparison
plt.subplot(3, 3, 3)

custom_order = ['SA', 'LabIR', 'SPIR']

# 1. Ekstrak data dan pastikan urutan lokasi konsisten
avg_improvement = improvement_df.groupby(['NormLocation', 'Method'])['DetectionRateDifference'].mean().reset_index()
avg_improvement['Method'] = pd.Categorical(avg_improvement['Method'], categories=custom_order, ordered=True)
avg_improvement = avg_improvement.sort_values('Method')

# 2. PIVOT - ubah index untuk legend
avg_pivot = avg_improvement.pivot(
    index='Method',
    columns='NormLocation',
    values='DetectionRateDifference'
)

# Ubah index untuk tampilan legend
if 'LabIR' in avg_pivot.index:
    avg_pivot = avg_pivot.rename(index={'LabIR': 'Lab-IR-BF'})
if 'SPIR' in avg_pivot.index:
    avg_pivot = avg_pivot.rename(index={'SPIR': 'SP-IR-BF'})

# 3. ATUR WARNA BERDASARKAN LOKASI (EXAKTA)
color_map = {
    "Lab": "#87CEEB",          # Lightblue (hex untuk konsistensi warna)
    "Silwood Park": "#90EE90"  # Lightgreen (hex untuk konsistensi warna)
}
colors = [color_map[col] for col in avg_pivot.columns]

# 4. PLOT DENGAN WARNA SPESIFIK
avg_pivot.plot(
    kind='bar',
    ax=plt.gca(),
    width=0.8,
    color=colors,   # <-- Inisiasi warna sesuai lokasi
    edgecolor='black',  # Opsional: tambah outline untuk kontras
    linewidth=0.8     # Opsional: ketebalan outline
)

# 5. ELEMEN VISUALISASI LAINNYA
plt.axhline(y=0, color='red', linestyle='--', alpha=0.7, label='No Improvement')
plt.title('(b) Average Detection Rate Improvement Across All Ranks', fontsize=12, fontweight='bold')
plt.xlabel('Signal Enhancement Methods')
plt.ylabel('Average Detection Rate Improvement')
plt.xticks(rotation=0)

# 6. LEGENDA DIATUR MANUAL (KARENA WARNA DITETAPKAN SECARA STATIS)
legend_elements = [
    Patch(facecolor=color_map["Lab"], edgecolor='black', label='Turret Lab'),
    Patch(facecolor=color_map["Silwood Park"], edgecolor='black', label='Silwood Park'),
    Line2D([0,0], [1,1], color='red', linestyle='--', label='No Improvement')
]
plt.legend(handles=legend_elements, loc='best')
plt.grid(True, alpha=0.3)

# Plot 5: Box plot of DETECTION RATE DIFFERENCES
plt.subplot(3, 3, 6)
box_data = []
for method in ['SA', 'LabIR', 'SPIR']:
    for location in ['Lab', 'Silwood Park']:
        method_improvements = improvement_df[
            (improvement_df['Method'] == method) &
            (improvement_df['NormLocation'] == location)
        ]['DetectionRateDifference'].values

        box_data.extend([(method, location, val) for val in method_improvements])

box_df = pd.DataFrame(box_data, columns=['Method', 'Location', 'DetectionRateDifference'])

# Ubah nama method untuk tampilan
box_df['Method_Display'] = box_df['Method'].replace({
    'LabIR': 'Lab-IR-BF',
    'SPIR': 'SP-IR-BF'
})

# Create grouped boxplot
positions = []
labels = []
box_data_for_plot = []
for i, method in enumerate(['SA', 'LabIR', 'SPIR']):
    lab_data = box_df[(box_df['Method'] == method) & (box_df['Location'] == 'Lab')]['DetectionRateDifference']
    sp_data = box_df[(box_df['Method'] == method) & (box_df['Location'] == 'Silwood Park')]['DetectionRateDifference']

    positions.extend([i*3 + 1, i*3 + 2])
    box_data_for_plot.extend([lab_data, sp_data])

    # Ubah label untuk tampilan
    display_method = 'Lab-IR-BF' if method == 'LabIR' else ('SP-IR-BF' if method == 'SPIR' else method)
    if i == 1:  # Middle method for labels
        labels.extend([f'{display_method}\nLab', f'{display_method}\nSP'])

bp = plt.boxplot(box_data_for_plot, positions=positions, patch_artist=True, widths=0.6)

# Color coding
colors = ['lightblue', 'lightgreen'] * 3
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)

legend_elements = [
    Patch(facecolor=color_map["Lab"], edgecolor='black', label='Turret Lab'),
    Patch(facecolor=color_map["Silwood Park"], edgecolor='black', label='Silwood Park'),
    Line2D([0], [0], color='red', linestyle='--', label='No Improvement')
]
plt.legend(handles=legend_elements, loc='upper left')
plt.axhline(y=0, color='red', linestyle='--', alpha=0.7, label='No Improvement')
plt.title('(d) Distribution of Detection Rate Improvements', fontsize=12, fontweight='bold')
plt.ylabel('Detection Rate Improvement')
plt.xlabel('Signal Enhancement Methods')
plt.xticks([1.5, 4.5, 7.5], ['SA', 'Lab-IR-BF', 'SP-IR-BF'])  # Ubah label x-axis
plt.grid(True, alpha=0.3)

# Plot 6: Statistical Significance Heatmap
plt.subplot(3, 3, 7)

# Location names for data processing and display
location_display_names = ['Turret Lab', 'Silwood Park']
location_data_names = ['Lab', 'Silwood Park']  # The actual column names in your DataFrame

# Hitung p-values dengan menggunakan nama data ('Lab', 'Silwood Park')
pvalue_matrix = []
methods_order = ['SA', 'LabIR', 'SPIR']

for method in methods_order:
    method_pvals = []
    for idx, display_name in enumerate(location_display_names):
        # Gunakan nama data untuk ekstraksi data dari dataframe
        location_data = location_data_names[idx]

        # Ambil data untuk paired test - menggunakan nama data ('Lab', 'Silwood Park')
        method_rates = improvement_df[
            (improvement_df['Method'] == method) &
            (improvement_df['NormLocation'] == location_data)
        ]['MethodRate'].values

        mono_rates = improvement_df[
            (improvement_df['Method'] == method) &
            (improvement_df['NormLocation'] == location_data)
        ]['MonoRate'].values

        # Wilcoxon signed-rank test
        if len(method_rates) == len(mono_rates) and len(method_rates) > 3:
            try:
                statistic, p_value = wilcoxon(method_rates, mono_rates, alternative='greater')
                method_pvals.append(p_value)
            except:
                method_pvals.append(1.0)
        else:
            method_pvals.append(1.0)

    pvalue_matrix.append(method_pvals)

# Ubah index untuk tampilan heatmap
pvalue_df = pd.DataFrame(pvalue_matrix, index=['SA', 'Lab-IR-BF', 'SP-IR-BF'], columns=location_display_names)

# Create heatmap
sns.heatmap(pvalue_df, annot=True, fmt='.4f', cmap='RdYlGn_r',
            center=0.05, vmin=0, vmax=0.1, cbar_kws={'label': 'P-value'})
plt.title('(e) Statistical Significance\n(Method > Mono, p < 0.05)', fontsize=12, fontweight='bold')
plt.xlabel('Location')
plt.ylabel('Signal Enhancement Methods')

# Plot 7: Summary Statistics Table
plt.subplot(3, 3, 8)
plt.axis('tight')
plt.axis('off')

# Create summary statistics
summary_stats = improvement_df.groupby(['Method', 'NormLocation']).agg({
    'DetectionRateDifference': ['mean', 'std', 'min', 'max']
}).round(3)

summary_stats.columns = ['AI_Mean', 'AI_Std', 'AI_Min', 'AI_Max']
summary_stats = summary_stats.reset_index()

# Ubah nama method untuk tampilan table
summary_stats['Method_Display'] = summary_stats['Method'].replace({
    'LabIR': 'Lab-IR-BF',
    'SPIR': 'SP-IR-BF'
})

# Create pivot table for better layout
ai_mean_pivot = summary_stats.pivot(index='Method_Display', columns='NormLocation', values='AI_Mean')
ai_std_pivot = summary_stats.pivot(index='Method_Display', columns='NormLocation', values='AI_Std')

# Display as table with better formatting
table_data = []
for method in ['SA', 'Lab-IR-BF', 'SP-IR-BF']:
    row_data = [method]
    for location in ['Lab', 'Silwood Park']:
        mean_val = ai_mean_pivot.loc[method, location] if method in ai_mean_pivot.index else 0
        std_val = ai_std_pivot.loc[method, location] if method in ai_std_pivot.index else 0
        formatted_val = f"{mean_val:.3f}±{std_val:.3f}"
        row_data.append(formatted_val)
    table_data.append(row_data)

# Create table with better styling - positioned higher
table = plt.table(
    cellText=table_data,
    colLabels=['Method', 'Turret Lab\n(Mean±SD)', 'Silwood Park\n(Mean±SD)'],
    cellLoc='center',
    loc='center',
    colWidths=[0.25, 0.37, 0.37],
    bbox=[0, 0.55, 1, 0.4]  # [left, bottom, width, height] - moved up
)

# Improve table appearance
table.auto_set_font_size(False)
table.set_fontsize(12)

# Style the header
for i in range(3):
    cell = table[(0, i)]
    cell.set_height(0.2)
    table[(0, i)].set_facecolor('#999')
    table[(0, i)].set_text_props(weight='bold', color='white')

# Style the method column
for i in range(1, 4):
    cell = table[(i, 0)]
    cell.set_height(0.15)
    table[(i, 0)].set_facecolor('#fff')
    table[(i, 0)].set_text_props(weight='bold')

# Colour coding for values
for i in range(1, 4):
    for j in range(1, 3):
        cell = table[(i, j)]
        cell.set_height(0.15)
        cell_text = table_data[i-1][j]
        mean_val = float(cell_text.split('±')[0])
        if mean_val > 0.10:  # High improvement
            table[(i, j)].set_facecolor('#C8E6C9')
        elif mean_val <= 0.10 and mean_val >= 0:  # Medium improvement
            table[(i, j)].set_facecolor('#F1F8E9')
        elif mean_val < 0.00:  # Negative (worse than mono)
            table[(i, j)].set_facecolor('#FFCDD2')

plt.title('(f) Detection Rate Improvement Summary Statistics', fontsize=12, fontweight='bold', pad=0)

# Create legend patches
legend_elements = [
    Patch(facecolor='#C8E6C9', alpha=1.0, label='Significant Improvement (AI > 0.10)'),
    Patch(facecolor='#F1F8E9', alpha=1.0,  label='Minor Improvement (0.00 ≤ AI ≤ 0.10)'),
    Patch(facecolor='#FFCDD2',  alpha=1.0, label='Worse than Mono (AI < 0.00)')
]

# Add legend below the table
plt.legend(
    handles=legend_elements,
    loc='upper center',
    bbox_to_anchor=(0.5, 0.5),  # Adjusted position
    ncol=1,
    fontsize=10.5,
    frameon=True,
    fancybox=True,
    shadow=False,
)

# Remove axis to clean up appearance
plt.axis('off')

# Plot 9: Rank Mapping Table
plt.subplot(3, 3, 9)
plt.axis('tight')
plt.axis('off')

# Create mapping data
rank_mapping_data = [
    ['1', '+30 dB', '1 m'],
    ['2', '+20 dB', '2 m'],
    ['3', '+10 dB', '4 m'],
    ['4', '0 dB', '8 m'],
    ['5', '-10 dB', '16 m'],
    ['6', '-20 dB', '32 m'],
    ['7', '-30 dB', '64 m']
]

# Create table
mapping_table = plt.table(
    cellText=rank_mapping_data,
    colLabels=['Rank', 'Turret Lab\n(SNR)', 'Silwood Park\n(Distance)'],
    cellLoc='center',
    loc='center',
    colWidths=[0.2, 0.4, 0.4],
    bbox=[0, 0.15, 1, 0.8]
)

# Style the table
mapping_table.auto_set_font_size(False)
mapping_table.set_fontsize(12)

# Header styling
for i in range(3):
    cell = mapping_table[(0, i)]
    cell.set_height(0.2)
    cell.set_facecolor('#999')
    cell.set_text_props(weight='bold', color='white')

# Data rows styling with alternating colours
for i in range(1, 8):  # 7 data rows
    for j in range(3):
        cell = mapping_table[(i, j)]
        cell.set_height(0.15)
        # Alternating row colours
        if i % 2 == 1:
            cell.set_facecolor('#F5F5F5')  # Light grey
        else:
            cell.set_facecolor('#FFFFFF')  # White
        # Rank column (bold)
        if j == 0:
            cell.set_text_props(weight='bold')

plt.title('(g) Rank Mapping Reference', fontsize=12, fontweight='bold', pad=0)

# Add explanatory text below table
plt.text(0.5, 0,
         'Rank 1 = Best conditions (highest SNR, closest distance)\n'
         'Rank 7 = Worst conditions (lowest SNR, farthest distance)',
         horizontalalignment='center',
         verticalalignment='center',
         transform=plt.gca().transAxes,
         fontsize=10.5,
         style='normal',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.2))

plt.tight_layout()
filename = f'{TIMESTAMP}_comprehensive_beamforming_analysis.png'
path = os.path.join(base, filename)
plt.savefig(path, dpi=300, bbox_inches='tight')
print(f"Saved figure {filename} to {base}")
plt.show()

# === STATISTICAL SUMMARY ===
print("\n===== STATISTICAL SUMMARY =====")
print("\n1. AVERAGE DETECTION RATE DIFFERENCES:")
avg_summary = improvement_df.groupby(['Method', 'NormLocation'])['DetectionRateDifference'].agg(['mean', 'std', 'count'])
print(avg_summary.round(3))

print("\n2. STATISTICAL SIGNIFICANCE TESTS (Wilcoxon Signed-Rank):")
for method in ['SA', 'LabIR', 'SPIR']:
    for location in ['Lab', 'Silwood Park']:
        method_rates = improvement_df[
            (improvement_df['Method'] == method) &
            (improvement_df['NormLocation'] == location)
        ]['MethodRate'].values

        mono_rates = improvement_df[
            (improvement_df['Method'] == method) &
            (improvement_df['NormLocation'] == location)
        ]['MonoRate'].values

        if len(method_rates) > 3:
            try:
                statistic, p_value = wilcoxon(method_rates, mono_rates, alternative='greater')

                # Dictionary for mapping
                sig_mapping = {
                    (p_value < 0.001): ("***", "p < 0.001"),
                    (0.001 <= p_value < 0.01): ("**", "0.001 ≤ p < 0.01"),
                    (0.01 <= p_value < 0.05): ("*", "0.01 ≤ p < 0.05"),
                    (p_value >= 0.05): ("ns", "p ≥ 0.05")
                }

                significance, range_desc = next(v for k, v in sig_mapping.items() if k)
                print(f"{method} vs Mono ({location}): p = {p_value:.4f} ({range_desc}) {significance}")
            except Exception as e:
                print(f"{method} vs Mono ({location}): Test failed - {e}")

print("\n3. BEST PERFORMINpG METHODS:")
best_methods = improvement_df.groupby(['NormLocation'])['DetectionRateDifference'].agg(['idxmax'])
for location in ['Lab', 'Silwood Park']:
    best_idx = improvement_df[improvement_df['NormLocation'] == location]['DetectionRateDifference'].idxmax()
    best_row = improvement_df.loc[best_idx]
    print(f"{location}: {best_row['Method']} (Rank {best_row['Rank']}) - AI = {best_row['DetectionRateDifference']:.3f}")

# LabIR at Lab vs SPIR at Silwood Park

In [ ]:
# Ambil confidence scores untuk kedua kondisi
labir_lab_confidence = df[
    (df['RecordingMethod'] == 'LabIR') &
    (df['NormLocation'] == 'Lab')
]['Confidence'].values

spir_sp_confidence = df[
    (df['RecordingMethod'] == 'SPIR') &
    (df['NormLocation'] == 'Silwood Park')
]['Confidence'].values

print(f"Jumlah confidence LabIR-Lab: {len(labir_lab_confidence)}")
print(f"Jumlah confidence SPIR-Silwood Park: {len(spir_sp_confidence)}")

In [ ]:
import scipy.stats as stats

# === STATISTICAL ANALYSIS OF CONFIDENCE SCORES ===
def compare_confidence_scores(labir_conf, spir_conf):
    """
    Bandingkan distribusi confidence score antara LabIR di Lab dan SPIR di Silwood Park
    """
    results = {}

    # 1. Deskriptif statistik
    results['stat_descriptive'] = {
        'labir_mean': np.mean(labir_conf),
        'labir_median': np.median(labir_conf),
        'labir_std': np.std(labir_conf),
        'labir_min': np.min(labir_conf),
        'labir_max': np.max(labir_conf),
        'labir_count': len(labir_conf),
        'spir_mean': np.mean(spir_conf),
        'spir_median': np.median(spir_conf),
        'spir_std': np.std(spir_conf),
        'spir_min': np.min(spir_conf),
        'spir_max': np.max(spir_conf),
        'spir_count': len(spir_conf)
    }

    # 2. Uji statistik (Mann-Whitney U test)
    stat, p_value = stats.mannwhitneyu(
        labir_conf, spir_conf,
        alternative='two-sided'
    )
    results['mann_whitney'] = {
        'statistic': stat,
        'p_value': p_value
    }

    # 3. Effect size (Cliff's Delta)
    n1, n2 = len(labir_conf), len(spir_conf)
    pairs = sum(1 for x in labir_conf for y in spir_conf if x > y) - sum(1 for x in labir_conf for y in spir_conf if x < y)
    cliff_delta = pairs / (n1 * n2)
    results['cliffs_delta'] = cliff_delta

    # 4. High-confidence detection rate
    high_threshold = 0.8
    high_labir = np.sum(labir_conf >= high_threshold) / len(labir_conf)
    high_spir = np.sum(spir_conf >= high_threshold) / len(spir_conf)
    results['high_confidence_rate'] = {
        'labir': high_labir,
        'spir': high_spir
    }

    return results

# Jalankan analisis
confidence_comparison = compare_confidence_scores(labir_lab_confidence, spir_sp_confidence)

# Tampilkan hasil
print("\n===== ANALISIS PERBANDINGAN CONFIDENCE SCORE =====")
print(f"\nStatistik Deskriptif:")
print(f"{'':<20} | {'LabIR-Lab':<10} | {'SPIR-Silwood':<12}")
print("-" * 50)
print(f"{'Mean':<20} | {confidence_comparison['stat_descriptive']['labir_mean']:.4f} | {confidence_comparison['stat_descriptive']['spir_mean']:.4f}")
print(f"{'Median':<20} | {confidence_comparison['stat_descriptive']['labir_median']:.4f} | {confidence_comparison['stat_descriptive']['spir_median']:.4f}")
print(f"{'SD':<20} | {confidence_comparison['stat_descriptive']['labir_std']:.4f} | {confidence_comparison['stat_descriptive']['spir_std']:.4f}")
print(f"{'Min':<20} | {confidence_comparison['stat_descriptive']['labir_min']:.4f} | {confidence_comparison['stat_descriptive']['spir_min']:.4f}")
print(f"{'Max':<20} | {confidence_comparison['stat_descriptive']['labir_max']:.4f} | {confidence_comparison['stat_descriptive']['spir_max']:.4f}")
print(f"{'High-Conf Rate':<20} | {confidence_comparison['high_confidence_rate']['labir']:.4f} | {confidence_comparison['high_confidence_rate']['spir']:.4f}")

# Interpretasi uji statistik
if confidence_comparison['mann_whitney']['p_value'] < 0.05:
    significance = "Signifikan"
else:
    significance = "Tidak Signifikan"

print(f"\nMann-Whitney U Test (p = {confidence_comparison['mann_whitney']['p_value']:.4f}): {significance}")
print(f"Cliff's Delta (effect size): {confidence_comparison['cliffs_delta']:.4f}")

# Interpretasi effect size
if abs(confidence_comparison['cliffs_delta']) < 0.147:
    effect_size = "Sangat Kecil"
elif abs(confidence_comparison['cliffs_delta']) < 0.33:
    effect_size = "Kecil"
elif abs(confidence_comparison['cliffs_delta']) < 0.474:
    effect_size = "Sedang"
else:
    effect_size = "Besar"
print(f"Effect Size: {effect_size}")

In [ ]:
# === BUAT VISUALISASI PERBANDINGAN CONFIDENCE SCORE ===
plt.figure(figsize=(14, 10))

# Plot 1: Boxplot dan Violin Plot
plt.subplot(2, 2, 1)
# Boxplot
sns.boxplot(data=[labir_lab_confidence, spir_sp_confidence], palette=['skyblue', 'lightgreen'])
# Violin plot
sns.violinplot(data=[labir_lab_confidence, spir_sp_confidence],
              palette=['skyblue', 'lightgreen'],
              inner=None,  # Remove inner markers
              alpha=0.3)  # Transparansi

plt.xticks([0, 1], ['LabIR (Lab Environment)', 'SPIR (Silwood Park)'])
plt.ylabel('Confidence Score (0.4 - 1.0)')
plt.title('(a) Distribution of Confidence Scores', fontsize=12, fontweight='bold')
plt.grid(True, axis='y', alpha=0.3)

# Tambahkan jumlah sample di bawah
plt.text(0, 0.38, f"n={len(labir_lab_confidence)}", ha='center',
         bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=2))
plt.text(1, 0.38, f"n={len(spir_sp_confidence)}", ha='center',
         bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=2))

# Plot 2: ECDF (Empirical Cumulative Distribution Function)
plt.subplot(2, 2, 2)
def plot_ecdf(data, label, color):
    x = np.sort(data)
    y = np.arange(1, len(x)+1) / len(x)
    plt.plot(x, y, label=label, color=color, linewidth=2)

plot_ecdf(labir_lab_confidence, f'LabIR-Lab (n={len(labir_lab_confidence)})', 'blue')
plot_ecdf(spir_sp_confidence, f'SPIR-Silwood (n={len(spir_sp_confidence)})', 'green')

plt.xlabel('Confidence Score')
plt.ylabel('Cumulative Probability')
plt.title('(b) ECDF of Confidence Scores', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend()

# Plot 3: Histogram dengan KDE
plt.subplot(2, 2, 3)
sns.histplot(labir_lab_confidence, kde=True, color='skyblue',
             label=f'LabIR-Lab (n={len(labir_lab_confidence)})',
             element='step', fill=True, alpha=0.5)
sns.histplot(spir_sp_confidence, kde=True, color='lightgreen',
             label=f'SPIR-Silwood (n={len(spir_sp_confidence)})',
             element='step', fill=True, alpha=0.5)

plt.xlabel('Confidence Score')
plt.ylabel('Frequency')
plt.title('(c) Histogram of Confidence Scores', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend()

# Plot 4: Confidence Score per Rank
plt.subplot(2, 2, 4)
confidence_by_rank = []

# Kumpulkan confidence per rank untuk LabIR-Lab
for rank in range(7):
    conf_scores = df[
        (df['RecordingMethod'] == 'LabIR') &
        (df['NormLocation'] == 'Lab') &
        (df['Rank'] == rank)
    ]['Confidence'].values

    for score in conf_scores:
        confidence_by_rank.append({
            'Method': 'LabIR-Lab',
            'Rank': rank,
            'Confidence': score
        })

# Kumpulkan confidence per rank untuk SPIR-Silwood
for rank in range(7):
    conf_scores = df[
        (df['RecordingMethod'] == 'SPIR') &
        (df['NormLocation'] == 'Silwood Park') &
        (df['Rank'] == rank)
    ]['Confidence'].values

    for score in conf_scores:
        confidence_by_rank.append({
            'Method': 'SPIR-Silwood',
            'Rank': rank,
            'Confidence': score
        })

confidence_rank_df = pd.DataFrame(confidence_by_rank)
sns.boxplot(x='Rank', y='Confidence', hue='Method', data=confidence_rank_df,
           palette={'LabIR-Lab': 'skyblue', 'SPIR-Silwood': 'lightgreen'})

plt.xlabel('Rank (Degradation Level)')
plt.ylabel('Confidence Score')
plt.title('(d) Confidence Score by Rank', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend(title='')

plt.tight_layout()
plt.savefig(os.path.join(base, f'{TIMESTAMP}_confidence_comparison.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# === ANALISIS PERBANDINGAN PER-RANK ===
print("\n===== ANALISIS CONFIDENCE SCORE PER-RANK =====")

confidence_by_rank_df = pd.DataFrame(columns=[
    'Rank', 'RankLabel', 'Location',
    'RecordingMethod', 'MeanConf', 'MedianConf', 'HighConfRate', 'Count'
])

# Lakukan untuk semua kombinasi yang relevan
for location, rank_labels in zip(['Lab', 'Silwood Park'], [rank_labels_lab, rank_labels_sp]):
    for rank in range(7):
        rank_label = rank_labels[rank]

        # LabIR di Lab
        if location == 'Lab':
            RecordingMethod = 'LabIR'
            conf_scores = df[
                (df['RecordingMethod'] == 'LabIR') &  # Perhatikan: di kode Anda sebenarnya menggunakan RecordingMethod
                (df['NormLocation'] == 'Lab') &
                (df['Rank'] == rank)
            ]['Confidence'].values

            if len(conf_scores) > 0:
                new_row = {
                    'Rank': rank,
                    'RankLabel': rank_label,
                    'Location': location,
                    'RecordingMethod': method,
                    'MeanConf': np.mean(conf_scores),
                    'MedianConf': np.median(conf_scores),
                    'HighConfRate': np.mean(conf_scores >= 0.8),
                    'Count': len(conf_scores)
                }
                confidence_by_rank_df = pd.concat(
                    [confidence_by_rank_df, pd.DataFrame([new_row])],
                    ignore_index=True
                )

        # SPIR di Silwood Park
        if location == 'Silwood Park':
            RecordingMethod = 'SPIR'
            conf_scores = df[
                (df['RecordingMethod'] == 'SPIR') &
                (df['NormLocation'] == 'Silwood Park') &
                (df['Rank'] == rank)
            ]['Confidence'].values

            if len(conf_scores) > 0:
                new_row = {
                    'Rank': rank,
                    'RankLabel': rank_label,
                    'Location': location,
                    'RecordingMethod': method,
                    'MeanConf': np.mean(conf_scores),
                    'MedianConf': np.median(conf_scores),
                    'HighConfRate': np.mean(conf_scores >= 0.8),
                    'Count': len(conf_scores)
                }
                confidence_by_rank_df = pd.concat(
                    [confidence_by_rank_df, pd.DataFrame([new_row])],
                    ignore_index=True
                )

# Tampilkan dalam format tabel yang rapi
print("\nConfidence Score per Rank:")
pivot_table = confidence_by_rank_df.pivot_table(
    index=['Rank', 'RankLabel'],
    columns=['Location', 'RecordingMethod'],
    values=['MeanConf', 'HighConfRate'],
    aggfunc='mean'
).round(3)

print(pivot_table)

# Identifikasi mana yang lebih baik per rank
rank_comparison = []
for rank in range(7):
    lab_data = confidence_by_rank_df[
        (confidence_by_rank_df['Location'] == 'Lab') &
        (confidence_by_rank_df['Rank'] == rank)
    ]
    sp_data = confidence_by_rank_df[
        (confidence_by_rank_df['Location'] == 'Silwood Park') &
        (confidence_by_rank_df['Rank'] == rank)
    ]

    if len(lab_data) > 0 and len(sp_data) > 0:
        lab_mean = lab_data['MeanConf'].values[0]
        sp_mean = sp_data['MeanConf'].values[0]
        winner = "LabIR" if lab_mean > sp_mean else "SPIR"
        rank_comparison.append({
            'Rank': rank,
            'RankLabel': lab_data['RankLabel'].values[0],
            'LabIR_MeanConf': lab_mean,
            'SPIR_MeanConf': sp_mean,
            'Winner': winner,
            'Difference': abs(lab_mean - sp_mean)
        })

rank_comparison_df = pd.DataFrame(rank_comparison)
print("\nPerbandingan per Rank (Siapa yang lebih baik?):")
print(rank_comparison_df[['Rank', 'RankLabel', 'LabIR_MeanConf', 'SPIR_MeanConf', 'Winner', 'Difference']].to_string(index=False))

# EKSTRAK DATA CONFIDENCE SCORE UNTUK ANALISIS

In [ ]:
# === EKSTRAK DATA CONFIDENCE SCORE UNTUK ANALISIS ===
confidence_data = []

for location in ['Lab', 'Silwood Park']:
    location_data = df_sorted[df_sorted['NormLocation'] == location]

    for method in ['LabIR', 'SPIR']:
        if location == 'Lab' and method == 'LabIR':
            # LabIR di Lab
            method_data = location_data[location_data['RecordingMethod'] == method]
            confidence_values = method_data['Confidence'].values

            for confidence in confidence_values:
                confidence_data.append({
                    'NormLocation': location,
                    'Method': method,
                    'Confidence': confidence
                })

        elif location == 'Silwood Park' and method == 'SPIR':
            # SPIR di Silwood Park
            method_data = location_data[location_data['RecordingMethod'] == method]
            confidence_values = method_data['Confidence'].values

            for confidence in confidence_values:
                confidence_data.append({
                    'NormLocation': location,
                    'Method': method,
                    'Confidence': confidence
                })

confidence_df = pd.DataFrame(confidence_data)
print(f"\nData confidence score terkumpul: {len(confidence_df)}")
print(confidence_df.groupby(['NormLocation', 'Method']).size())

# === ANALISIS STATISTIK CONFIDENCE SCORE ===
def analyze_confidence_comparison(confidence_df):
    """
    Analisis komparatif confidence score antara LabIR di Lab dan SPIR di Silwood Park
    """
    print("\n" + "="*60)
    print("ANALISIS CONFIDENCE SCORE: LabIR vs SPIR")
    print("="*60)

    # Pisahkan data
    labir_lab_data = confidence_df[
        (confidence_df['Method'] == 'LabIR') &
        (confidence_df['NormLocation'] == 'Lab')
    ]['Confidence']

    spir_sp_data = confidence_df[
        (confidence_df['Method'] == 'SPIR') &
        (confidence_df['NormLocation'] == 'Silwood Park')
    ]['Confidence']

    print(f"LabIR di Lab: {len(labir_lab_data)} samples")
    print(f"SPIR di Silwood Park: {len(spir_sp_data)} samples")

    # Statistik deskriptif
    print("\n--- STATISTIK DESKRIPTIF ---")
    labir_stats = labir_lab_data.describe()
    spir_stats = spir_sp_data.describe()

    stats_comparison = pd.DataFrame({
        'LabIR_Lab': labir_stats,
        'SPIR_SP': spir_stats
    }).round(3)

    print(stats_comparison)

    # Uji normalitas
    print("\n--- UJI NORMALITAS (Shapiro-Wilk) ---")
    from scipy.stats import shapiro

    _, labir_p = shapiro(labir_lab_data)
    _, spir_p = shapiro(spir_sp_data)

    print(f"LabIR di Lab: p-value = {labir_p:.4f} {'(Normal)' if labir_p > 0.05 else '(Tidak Normal)'}")
    print(f"SPIR di Silwood Park: p-value = {spir_p:.4f} {'(Normal)' if spir_p > 0.05 else '(Tidak Normal)'}")

    # Uji perbedaan statistik
    print("\n--- UJI PERBEDAAN STATISTIK ---")

    if labir_p > 0.05 and spir_p > 0.05:
        # Kedua distribusi normal, gunakan t-test
        from scipy.stats import ttest_ind
        t_stat, p_value = ttest_ind(labir_lab_data, spir_sp_data)
        test_name = "Independent t-test"
    else:
        # Distribusi tidak normal, gunakan Mann-Whitney U
        from scipy.stats import mannwhitneyu
        u_stat, p_value = mannwhitneyu(labir_lab_data, spir_sp_data)
        test_name = "Mann-Whitney U test"

    print(f"{test_name}: p-value = {p_value:.4f}")

    if p_value < 0.05:
        print("→ Perbedaan SIGNIFIKAN secara statistik")

        # Hitung effect size
        def cliffs_delta(x, y):
            """Calculate Cliff's Delta effect size"""
            n_x, n_y = len(x), len(y)
            pairs = sum(1 for i in x for j in y if i > j) - sum(1 for i in x for j in y if i < j)
            return pairs / (n_x * n_y)

        effect_size = cliffs_delta(labir_lab_data, spir_sp_data)
        print(f"Effect size (Cliff's Delta): {effect_size:.3f}")

        # Interpretasi effect size
        if abs(effect_size) < 0.147:
            size_label = "negligible"
        elif abs(effect_size) < 0.33:
            size_label = "small"
        elif abs(effect_size) < 0.474:
            size_label = "medium"
        else:
            size_label = "large"

        print(f"Interpretasi effect size: {size_label}")

    else:
        print("→ Tidak ada perbedaan signifikan secara statistik")

    # Visualisasi distribusi
    print("\n--- VISUALISASI DISTRIBUSI ---")
    plt.figure(figsize=(15, 5))

    # Box plot
    plt.subplot(1, 2, 1)
    plot_data = [labir_lab_data, spir_sp_data]
    plt.boxplot(plot_data, labels=['LabIR (Lab)', 'SPIR (SP)'])
    plt.title('Distribusi Confidence Score')
    plt.ylabel('Confidence Score')

    # ECDF plot
    plt.subplot(1, 2, 2)
    def plot_ecdf(data, label, color):
        x = np.sort(data)
        y = np.arange(1, len(x)+1) / len(x)
        plt.plot(x, y, label=label, color=color, linewidth=2)

    plot_ecdf(labir_lab_data, 'LabIR (Lab)', 'blue')
    plot_ecdf(spir_sp_data, 'SPIR (SP)', 'green')
    plt.xlabel('Confidence Score')
    plt.ylabel('ECDF')
    plt.title('Empirical CDF Comparison')
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    return {
        'labir_lab_data': labir_lab_data,
        'spir_sp_data': spir_sp_data,
        'p_value': p_value,
        'effect_size': effect_size if p_value < 0.05 else None
    }

# Jalankan analisis
confidence_results = analyze_confidence_comparison(confidence_df)

In [ ]:
import matplotlib.pyplot as plt
import plotly.express as px
import pandas as pd
import numpy as np
import scipy.stats as stats # Import stats

# Assuming confidence_df DataFrame is available from previous cells
# If not, you might need to re-run the cells that create it.

plt.style.use('seaborn-v0_8-darkgrid') # Use a standard style for better appearance


# Modifikasi fungsi analyze_confidence_comparison
def analyze_confidence_comparison(confidence_df):
    # Filter data (pastikan hanya dua grup yang valid)
    valid_groups = confidence_df[
        ((confidence_df['Method'] == 'LabIR') & (confidence_df['NormLocation'] == 'Lab')) |
        ((confidence_df['Method'] == 'SPIR') & (confidence_df['NormLocation'] == 'Silwood Park'))
    ].copy() # Use .copy() to avoid SettingWithCopyWarning

    # Buat label kategori baru untuk sumbu X (Wajib!)
    valid_groups['Group_Label'] = valid_groups.apply(
        lambda x: f"LabIR (Lab)" if (x['Method'] == 'LabIR' and x['NormLocation'] == 'Lab')
                 else "SPIR (Silwood Park)", # Changed label for clarity
        axis=1
    )

    # --- INI ADALAH STRUKTUR DATA YANG BENAR ---
    plot_data = valid_groups[["Group_Label", "Confidence"]]

    # Analisis statistik
    labir_data = plot_data[plot_data['Group_Label'] == 'LabIR (Lab)']['Confidence']
    spir_data = plot_data[plot_data['Group_Label'] == 'SPIR (Silwood Park)']['Confidence']

    # Ensure data is numeric and handle potential NaNs before statistical test
    labir_data = pd.to_numeric(labir_data, errors='coerce').dropna()
    spir_data = pd.to_numeric(spir_data, errors='coerce').dropna()


    # Uji statistik (contoh) - Only perform if sufficient data
    u_stat, p_val = np.nan, np.nan # Initialize with NaN
    if len(labir_data) > 1 and len(spir_data) > 1:
        u_stat, p_val = stats.mannwhitneyu(labir_data, spir_data, alternative='two-sided')
        print(f"Mann-Whitney U-value: {u_stat:.2f}, p-value: {p_val:.6f}")
    else:
        print("Not enough data to perform Mann-Whitney U test.")


    # Kembalikan data untuk plotting (format long-form)
    return plot_data

# Jalankan analysis & ambil data plot
# Ensure confidence_df is available or created before this cell
# Example: confidence_df = ... (code to create confidence_df)

# Assuming confidence_df exists from previous cells
# Filter and prepare data for plotting
plot_data = analyze_confidence_comparison(confidence_df)

# Ensure Confidence column is numeric and drop NaNs
plot_data['Confidence'] = pd.to_numeric(plot_data['Confidence'], errors='coerce')
plot_data = plot_data.dropna(subset=['Confidence'])

# Sample data if needed for performance with plotly
# Use value_counts to understand group sizes before sampling
# print("\nGroup Sizes before Sampling:")
# print(plot_data['Group_Label'].value_counts())

# Sample data if any group is too large
sample_limit = 5000 # Adjust sample size as needed
sampled_groups = []
for group_label, group_data in plot_data.groupby('Group_Label'):
    if len(group_data) > sample_limit:
        sampled_groups.append(group_data.sample(sample_limit, random_state=42)) # Added random_state for reproducibility
        # print(f"Sampled {sample_limit} from {group_label}")
    else:
        sampled_groups.append(group_data)
        # print(f"Included all {len(group_data)} from {group_label}")

if sampled_groups:
    plot_data_sample = pd.concat(sampled_groups).reset_index(drop=True)
    # print(f"Total samples for plotting: {len(plot_data_sample)}")
else:
    plot_data_sample = pd.DataFrame() # Handle case where no data is available

if not plot_data_sample.empty:
    fig = px.violin(
        plot_data_sample,
        y='Confidence',
        x='Group_Label',
        box=True,
        points=False,  # Corrected value
        color='Group_Label',
        hover_data=plot_data_sample.columns.tolist()
    )
    fig.update_layout(
        title='Confidence Score Distribution: LabIR (Lab) vs SPIR (Silwood Park)', # Updated title
        yaxis_title='Confidence Score',
        xaxis_title='Recording Method and Environment', # Updated axis title
        font=dict(size=12)
    )
    fig.show()
else:
    print("\nNo data available for plotting after filtering and sampling.")

# 1. SAMPEL DATA (wajib untuk dataset besar >10.000)
plot_data_sample = plot_data.groupby('Group_Label', group_keys=False).apply(
    lambda g: g.sample(min(len(g), 2000), random_state=42)
)

# 2. KONVERT KE NUMERIK (pastikan tidak ada string)
plot_data_sample['Confidence'] = plot_data_sample['Confidence'].astype(float)

# 3. PLOT DENGAN PARAMETER TERBARU (tanpa warning)
plt.figure(figsize=(8, 6))

# SOLUSI TANPA WARNING SEABORN v0.14+
ax = sns.violinplot(
    data=plot_data_sample,
    x='Group_Label',
    y='Confidence',
    hue='Group_Label',          # [PERBAIKAN: Solusi FutureWarning pallette]
    palette="Set2",
    cut=0,
    inner="quartile",
    density_norm='count',       # [PERBAIKAN: Ganti scale -> density_norm]
    bw_adjust=0.15,             # [PERBAIKAN: Ganti bw -> bw_adjust]
    legend=False,               # [PERBAIKAN: Hilangin legend duplikat]
    width=0.8                   # Lebar violin (opsional)
)

# GUNAKAN STRIPPLOT (bukan swarmplot) untuk visualisasi titik
# sns.stripplot(
#     data=plot_data_sample,
#     x='Group_Label',
#     y='Confidence',
#     color='black',
#     alpha=0.25,
#     size=2.5,
#     jitter=0.25,                # Kontrol jarak titik
#     linewidth=0.5
# )

# 4. HIASAN TAMBAHAN
plt.title('Perbandingan Confidence Score: LabIR vs SPIR', fontsize=14, pad=15)
plt.ylabel('Confidence Score', fontsize=12, labelpad=10)
plt.xlabel('')  # Kosongkan label X

# Tampilkan statistik di plot
medians = plot_data.groupby('Group_Label')['Confidence'].median()
counts = plot_data['Group_Label'].value_counts()

for i, (group, median) in enumerate(medians.items()):
    plt.text(i, median + 0.03, f"Median: {median:.2f}",
             ha='center', fontsize=9)
    plt.text(i, -0.08, f"n={counts[group]}",
             ha='center', fontsize=10, color='darkred')

plt.grid(axis='y', linestyle='--', alpha=0.3, linewidth=0.7)
plt.tight_layout()
plt.show()

# ANALISIS CONFIDENCE SCORE PER RANK

In [ ]:
# === ANALISIS CONFIDENCE SCORE PER RANK ===
def analyze_confidence_by_rank(df_sorted):
    """
    Analisis confidence score per rank untuk LabIR dan SPIR
    """
    print("\n" + "="*60)
    print("ANALISIS CONFIDENCE SCORE PER RANK")
    print("="*60)

    rank_confidence_data = []

    for location in ['Lab', 'Silwood Park']:
        location_data = df_sorted[df_sorted['NormLocation'] == location]

        for method in ['LabIR', 'SPIR']:
            if (location == 'Lab' and method == 'LabIR') or (location == 'Silwood Park' and method == 'SPIR'):
                method_data = location_data[location_data['RecordingMethod'] == method]

                for rank in range(7):
                    rank_data = method_data[method_data['Rank'] == rank]
                    if len(rank_data) > 0:
                        mean_confidence = rank_data['Confidence'].mean()
                        std_confidence = rank_data['Confidence'].std()
                        count = len(rank_data)

                        rank_confidence_data.append({
                            'NormLocation': location,
                            'Method': method,
                            'Rank': rank,
                            'MeanConfidence': mean_confidence,
                            'StdConfidence': std_confidence,
                            'Count': count
                        })

    rank_confidence_df = pd.DataFrame(rank_confidence_data)

    # Visualisasi
    plt.figure(figsize=(12, 6))

    # Plot mean confidence per rank
    for location in ['Lab', 'Silwood Park']:
        loc_data = rank_confidence_df[rank_confidence_df['NormLocation'] == location]
        method = 'LabIR' if location == 'Lab' else 'SPIR'
        method_data = loc_data[loc_data['Method'] == method]

        plt.errorbar(
            method_data['Rank'] + 1,
            method_data['MeanConfidence'],
            yerr=method_data['StdConfidence'],
            label=f'{method} ({location})',
            marker='o' if location == 'Lab' else 's',
            capsize=5,
            linewidth=2
        )

    plt.xlabel('Rank')
    plt.ylabel('Mean Confidence Score')
    plt.title('Mean Confidence Score per Rank')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.xticks(range(1, 8))
    plt.show()

    # Tampilkan tabel
    print("\n--- RINGKASAN CONFIDENCE SCORE PER RANK ---")
    display_df = rank_confidence_df.pivot_table(
        index=['NormLocation', 'Method', 'Rank'],
        values=['MeanConfidence', 'StdConfidence', 'Count']
    ).round(3)

    print(display_df)

    return rank_confidence_df

# Jalankan analisis per rank
rank_confidence_results = analyze_confidence_by_rank(df_sorted)

# 2. STATISTICAL SIGNIFICANCE TESTS (Wilcoxon Signed-Rank)

In [ ]:
import numpy as np
from scipy.stats import wilcoxon, norm
import math

print("\n2. STATISTICAL SIGNIFICANCE TESTS (Wilcoxon Signed-Rank):")
print("=" * 80)

for method in ['SA', 'LabIR', 'SPIR']:
    for location in ['Lab', 'Silwood Park']:
        method_rates = improvement_df[
            (improvement_df['Method'] == method) &
            (improvement_df['NormLocation'] == location)
        ]['MethodRate'].values

        mono_rates = improvement_df[
            (improvement_df['Method'] == method) &
            (improvement_df['NormLocation'] == location)
        ]['MonoRate'].values

        if len(method_rates) > 3:
            print(f"\n🔍 DETAILED CALCULATION: {method} vs Mono ({location})")
            print("-" * 60)

            # Step 1: Show raw data
            print(f"📊 Raw Data (n = {len(method_rates)}):")
            print(f"   Method rates: {method_rates}")
            print(f"   Mono rates:   {mono_rates}")

            # Step 2: Calculate differences
            differences = method_rates - mono_rates
            print(f"\n📈 Step 1 - Calculate Differences:")
            print(f"   Differences = Method - Mono:")
            for i, (m, mono, diff) in enumerate(zip(method_rates, mono_rates, differences)):
                print(f"   Pair {i+1}: {m:.4f} - {mono:.4f} = {diff:+.4f}")
            print(f"   All differences: {differences}")

            # Step 3: Remove zero differences and get absolute values
            non_zero_diffs = differences[differences != 0]
            abs_diffs = np.abs(non_zero_diffs)
            print(f"\n📏 Step 2 - Remove Zeros & Get Absolute Values:")
            print(f"   Non-zero differences: {non_zero_diffs}")
            print(f"   Absolute differences: {abs_diffs}")
            print(f"   n_effective = {len(non_zero_diffs)}")

            # Step 4: Rank the absolute differences
            from scipy.stats import rankdata
            ranks = rankdata(abs_diffs, method='average')
            print(f"\n🏆 Step 3 - Rank Absolute Differences:")
            print(f"   Absolute differences: {abs_diffs}")
            print(f"   Ranks:               {ranks}")

            # Show ranking details
            unique_vals = np.unique(abs_diffs)
            print(f"   Ranking details:")
            for val in unique_vals:
                indices = np.where(abs_diffs == val)[0]
                if len(indices) > 1:
                    avg_rank = np.mean(ranks[indices])
                    print(f"     Value {val:.4f} appears {len(indices)} times → average rank = {avg_rank}")
                else:
                    print(f"     Value {val:.4f} appears 1 time → rank = {ranks[indices[0]]}")

            # Step 5: Calculate W+ and W-
            w_plus = np.sum(ranks[non_zero_diffs > 0])
            w_minus = np.sum(ranks[non_zero_diffs < 0])
            print(f"\n⚖️ Step 4 - Calculate Test Statistics:")
            print(f"   Positive differences: {non_zero_diffs[non_zero_diffs > 0]}")
            print(f"   Negative differences: {non_zero_diffs[non_zero_diffs < 0]}")
            print(f"   W+ (sum of ranks for positive diffs) = {w_plus}")
            print(f"   W- (sum of ranks for negative diffs) = {w_minus}")
            print(f"   Check: W+ + W- = {w_plus + w_minus} = n(n+1)/2 = {len(non_zero_diffs)*(len(non_zero_diffs)+1)/2}")

            # Step 6: Determine test statistic based on alternative
            test_statistic = w_plus  # For 'greater' alternative
            print(f"\n📊 Step 5 - Test Statistic:")
            print(f"   Alternative hypothesis: 'greater' (method > mono)")
            print(f"   Test statistic = W+ = {test_statistic}")

            # Step 7: Calculate p-value
            n = len(non_zero_diffs)
            print(f"\n🧮 Step 6 - P-value Calculation:")

            if n <= 25:
                print(f"   Small sample (n = {n} ≤ 25): Using exact distribution")
            else:
                print(f"   Large sample (n = {n} > 25): Using normal approximation")
                mu = n * (n + 1) / 4
                sigma = math.sqrt(n * (n + 1) * (2 * n + 1) / 24)
                z_score = (test_statistic - 0.5 - mu) / sigma  # with continuity correction
                p_manual = 1 - norm.cdf(z_score)

                print(f"   Expected value μ = n(n+1)/4 = {n}×{n+1}/4 = {mu}")
                print(f"   Standard deviation σ = √[n(n+1)(2n+1)/24] = √[{n}×{n+1}×{2*n+1}/24] = {sigma:.4f}")
                print(f"   Z-score = (W+ - 0.5 - μ)/σ = ({test_statistic} - 0.5 - {mu})/{sigma:.4f} = {z_score:.4f}")
                print(f"   P-value = 1 - Φ(Z) = 1 - Φ({z_score:.4f}) = {p_manual:.6f}")

            # Step 8: Compare with scipy result
            try:
                statistic_scipy, p_value_scipy = wilcoxon(method_rates, mono_rates, alternative='greater')
                print(f"\n✅ Step 7 - Verification with SciPy:")
                print(f"   SciPy statistic = {statistic_scipy}")
                print(f"   SciPy p-value = {p_value_scipy:.6f}")

                if n > 25:
                    print(f"   Manual p-value = {p_manual:.6f}")
                    print(f"   Difference = {abs(p_value_scipy - p_manual):.8f}")

                # Significance interpretation
                sig_mapping = {
                    (p_value_scipy < 0.001): ("***", "p < 0.001"),
                    (0.001 <= p_value_scipy < 0.01): ("**", "0.001 ≤ p < 0.01"),
                    (0.01 <= p_value_scipy < 0.05): ("*", "0.01 ≤ p < 0.05"),
                    (p_value_scipy >= 0.05): ("ns", "p ≥ 0.05")
                }
                significance, range_desc = next(v for k, v in sig_mapping.items() if k)

                print(f"\n🎯 FINAL RESULT:")
                print(f"   {method} vs Mono ({location}): p = {p_value_scipy:.4f} ({range_desc}) {significance}")

                if significance != "ns":
                    print(f"   ✓ Statistically significant: {method} performs better than Mono")
                else:
                    print(f"   ✗ Not statistically significant: No evidence {method} is better than Mono")

            except Exception as e:
                print(f"   ❌ SciPy test failed: {e}")

            print("\n" + "="*80)